# Checking the EPCAPE seasonal-averages table: the rows attributed to Kavin

Lynn asked for an independent check of every value in the EPCAPE seasonal-averages table
(`Table-EPCAPE_SeasonalAverages.xlsx`, sheet *Cloud&MetQuantities*) whose author column names Kavin.
For each of those rows this notebook

1. says which data and which definition reproduce the quantity,
2. computes the five averages: the whole campaign and the four seasons, each as mean ± standard deviation, and
3. sets them next to the sheet's values, with the percentage difference of the means.

**Quality control.** Every ARM variable that has a `qc_<variable>` companion keeps only samples whose QC value is **exactly 0**, meaning the sample passed every test, including those ARM rates "Indeterminate". Variables distributed without a QC field are used as they are; Section 11 lists which ones.

**Seasons** follow the sheet's column headers: Spring = Apr–Jun 2023, Summer = Jul–Sep 2023, Fall = Oct–Dec 2023, Winter = Feb 15–Mar 2023 + Jan–Feb 14 2024, EPCAPE = 2023-02-15 to 2024-02-14 (UTC).
Each statistic is the arithmetic mean ± sample standard deviation over all samples in the season. The campaign value is therefore sample-weighted, not the mean of the four seasons. The kind of sample (5-min mean, radiosonde launch, rain day, …) is printed with every table, because the standard deviation depends on it.

**Where a definition is undocumented** (rain, PBL height), several readings are computed and ranked by how well they reproduce the sheet. The best match is a hypothesis about Kavin's method to confirm with him, not proof of it.

**Kavin's own code and input files** (shared 2026-10-07) are re-run in Sections 1b (SW), 2b (LW), 3b (LWP), 9b (FM-120 hours), 9c (activated fractions) and 9d (droplet number and diameter, using Abbey Williams' cloud/haze classes) through `kavin.py`, which reproduces his MATLAB live script `ForAndrewVariablesmlx.mlx` line by line. Each of those sections takes three steps:

1. reproduce the sheet from his files with his code;
2. correct any quirk of the code on the same data;
3. set the result next to this check.

Each difference can then be traced to the data or to the method.

## Running it

Download everything once, from the `EPCAPE` folder (resumable; files already present are skipped):

```bash
python comparisons/seasonal_averages/download_data.py
```

This fetches server-side subsets of the ARM products listed in `config.yaml` (with their `qc_` fields), builds one combined file per time-series product in `<data folder>/processed/`, and downloads the Russell-group files (AMS at Mt. Soledad, GCVI enhancement factors, Low Cloud Periods) from the UC San Diego Library into `<data folder>/processed/Aerosol Quantification Products/`.
The spreadsheet is read from `~/Downloads/Table-EPCAPE_SeasonalAverages.xlsx` unless `EPCAPE_SEASONAL_TABLE` points elsewhere.
Outputs (tables, settings, figure) go to `<data folder>/processed/derived/seasonal_averages_check/`.

In [1]:
# --- Setup: make the EPCAPE package importable -------------------------------
# The EPCAPE folder is itself the Python package (config.yaml at its top level;
# shared code in download_data/, analysis_tools/, plotting/), so its PARENT folder goes on sys.path.
import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Walk up from the notebook's folder to the EPCAPE folder (holds config.yaml and __init__.py)."""
    for p in [start, *start.parents]:
        if (p / "config.yaml").is_file() and (p / "__init__.py").is_file():
            return p
    raise FileNotFoundError(f"Could not find the EPCAPE folder above {start}")


REPO = find_repo_root()
if str(REPO.parent) not in sys.path:
    sys.path.insert(0, str(REPO.parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from EPCAPE.analysis_tools import qc, units
from EPCAPE.download_data.config import active_machine
from EPCAPE.analysis_tools.derived import git_revision
from EPCAPE.plotting import INK, INK_2, INK_3, use_style
from EPCAPE.analysis_tools.products import load_product
from EPCAPE.comparisons.seasonal_averages import kavin as K
from EPCAPE.comparisons.seasonal_averages import quantities as Q
from EPCAPE.comparisons.seasonal_averages import seasonal as S
from EPCAPE.comparisons.seasonal_averages import sources as SRC

use_style()
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 90)
MACHINE = active_machine()
GIT_REVISION = git_revision()
print(f"EPCAPE folder: {REPO}\nMachine: {MACHINE.name}   data folder: {MACHINE.data_root}\ngit revision: {GIT_REVISION}")

EPCAPE folder: /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE
Machine: local   data folder: /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data
git revision: 7ac767f-dirty


In [2]:
# --- Settings: every choice that changes a number is set here ---------------
TABLE_XLSX = Path(os.environ.get("EPCAPE_SEASONAL_TABLE", "/Users/andrewbuggee/Documents/Scripps - UCSD/EPCAPE/Lynn's BAMS paper/Table-EPCAPE_SeasonalAverages.xlsx")).expanduser()
# Kavin's input files and the MATLAB live script that reads them were shared on 2026-10-07; Sections 1b, 2b,
# 3b, 9b and 9c re-run that code on these files (kavin.py). The folder is git-ignored.
KAVIN_DIR = Path(os.environ.get("EPCAPE_KAVIN_DIR", REPO / "Kavin_data_for_BAMS_paper")).expanduser()

# Averaging windows for the radiation rows, in MINUTES: any non-negative length (fractions allowed).
# 0 = no further averaging: the statistics are over every RADFLUX value that passes QC = 0.
# RADFLUX values are already 1-min means of 1-s SKYRAD samples, so 0 and 1 give the same result.
AVG_WINDOW_SW_MIN = 5.0  # SW transmittance (Kavin: 5-min for SW)
AVG_WINDOW_LW_MIN = 0  # downwelling LW (Kavin: 5-min for LW)
AVG_PERIOD_CBH = 0  # cloud base (ceilometer) averaging window in MINUTES (any number >= 0);
#                    0 = no time averaging: every cloudy, QC = 0 16-s sample enters the statistics.
AVG_PERIOD_CT = 0  # cloud top (KAZR-ARSCL) averaging window in MINUTES (any number >= 0);
#                   0 = no time averaging: every 4-s low-cloud sample enters the statistics.
CT_CLOUD_SOURCES = None  # cloud top: ARSCL cloud_source_flag codes allowed at the top gate; None = any.
#   0 no radar+lidar data, 1 clear, 2 radar+lidar, 3 radar only, 4 lidar only,
#   5 radar (lidar missing), 6 lidar (radar missing). e.g. (2, 3, 5) = tops the radar saw.
#   Needs the opt-in ~10 GB download: download_data.py --only cloud_source_arscl_M1
AVG_PERIOD_LWP = 0  # LWP averaging window in MINUTES (any number >= 0); each value = mean of the cloudy samples in it.
#                    0 = no time averaging: every sample that passes the QC, cloud and rain filters enters the statistics.


SZA_MAX_DEG = 80.0  # SW transmittance: daytime = solar zenith angle <= 80 deg (ratio is noisy near the horizon)

LOW_CLOUD_MAX_M = 3500.0  # "low cloud": top (KAZR) or base (ceilometer) <= 3.5 km, the limit for low cloud classification

LWP_RAIN_WINDOW_MIN = 30.0  # MWR LWP: "rain-free" = no rain within ±30 min at the Pier Parsivel (wet radome)

RAIN_THRESHOLD_MM_H = 0.1  # a 1-min disdrometer sample is "raining" at >= 0.1 mm/h
RAIN_MAX_GAP_MIN = 60.0  # raining samples closer than 60 min belong to one rain event
RAIN_MIN_MM = 0.1  # minimum accumulation of a rain event / rain day / wet hour

GCVI_MIN_SEGMENT_MIN = 15.0  # "minimum sampling 15 min" (sheet rows 33-34)

# FM-120 cloud / haze classification of each 5-min interval (sheet rows 31-32), with the
# operational definitions of Lynn Russell's EPCAPE BAMS paper: haze = "low visibility (<5 km)
# but low liquid water content (<0.01 g m-3)"; cloud = "visibility <1 km and liquid water
# >0.01 g m-3". Visibility = Mt. Soledad record (config.yaml fm120.visibility_files).
#   cloud = visibility < FM120_CLOUD_VIS_MAX_M AND FM-120 LWC > FM120_CLOUD_LWC_MIN_GM3
#   haze  = visibility < FM120_HAZE_VIS_MAX_M  AND FM-120 LWC < FM120_HAZE_LWC_MAX_GM3 (and not cloud)
FM120_CLOUD_VIS_MAX_M = 1000.0  # cloud: visibility < 1 km ...
FM120_CLOUD_LWC_MIN_GM3 = 0.01  # ... AND liquid water content > 0.01 g m-3
FM120_HAZE_VIS_MAX_M = 5000.0  # haze: visibility < 5 km ...
FM120_HAZE_LWC_MAX_GM3 = 0.01  # ... AND liquid water content < 0.01 g m-3
FM120_VIS_STAT = "median"  # visibility of a 5-min interval from its ~1-s samples: "median" (the condition
#                            held for more than half the interval) or "mean" (the 5-min average, like the
#                            FM-120's own LWC; pulled up by clear-air seconds)
FM120_VIS_MIN_SAMPLES = 125  # an interval needs >= this many valid 1-s visibility samples (a full one has ~250)
# Andrew's filter set 2 (2026-10-08), shown next to the BAMS-paper definitions for rows 31-32:
#   cloud = visibility < FM120_CLOUD_VIS_MAX_M and LWC > FM120_CLOUD_LWC_MIN_GM3 (unchanged)
#   haze  = (FM120_CLOUD_VIS_MAX_M < visibility < FM120_HAZE2_VIS_MAX_M and LWC < FM120_HAZE2_LWC_MAX_GM3)
#           or (visibility < FM120_CLOUD_VIS_MAX_M and LWC < FM120_HAZE2_LWC_MAX_GM3)
FM120_HAZE2_VIS_MAX_M = 5000.0  # haze, set 2: visibility < 5 km ... (the same upper limit as the BAMS-paper haze)
FM120_HAZE2_LWC_MAX_GM3 = 0.01  # ... and LWC < 0.01 g m-3 (in both of its visibility ranges)
CLOSE_PCT = 10.0  # reading aid only: flags |ours - sheet| <= 10 % of the sheet's mean
OUT_DIR = MACHINE.processed_dir() / "derived" / "seasonal_averages_check"

SETTINGS = {
    k: (str(v) if isinstance(v, Path) else v)
    for k, v in dict(globals()).items()
    if k.isupper() and k not in ("REPO", "MACHINE", "GIT_REVISION", "SETTINGS")
}
SETTINGS

{'INK': '#0b0b0b',
 'INK_2': '#52514e',
 'INK_3': '#8a8984',
 'K': <module 'EPCAPE.comparisons.seasonal_averages.kavin' from '/Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/comparisons/seasonal_averages/kavin.py'>,
 'Q': <module 'EPCAPE.comparisons.seasonal_averages.quantities' from '/Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/comparisons/seasonal_averages/quantities.py'>,
 'S': <module 'EPCAPE.comparisons.seasonal_averages.seasonal' from '/Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/comparisons/seasonal_averages/seasonal.py'>,
 'SRC': <module 'EPCAPE.comparisons.seasonal_averages.sources' from '/Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/comparisons/seasonal_averages/sources.py'>,
 'TABLE_XLSX': "/Users/andrewbuggee/Documents/Scripps - UCSD/EPCAPE/Lynn's BAMS paper/Table-EPCAPE_SeasonalAverages.xlsx",
 'KAVIN_DIR': '/Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/Kavin_data_for_BAMS_paper',
 'AVG_WINDOW_SW_MI

## The sheet's rows attributed to Kavin

In [3]:
# =============================================================================
# The sheet's values: every row whose Author(s) column (J) names Kavin
# =============================================================================
# Input: TABLE_XLSX (set in the Settings cell), sheet "Cloud&MetQuantities".
# S.read_table_rows parses each season's text cell ("0.58 ± 0.29", "1.0 * 10^2 ± 66",
# "626 + 297", "2.8 ± n/a", ...) into <season>_mean, <season>_std, keeps the original
# text as <season>_raw, and records any assumption it made in <season>_note.
# Seasons = columns E-I: EPCAPE (whole campaign), Spring, Summer, Fall, Winter.
TABLE = S.read_table_rows(TABLE_XLSX)
print(f"{len(TABLE)} rows attributed to Kavin in {TABLE_XLSX.name} (sheet Cloud&MetQuantities)")
# Show the descriptive columns and the cell text exactly as typed in the sheet
display(TABLE[["quantity", "units", "location", "method"] + [f"{s}_raw" for s in S.SEASON_ORDER]])

30 rows attributed to Kavin in Table-EPCAPE_SeasonalAverages.xlsx (sheet Cloud&MetQuantities)


,quantity,units,location,method,EPCAPE_raw,Spring_raw,Summer_raw,Fall_raw,Winter_raw
sheet_row,,,,,,,,,
7,Shortwave (SW) Transmittance,%,Soledad,"SKYRAD, RADFLUX1LONG",0.58 ± 0.29,0.50 ± 0.31,0.56 ± 0.28,0.66 ± 0.24,0.61 ± 0.28
8,Longwave (LW) Flux,W/m2,Soledad,From Lubin,350 ± 30,370 ± 10,380 ± 10,323 ± 5,308 ± 3
10,Liquid Water Path,g/m2,Pier,From Virendra (?),56 ± 127,92 ± 161,59 ± 118,42 ± 100,33 ± 92
15,Cloud Top Height,m,Pier,KAZR,920 ± 380,890 ± 330,774 ± 353,1057 ± 416,1192 ± 397
17,Boundary Layer Height,m,Pier,PBLH-Thermo (Zhang),899 ± 305,886 ± 256,764 ± 206,1014 ± 343,1124 ± 442
18,Boundary Layer Height,m,Pier,PBLH-MPL,790 ±.274,765 ± 255,728 ± 248,844 ± 313,970 ± 288
19,Boundary Layer Height,m,Pier,BL-Depth (Miller),703 ± 275,796 ± 255,679 ± 261,626 + 297,649 ± 272
20,Lifting Condensation Level,m,Pier,Radiosondes,637 ± 483,490 ± 261,361 ± 165,917 ± 676,787 ± 428
21,Cloud Base Height,m,Pier,Ceilometer,661 ± 316,619 ± 270,554 ± 254,836 ± 389,929 ± 362


In [4]:
# --- Bookkeeping shared by every section --------------------------------------
# Every section appends to these three containers; Section 11 turns them into
# the summary table, the QC table and the list of rows that could not be checked.
SUMMARY = []  # one comparison table per (sheet row, definition); concatenated in Section 11
QC_INFO = []  # one dict per screened variable: samples before / after QC = 0
NOT_CHECKED = {}  # sheet row -> why it could not be re-derived from available data


def check_stats(sheet_row, stats, definition, sample, *, primary=False, show=True, reference=None, quantity=None):
    """Seasonal statistics that are already computed, next to the sheet's (or another reference's) values.

    Inputs
      stats : DataFrame with columns mean, std, median, n and rows S.SEASON_ORDER; for
              example Kavin's own averaging re-run in kavin.py (Sections 1b-9c), whose
              seasons and weighting differ from S.seasonal_stats
      the rest as in check()
    Returns the comparison table (one row per season; also appended to SUMMARY)."""
    stats = stats[["mean", "std", "median", "n"]]
    # side by side with the sheet (or the given reference); diff_pct = 100 (ours - sheet) / sheet
    if reference is None:
        cmp = S.compare(stats, TABLE, sheet_row, definition, close_pct=CLOSE_PCT)
    else:
        cmp = S.compare_to(stats, reference, sheet_row, quantity, definition, close_pct=CLOSE_PCT)
    cmp["sample"] = sample
    cmp["primary"] = primary
    SUMMARY.append(cmp)
    if show:
        tag = "" if primary else "  *(sensitivity test)*"
        display(Markdown(f"**Row {sheet_row}: {definition}**{tag}  \nOne sample = {sample}."))
        display(S.show(cmp))
    return cmp


def check(sheet_row, values, definition, sample, *, primary=True, show=True, reference=None, quantity=None):
    """Seasonal mean ± std of `values` next to the sheet's (or another reference's) values.

    Inputs
      sheet_row  : spreadsheet row number (or a label like "35+39" for combined rows)
      values     : pandas Series indexed by UTC time; NaN samples are ignored
      definition : one-line description of how `values` were made (shown and saved)
      sample     : what ONE value in `values` is (e.g. "5-min mean", "radiosonde launch")
      primary    : True for the definition carried into the summary table and figure;
                   False for sensitivity tests
      show       : display the comparison table here
      reference  : compare against this DataFrame (mean, std, raw per season) instead
                   of the sheet row, e.g. a combination of several rows
      quantity   : row label to use with `reference`
    Returns the comparison table (one row per season; also appended to SUMMARY)."""
    # mean, std (ddof = 1), median and count per season (seasons defined in seasonal.py)
    return check_stats(sheet_row, S.seasonal_stats(values), definition, sample, primary=primary, show=show,
                       reference=reference, quantity=quantity)


def compare_candidates(sheet_row, candidates, *, label="", mark_primary=True):
    """Several readings of an undocumented definition, ranked by how well each reproduces the sheet.

    Inputs
      sheet_row    : spreadsheet row number
      candidates   : {name: (Series of values, what one value is)}
      label        : prefix for every definition name (e.g. the instrument)
      mark_primary : mark the best-ranked candidate as primary (the rain cell
                     sets this False and chooses one definition across rows)
    Ranking (seasonal.closest_definition): median |% difference| of the means over
    the five columns, with the median |log2 std ratio| as a second indicator.
    Returns (ranking, comparisons)."""
    # one comparison per candidate, kept out of the display (shown together below)
    cmps = [
        check(sheet_row, v, f"{label}{name}", sample, primary=False, show=False)
        for name, (v, sample) in candidates.items()
    ]
    ranking = S.closest_definition(cmps)
    if mark_primary:
        best = ranking["definition"].iloc[0]
        for c in cmps:
            c["primary"] = c["definition"].iloc[0] == best
    # one line per candidate ("mean ± std" per season) plus the sheet's own line
    wide = pd.concat({c["definition"].iloc[0]: c.set_index("season")["ours"] for c in cmps}, axis=1).T
    wide.loc["SHEET (Kavin)"] = cmps[0].set_index("season")["table"]
    display(wide)
    display(ranking.round(2))
    return ranking, cmps

## 0. Consistency checks on the sheet itself (no data needed)

In [5]:
# =============================================================================
# Section 0: internal consistency of the sheet (uses only TABLE, no data)
# =============================================================================
# Each check adds one row to `issues`: which sheet row, what was checked, what was
# found, and ok = True/False. Only the failures are displayed.
issues = []
# (a) Sampling hours: the four seasons should add up to the campaign total.
#     Rows 33 (cloud) and 34 (haze) are hours; a 2 % tolerance allows for rounding.
for r in (33, 34):
    seas = [TABLE.loc[r, f"{s}_mean"] for s in ("Spring", "Summer", "Fall", "Winter")]
    total = TABLE.loc[r, "EPCAPE_mean"]
    issues.append({
        "sheet_row": r, "check": "seasonal hours add up to the campaign total",
        "found": f"{' + '.join(f'{v:g}' for v in seas)} = {sum(seas):g}  vs  {total:g}",
        "ok": bool(abs(sum(seas) - total) <= 0.02 * total),
    })
# (b) Rows labelled "Mass Fraction": four species' fractions cannot add up to more than 1.
#     35-38 = cloud-residual sulfate, nitrate, ammonium, organics; 39-42 = the haze residuals.
for rows, kind in (((35, 36, 37, 38), "cloud residual"), ((39, 40, 41, 42), "haze residual")):
    for s in S.SEASON_ORDER:
        tot = sum(TABLE.loc[r, f"{s}_mean"] for r in rows)
        issues.append({
            "sheet_row": f"{rows[0]}-{rows[-1]}", "check": f"{kind} 'mass fractions' sum to <= 1 ({s})",
            "found": f"sum = {tot:.2f}", "ok": bool(tot <= 1.0),
        })
# (c) Cells that needed an assumption to read (the parser's note, e.g. "+" read as "±"),
#     or whose std is under 1 % of the mean (likely a typo, e.g. "790 ±.274").
for r in TABLE.index:
    for s in S.SEASON_ORDER:
        raw, m, sd, note = (TABLE.loc[r, f"{s}_{k}"] for k in ("raw", "mean", "std", "note"))
        if note and note not in ("empty", "no ± given", "n/a", "std n/a"):
            issues.append({"sheet_row": r, "check": f"cell {S.SEASON_COLUMNS[s]}{r} reads cleanly",
                           "found": f"{raw!r}: {note}", "ok": False})
        if np.isfinite(sd) and np.isfinite(m) and m > 0 and sd < 0.01 * m:
            issues.append({"sheet_row": r, "check": f"cell {S.SEASON_COLUMNS[s]}{r}: std >= 1 % of mean",
                           "found": f"{raw!r}", "ok": False})
ISSUES = pd.DataFrame(issues)
display(ISSUES[~ISSUES["ok"]])  # failures only

,sheet_row,check,found,ok
1,34,seasonal hours add up to the campaign total,52 + 101 + 500 + 35 = 688 vs 236,False
2,35-38,cloud residual 'mass fractions' sum to <= 1 (EPCAPE),sum = 1.64,False
3,35-38,cloud residual 'mass fractions' sum to <= 1 (Spring),sum = 1.16,False
4,35-38,cloud residual 'mass fractions' sum to <= 1 (Summer),sum = 1.83,False
5,35-38,cloud residual 'mass fractions' sum to <= 1 (Fall),sum = 1.93,False
10,39-42,haze residual 'mass fractions' sum to <= 1 (Fall),sum = 1.01,False
12,8,cell I8: std >= 1 % of mean,'308 ± 3',False
13,18,cell E18: std >= 1 % of mean,'790 ±.274',False
14,19,cell H19 reads cleanly,'626 + 297': '+' read as '±',False


Other things noticed while reading the sheet. These don't need data, but they should be fixed before the table is published:

* **Rows 35–42 are labelled "Mass Fraction"**, but each season's four values add up to more than 1, as listed above. In the cell comment on E35 Lynn asked for **mass concentrations** in the table. The values are concentrations (µg m⁻³), and the labels and the missing units should say so.
* **H34 "5.0\*10^2" h** of fall haze sampling cannot be right. The seasons would then sum to 688 h against E34 = 236 h, and all GCVI sampling in Oct–Dec 2023 amounts to only ~110 h (Section 10). E34 − (F34 + G34 + I34) = 48 h, so the value is presumably 5.0×10¹. **Confirmed:** Kavin's code gives 49.5 h (Section 10e).
* **Units:** B7 says "%" but the values are fractions (0–1). B25 and B28 say m⁻³, but ~200 is cm⁻³. Rows 26 and 29 (effective diameter) and 35–42 have no units. B55 reads "g/^#3".
* **Typos in values:** E18 "790 ±.274" (presumably ± 274) and H19 "626 + 297" (presumably ±).
* **Rows 22–24 ("Rain Accumulation", mm/hr):** an accumulation is in mm, a rate in mm/hr. Section 8 shows which one the numbers are. "LDS1" refers to a facility that does not exist in ARM's EPCAPE data: the disdrometers were at M1 (Scripps Pier) and S2 (Mt. Soledad).
* **Rows 7–8 say "Soledad"**, but ARM's SKYRAD and RADFLUX data for EPCAPE exist only for M1, the Scripps Pier. ARM Live lists no `epcskyrad*S2`, `epcradflux*S2` or `epcgndrad*S2` files.

## 1. Shortwave transmittance (row 7)

Kavin's note describes the ratio of 5-min-averaged downwelling SW from SKYRAD (`radflux1long` dataset) to an *idealized clear-sky analytical model (Atwater and Ball 1981)*.
Here the numerator is the same quantity: RADFLUX `downwelling_shortwave`, which is the SKYRAD measurement, with QC = 0.
The denominator is **RADFLUX's own clear-sky estimate** (`clearsky_downwelling_shortwave`). That is an empirical fit to this site's clear-sky periods (Long & Ackerman 2000, *JGR* 105, 15609), not the Atwater & Ball model. I could not verify the Atwater & Ball equations from an accessible source, so I did not implement them.
Clear-sky models typically differ by a few percent, which shifts the ratio by about as much.
A sensitivity test below divides by top-of-atmosphere irradiance instead. Together the two bracket how bright the sheet's clear-sky model must be.
`clearsky_downwelling_shortwave` has no QC field. Both numerator and denominator are averaged over the same valid 1-min samples. Only samples with SZA ≤ `SZA_MAX_DEG` are used.

In [6]:
# =============================================================================
# Section 1: SW transmittance (sheet row 7)
# =============================================================================
# Inputs
#   radflux_M1 = epcradflux1longM1.c2 (RADFLUX VAP, Scripps Pier), 1-min values. Each
#   value is a 1-min MEAN of 1-s SKYRAD samples, time-stamped at the END of the minute.
#     downwelling_shortwave           measured SW down (W m-2); qc_ field -> QC = 0
#     clearsky_downwelling_shortwave  RADFLUX's fitted clear-sky SW down (Long & Ackerman
#                                     2000); no qc_ field, used as distributed
#     cosine_zenith                   cosine of the solar zenith angle (geometry, no QC)
#   SZA_MAX_DEG        daytime cut: only samples with solar zenith angle <= 80 deg
#   AVG_WINDOW_SW_MIN  averaging window in minutes; 0 = every 1-min value on its own
# Q.sw_transmittance averages numerator and denominator over the SAME valid samples,
# then takes the ratio; values > 1 (cloud-edge enhancement) are kept.

# --- 1. Load the campaign RADFLUX file (also used for LW in Section 2) ---------------
rad = load_product("radflux_M1")

# --- 2. Transmittance and its comparison with the sheet ------------------------------
sw_t, info = Q.sw_transmittance(rad, sza_max_deg=SZA_MAX_DEG, window_min=AVG_WINDOW_SW_MIN)
QC_INFO.extend(info["qc"])  # feeds the QC table in Section 11
check(7, sw_t,
      f"SW↓ / RADFLUX clear-sky SW↓, {Q.describe_window(AVG_WINDOW_SW_MIN)}, SZA ≤ {SZA_MAX_DEG:g}°, QC = 0 (Scripps Pier)",
      f"{Q.describe_window(AVG_WINDOW_SW_MIN)} (daytime)")

# --- 3. Optional sensitivity tests (commented out; uncomment to run) -----------------
# 3a. How much does the daytime cut move the means?
# sens = {f"SZA ≤ {z}°": S.seasonal_stats(Q.sw_transmittance(rad, sza_max_deg=z, window_min=AVG_WINDOW_SW_MIN)[0])["mean"]
#         for z in (70, 80, 85)}
# display(Markdown("Seasonal mean transmittance for three daytime cuts:"))
# display(pd.DataFrame(sens).T.round(3))
# # Clear-sky ratios scatter around 1.00, so only > 1.05 is counted as cloud-edge enhancement.
# print(f"5-min ratios > 1.05 (cloud-edge enhancement): {100 * (sw_t.dropna() > 1.05).mean():.1f} %")

# 3b. Bracketing the denominator: SW / top-of-atmosphere SW (clearness index). If the sheet's
# numerator is the same SKYRAD measurement, sheet / clearness index = how bright Kavin's
# clear-sky model is relative to TOA, to set next to RADFLUX's clear-sky / TOA.
# kt, _ = Q.sw_clearness_index(rad, sza_max_deg=SZA_MAX_DEG, window_min=AVG_WINDOW_SW_MIN)
# kt_cmp = check(7, kt, "SW↓ / TOA SW↓ (clearness index; NOT the sheet's definition)", "5-min mean (daytime)",
#                primary=False)
# cs_over_toa = S.seasonal_stats(
#     Q.sw_clearness_index(rad.assign(downwelling_shortwave=rad["clearsky_downwelling_shortwave"]).drop_vars(
#         "qc_downwelling_shortwave", errors="ignore"), sza_max_deg=SZA_MAX_DEG, window_min=AVG_WINDOW_SW_MIN)[0])["mean"]
# implied = kt_cmp.set_index("season")["ours_mean"] / kt_cmp.set_index("season")["table_mean"]
# display(pd.DataFrame({"RADFLUX clear-sky / TOA": cs_over_toa,
#                       "implied sheet clear-sky / TOA": implied}).T.round(3))

**Row 7: SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)**  
One sample = 5-min mean (daytime).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.71 ± 0.32,45458,0.58 ± 0.29,21.9,1.10,False
Spring,0.61 ± 0.34,13048,0.50 ± 0.31,22.3,1.08,False
Summer,0.67 ± 0.31,12755,0.56 ± 0.28,19.0,1.10,False
Fall,0.84 ± 0.25,9706,0.66 ± 0.24,27.1,1.04,False
Winter,0.76 ± 0.32,9949,0.61 ± 0.28,23.9,1.13,False


,sheet_row,quantity,definition,season,ours_mean,ours_std,n,table_mean,table_std,ours,table,diff_pct,std_ratio,close,sample,primary
0,7,Shortwave (SW) Transmittance,"SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)",EPCAPE,0.706948,0.318808,45458,0.58,0.29,0.71 ± 0.32,0.58 ± 0.29,21.887553,1.099337,False,5-min mean (daytime),True
1,7,Shortwave (SW) Transmittance,"SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)",Spring,0.611362,0.336321,13048,0.50,0.31,0.61 ± 0.34,0.50 ± 0.31,22.272307,1.084907,False,5-min mean (daytime),True
2,7,Shortwave (SW) Transmittance,"SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)",Summer,0.666395,0.308537,12755,0.56,0.28,0.67 ± 0.31,0.56 ± 0.28,18.999179,1.101918,False,5-min mean (daytime),True
3,7,Shortwave (SW) Transmittance,"SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)",Fall,0.838567,0.248715,9706,0.66,0.24,0.84 ± 0.25,0.66 ± 0.24,27.055547,1.036312,False,5-min mean (daytime),True
4,7,Shortwave (SW) Transmittance,"SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)",Winter,0.755894,0.316701,9949,0.61,0.28,0.76 ± 0.32,0.61 ± 0.28,23.917012,1.131075,False,5-min mean (daytime),True


### 1b. Kavin's data and code: Dan Lubin's hourly transmittances

Kavin's row-7 values come from Dan Lubin's tables, `<MON>_SW_Boxplot.csv`. Each holds one transmittance per day of the month and hour of day, with hours in **PST (UTC−8)**, and his live script (`ForAndrewVariablesmlx.mlx`, section "Dan LW SW Code") averages them. `kavin.py` re-runs that code line by line:

* MATLAB's `readtable` reads the files' header line as a data row, so `rows 2:31` are **days 1–30**. Day 31 of the seven 31-day months is dropped.
* `Var2:Var24` are **hours 1–23**: hour 0 (night) is dropped, and a column of NaN is added.
* Day 1 of every file holds **−0.05 placeholders** in the night hours. They are kept, so they enter the means.
* A season is the **mean and std of every (day, hour) cell** of its three stacked months. Each daylight hour counts once, whatever the sun elevation. Seasons are **calendar quarters**: winter = Jan–Mar, not the sheet's FM23 + JF24.

Lubin's files for JAN, FEB, MAR and JUN are not in the folder, so only **summer and fall** can be reproduced. The other seasons are listed with the months available.

**Matched hours.** Putting each Lubin cell next to RADFLUX's hourly transmittance for the same UTC hour (measured / RADFLUX clear-sky, over the same QC = 0 minutes, no sun-elevation cut) separates two things: the **clear-sky denominator** (Atwater & Ball vs RADFLUX's fitted clear sky) and the **averaging** (every daylight hour vs 5-min values with SZA ≤ 80°). On hours that RADFLUX finds clear (measured within ±2 % of its clear-sky estimate), a clear-sky model that fits the sky would give a transmittance near 1.

In [7]:
# =============================================================================
# Section 1b: SW transmittance from Kavin's input files (Dan Lubin's hourly tables)
# =============================================================================
# Inputs
#   KAVIN_DIR/<MON>_SW_Boxplot.csv  Dan Lubin's hourly SW transmittance (K.read_lubin_sw): rows = day of
#                                  month, columns = hour of day in PST (UTC-8); day 1 holds -0.05
#                                  placeholders at night; values are measured SW / a clear-sky model
#   rad (Section 1)                RADFLUX at the Pier, for the matched-hour comparison
# Kavin's code stacks the cells of each season's three months (days 1-30, hours 1-23) and takes
# their mean and sample std (K.kavin_sw_stats); seasons = calendar quarters.

# --- 1. Read Lubin's tables; which months are here? -------------------------------------
LUBIN_SW = K.read_lubin_sw(KAVIN_DIR)
sw_missing = [K.MONTH_ABBR[m - 1] for m in range(1, 13) if m not in LUBIN_SW]
print(f"Lubin SW tables present: {', '.join(K.MONTH_ABBR[m - 1] for m in sorted(LUBIN_SW))}; "
      f"missing: {', '.join(sw_missing) or 'none'}")

# --- 2. Kavin's code on Kavin's data: reproduces the sheet where every month is present ----
#        (a season with a missing month shows NaN: it cannot be reproduced)
sw_kavin = K.kavin_sw_stats(LUBIN_SW, exact=True)
check_stats(7, sw_kavin, "Lubin's hourly transmittance, Kavin's code: days 1–30, hours 1–23 PST, day-1 −0.05 "
            "placeholders kept; calendar quarters", "one hour of one day (a cell of Lubin's table)")

# --- 3. Same data, corrected: every day and hour, placeholders removed ------------------------
sw_lubin = K.kavin_sw_stats(LUBIN_SW, exact=False)
check_stats(7, sw_lubin, "Lubin's hourly transmittance, every day and hour, placeholders removed; calendar quarters",
            "one hour of one day (a cell of Lubin's table)")

# --- 4. Seasons with a missing month: what the available months give ---------------------------
display(Markdown("Available months only. These are not comparable with the sheet; they show what the missing files leave out:"))
display(pd.DataFrame({
    "Kavin's code": [S.format_pm(m, s) for m, s in zip(sw_kavin["partial_mean"], sw_kavin["partial_std"])],
    "corrected": [S.format_pm(m, s) for m, s in zip(sw_lubin["partial_mean"], sw_lubin["partial_std"])],
    "months missing": sw_kavin["months_missing"],
}, index=S.SEASON_ORDER).T)

# --- 5. Matched hours: Lubin vs RADFLUX, hour by hour -----------------------------------------
# RADFLUX time stamps mark the END of each minute: shift by 1 min so each hourly bin [h, h+1)
# holds the minutes inside it. One RADFLUX transmittance per hour = <SW> / <clear-sky SW> over
# the same QC = 0 minutes, with no sun-elevation cut (Lubin's tables keep every daylight hour).
sw_1min = Q.to_series(Q.qc0_values(rad, "downwelling_shortwave")[0])
cs_1min = Q.to_series(Q.qc0_values(rad, "clearsky_downwelling_shortwave")[0])
mu0_1min = Q.to_series(rad["cosine_zenith"])
for series_1min in (sw_1min, cs_1min, mu0_1min):
    series_1min.index = series_1min.index - pd.Timedelta(minutes=1)
rad_t_hourly = K.hourly_ratio(sw_1min, cs_1min)  # needs >= 30 usable minutes in the hour
mu0_hourly = mu0_1min.resample("1h").mean()  # mean cosine of the solar zenith angle in the hour
del sw_1min, cs_1min, mu0_1min

# 5a. Which time zone are Lubin's hours in? Correlation with RADFLUX for several UTC offsets.
lag = {}
for offset_h in (0, 7, 8, 9):
    pair = pd.concat({"lubin": K.lubin_sw_series(LUBIN_SW, utc_offset_h=offset_h), "radflux": rad_t_hourly},
                     axis=1).dropna()
    lag[f"Lubin hour + {offset_h} h = UTC"] = {"matched hours": len(pair), "correlation r": pair.corr().iloc[0, 1]}
display(Markdown("Lubin's hour labels vs RADFLUX: the correlation peaks at +8 h, i.e. Pacific Standard Time"))
display(pd.DataFrame(lag).T.round(3))

# 5b. With PST hours: the same hours side by side
matched = pd.concat({"lubin": K.lubin_sw_series(LUBIN_SW), "radflux": rad_t_hourly, "mu0": mu0_hourly},
                    axis=1).dropna(subset=["lubin", "radflux"])
sw_ratio = matched["radflux"] / matched["lubin"]
print(f"Matched hours: {len(matched):,}.  RADFLUX / Lubin transmittance: median {sw_ratio.median():.3f} "
      f"(interquartile {sw_ratio.quantile(0.25):.3f}-{sw_ratio.quantile(0.75):.3f})")

# 5c. Clear hours (RADFLUX measured SW within ±2 % of its clear-sky estimate): what does Lubin's
#     transmittance read, by sun elevation (mean cosine of the zenith angle in the hour)?
clear_hours = matched[(matched["radflux"] > 0.98) & (matched["radflux"] < 1.02)]
mu0_bins = pd.cut(clear_hours["mu0"], [0.0, 0.3, 0.5, 0.7, 1.0])
display(Markdown("Hours RADFLUX finds clear: Lubin's transmittance by sun elevation (a clear-sky model that fits the sky would read ~1):"))
display(clear_hours.groupby(mu0_bins, observed=True).agg(
    hours=("lubin", "size"), lubin_median=("lubin", "median"), radflux_median=("radflux", "median")).round(3))

# 5d. Step by step from the sheet to this check, seasonal means (calendar quarters, PST months).
#     Only summer and fall have every month in Lubin's files.
SW_STEPS = pd.DataFrame({
    "sheet (row 7)": S.table_values(TABLE, 7)["mean"],
    "Kavin's code on Lubin's tables": sw_kavin["mean"],
    "same tables, corrected": sw_lubin["mean"],
    "Lubin, matched hours only": K.quarter_stats(matched["lubin"])["mean"],
    "RADFLUX on the same hours": K.quarter_stats(matched["radflux"])["mean"],
    "this check (Section 1: 5-min, SZA ≤ 80°, sheet seasons)": S.seasonal_stats(sw_t)["mean"],
}).T
display(Markdown("**SW transmittance, seasonal means step by step** (EPCAPE and spring use only the months present)"))
display(SW_STEPS[S.SEASON_ORDER].round(3))

Lubin SW tables present: APR, MAY, JUL, AUG, SEP, OCT, NOV, DEC; missing: JAN, FEB, MAR, JUN


**Row 7: Lubin's hourly transmittance, Kavin's code: days 1–30, hours 1–23 PST, day-1 −0.05 placeholders kept; calendar quarters**  *(sensitivity test)*  
One sample = one hour of one day (a cell of Lubin's table).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,–,0,0.58 ± 0.29,NaN,NaN,False
Spring,–,0,0.50 ± 0.31,NaN,NaN,False
Summer,0.56 ± 0.28,1270,0.56 ± 0.28,-0.7,1.00,True
Fall,0.66 ± 0.24,1054,0.66 ± 0.24,-0.4,1.01,True
Winter,–,0,0.61 ± 0.28,NaN,NaN,False


**Row 7: Lubin's hourly transmittance, every day and hour, placeholders removed; calendar quarters**  *(sensitivity test)*  
One sample = one hour of one day (a cell of Lubin's table).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,–,0,0.58 ± 0.29,NaN,NaN,False
Spring,–,0,0.50 ± 0.31,NaN,NaN,False
Summer,0.56 ± 0.27,1273,0.56 ± 0.28,0.6,0.97,True
Fall,0.68 ± 0.21,1041,0.66 ± 0.24,3.0,0.88,True
Winter,–,0,0.61 ± 0.28,NaN,NaN,False


...

Matched hours: 3,087.  RADFLUX / Lubin transmittance: median 1.107 (interquartile 1.065-1.175)


Hours RADFLUX finds clear: Lubin's transmittance by sun elevation (a clear-sky model that fits the sky would read ~1):

,hours,lubin_median,radflux_median
mu0,,,
"(0.0, 0.3]",74,0.767,0.996
"(0.3, 0.5]",148,0.856,1.000
"(0.5, 0.7]",247,0.898,1.000
"(0.7, 1.0]",271,0.937,0.998


**SW transmittance, seasonal means step by step** (EPCAPE and spring use only the months present)

,EPCAPE,Spring,Summer,Fall,Winter
sheet (row 7),0.580,0.500,0.560,0.660,0.610
Kavin's code on Lubin's tables,NaN,NaN,0.556,0.657,NaN
"same tables, corrected",NaN,NaN,0.563,0.680,NaN
"Lubin, matched hours only",0.593,0.514,0.570,0.689,NaN
RADFLUX on the same hours,0.678,0.573,0.637,0.818,NaN
"this check (Section 1: 5-min, SZA ≤ 80°, sheet seasons)",0.707,0.611,0.666,0.839,0.756


## 2. Downwelling longwave (row 8)

The sheet's source is "From Lubin", at Soledad. The ARM data are the RADFLUX/SKYRAD pyrgeometers **at the Scripps Pier**. Section 2b shows that Lubin's LW tables *are* the Pier record: his monthly means agree with RADFLUX at the Pier within 0.4 W m⁻². So the sheet's location is wrong and this is the same instrument.
The sheet's standard deviations (3–10 W m⁻² within a season) are far smaller than the spread of the 1-min values. Section 2b shows that they are the hour-to-hour spread of the median diurnal cycle. The second table here shows monthly means for comparison.

**What is the measurement uncertainty of the pyrgeometer? Should be incorporated when rounding**

In [8]:
# =============================================================================
# Section 2: downwelling longwave (sheet row 8)
# =============================================================================
# Inputs
#   rad (Section 1) = radflux_M1, variable downwelling_longwave (W m-2): 1-min means of
#                     1-s SKYRAD pyrgeometer samples at the Scripps Pier; qc_ field -> QC = 0.
#                     The sheet's value is "From Lubin" at Soledad, so this is a
#                     comparison with another instrument ~250 m lower, not a reproduction.
#   AVG_WINDOW_LW_MIN averaging window in minutes; 0 = every 1-min value on its own
# Day and night are both used.

# --- 1. LW down, QC = 0, averaged over AVG_WINDOW_LW_MIN --------------------------------
lw, info = Q.lw_down(rad, window_min=AVG_WINDOW_LW_MIN)
QC_INFO.extend(info["qc"])  # feeds the QC table in Section 11
check(8, lw, f"RADFLUX downwelling LW, {Q.describe_window(AVG_WINDOW_LW_MIN)}, QC = 0 (Scripps Pier, not Soledad)",
      Q.describe_window(AVG_WINDOW_LW_MIN))

# --- 2. Sensitivity: statistics across calendar-month means ---------------------------
# The sheet's 3-10 W m-2 standard deviations match the spread of MONTHLY means,
# not of minute-to-minute values; S.monthly_means gives one value per month.
check(8, S.monthly_means(lw), "RADFLUX downwelling LW, monthly means (std across months)", "calendar-month mean",
      primary=False)

**Row 8: RADFLUX downwelling LW, every QC = 0 sample (no further averaging), QC = 0 (Scripps Pier, not Soledad)**  
One sample = every QC = 0 sample (no further averaging).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,347 ± 41,522527,350 ± 30,-0.9,1.37,True
Spring,355 ± 36,131040,370 ± 10,-4.0,3.57,True
Summer,381 ± 28,131819,380 ± 10,0.2,2.80,True
Fall,332 ± 34,132419,323 ± 5,2.8,6.85,True
Winter,318 ± 36,127249,308 ± 3,3.3,12.04,True


**Row 8: RADFLUX downwelling LW, monthly means (std across months)**  *(sensitivity test)*  
One sample = calendar-month mean.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,344 ± 28,13,350 ± 30,-1.8,0.93,True
Spring,355 ± 15,3,370 ± 10,-4.0,1.46,True
Summer,380.9 ± 3.4,3,380 ± 10,0.2,0.34,True
Fall,332 ± 18,3,323 ± 5,2.7,3.69,True
Winter,316.6 ± 8.7,4,308 ± 3,2.8,2.91,True


,sheet_row,quantity,definition,season,ours_mean,ours_std,n,table_mean,table_std,ours,table,diff_pct,std_ratio,close,sample,primary
0,8,Longwave (LW) Flux,"RADFLUX downwelling LW, monthly means (std across months)",EPCAPE,343.818812,27.764431,13,350.0,30.0,344 ± 28,350 ± 30,-1.766054,0.925481,True,calendar-month mean,False
1,8,Longwave (LW) Flux,"RADFLUX downwelling LW, monthly means (std across months)",Spring,355.043982,14.570129,3,370.0,10.0,355 ± 15,370 ± 10,-4.042167,1.457013,True,calendar-month mean,False
2,8,Longwave (LW) Flux,"RADFLUX downwelling LW, monthly means (std across months)",Summer,380.876808,3.428037,3,380.0,10.0,380.9 ± 3.4,380 ± 10,0.230739,0.342804,True,calendar-month mean,False
3,8,Longwave (LW) Flux,"RADFLUX downwelling LW, monthly means (std across months)",Fall,331.842257,18.432880,3,323.0,5.0,332 ± 18,323 ± 5,2.737541,3.686576,True,calendar-month mean,False
4,8,Longwave (LW) Flux,"RADFLUX downwelling LW, monthly means (std across months)",Winter,316.588854,8.730148,4,308.0,3.0,316.6 ± 8.7,308 ± 3,2.788589,2.910049,True,calendar-month mean,False


### 2b. Kavin's data and code: Dan Lubin's monthly LW diurnal cycles

Kavin's row-8 values come from Lubin's `*_LW_Stepplot.csv` tables. For each month, these give the **median across days of each hour's LW↓** (hours in PST), with the minimum, quartiles and maximum alongside. Kavin's code reads the medians from the 3-month files (`DJF`, `MAM`, `JJA`, `SON`). The per-month files hold the same medians, which is checked for SEP, OCT and NOV, so they stand in for 3-month files that are missing. The steps:

* **mean** = the mean across the season's months of the median diurnal cycle, averaged over its 24 hours;
* **std** = the **hour-to-hour spread of that 24-hour cycle**. This is why the sheet's stds are 3–10 W m⁻²: they describe the diurnal cycle, not day-to-day variability;
* seasons are calendar quarters, and the campaign uses all 12 × 24 medians.

Lubin's files cover only APR, AUG, SEP, OCT, NOV and DEC, so only **fall** can be reproduced. Applying the **same statistic to RADFLUX at the Pier**, over all twelve months, shows what the full set would give. The month-by-month table tests whether Lubin's LW is the Pier record.

In [9]:
# =============================================================================
# Section 2b: downwelling LW from Kavin's input files (Dan Lubin's monthly diurnal cycles)
# =============================================================================
# Inputs
#   KAVIN_DIR/{DJF,MAM,JJA,SON}_LW_Stepplot.csv  per-month columns of hourly medians (what Kavin reads)
#   KAVIN_DIR/<MON>_LW_Stepplot.csv             Hour, Min, 25%ile, Median, 75%ile, Max for one month
#   rad (Section 1)                             RADFLUX downwelling_longwave at the Pier, QC = 0
# K.kavin_lw_stats: season = mean of its months' median diurnal cycles; std = spread over the 24 hours.

# --- 1. Lubin's monthly hourly medians (W m-2) ---------------------------------------------------
LUBIN_LW, lw_sources = K.read_lubin_lw_medians(KAVIN_DIR)
print("Lubin LW medians: " + "; ".join(f"{K.MONTH_ABBR[m - 1]} from {f}" for m, f in sorted(lw_sources.items())))
print("Missing months:", ", ".join(K.MONTH_ABBR[m - 1] for m in range(1, 13) if m not in lw_sources) or "none")

# --- 2. Kavin's code on Kavin's data ---------------------------------------------------------------
lw_kavin = K.kavin_lw_stats(LUBIN_LW)
check_stats(8, lw_kavin, "Lubin's monthly hourly medians, Kavin's code: mean of the seasonal median diurnal cycle; "
            "std = spread of its 24 hours; calendar quarters", "one hour of the median diurnal cycle")

# --- 3. The same statistic on RADFLUX at the Pier, all twelve months ----------------------------
#        1-min LW (time stamps = END of the minute -> shifted to the start), campaign only, then
#        hourly means in PST, then each month's median of each hour (K.monthly_hourly_medians)
lw_1min = Q.to_series(Q.qc0_values(rad, "downwelling_longwave")[0])
lw_1min.index = lw_1min.index - pd.Timedelta(minutes=1)
lw_1min = lw_1min[S.season_mask(lw_1min.index, "EPCAPE")]
RAD_LW_MEDIANS = K.monthly_hourly_medians(lw_1min)
check_stats(8, K.kavin_lw_stats(RAD_LW_MEDIANS), "RADFLUX LW (Pier), QC = 0, Kavin's statistic: monthly hourly "
            "medians (PST), mean of the seasonal median diurnal cycle; calendar quarters",
            "one hour of the median diurnal cycle")

# --- 4. Month by month: is Lubin's LW the Pier record? ---------------------------------------------
#        mean of the 24 hourly medians, and the hour shift that best aligns the two diurnal cycles
by_month = {}
for m in LUBIN_LW.columns:
    lub, pier = LUBIN_LW[m].to_numpy(), RAD_LW_MEDIANS[m].to_numpy()
    r_by_shift = [np.corrcoef(np.roll(pier, -k), lub)[0, 1] for k in range(24)]
    by_month[K.MONTH_ABBR[m - 1]] = {"Lubin (W m⁻²)": lub.mean(), "RADFLUX Pier (W m⁻²)": pier.mean(),
                                     "Lubin − Pier (W m⁻²)": lub.mean() - pier.mean(),
                                     "max |hourly diff| (W m⁻²)": np.abs(lub - pier).max(),
                                     "best hour shift": int(np.argmax(r_by_shift)), "r at that shift": max(r_by_shift)}
display(Markdown("Mean of the 24 monthly hourly medians, Lubin vs RADFLUX at the Pier (both PST hours):"))
display(pd.DataFrame(by_month).T.round(2))
del rad, lw_1min  # free the RADFLUX file; not needed after this section

Lubin LW medians: APR from APR_LW_Stepplot.csv; AUG from AUG_LW_Stepplot.csv; SEP from SON_LW_Stepplot.csv; OCT from SON_LW_Stepplot.csv; NOV from SON_LW_Stepplot.csv; DEC from DEC_LW_Stepplot.csv
Missing months: JAN, FEB, MAR, MAY, JUN, JUL


**Row 8: Lubin's monthly hourly medians, Kavin's code: mean of the seasonal median diurnal cycle; std = spread of its 24 hours; calendar quarters**  *(sensitivity test)*  
One sample = one hour of the median diurnal cycle.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,–,0,350 ± 30,NaN,NaN,False
Spring,–,0,370 ± 10,NaN,NaN,False
Summer,–,0,380 ± 10,NaN,NaN,False
Fall,323.0 ± 4.9,24,323 ± 5,0.0,0.97,True
Winter,–,0,308 ± 3,NaN,NaN,False


**Row 8: RADFLUX LW (Pier), QC = 0, Kavin's statistic: monthly hourly medians (PST), mean of the seasonal median diurnal cycle; calendar quarters**  *(sensitivity test)*  
One sample = one hour of the median diurnal cycle.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,346 ± 34,288,350 ± 30,-1.2,1.15,True
Spring,367 ± 13,24,370 ± 10,-0.9,1.33,True
Summer,384 ± 14,24,380 ± 10,1.1,1.35,True
Fall,323.0 ± 4.8,24,323 ± 5,0.0,0.97,True
Winter,308.7 ± 3.5,24,308 ± 3,0.2,1.16,True


Mean of the 24 monthly hourly medians, Lubin vs RADFLUX at the Pier (both PST hours):

,Lubin (W m⁻²),RADFLUX Pier (W m⁻²),Lubin − Pier (W m⁻²),max |hourly diff| (W m⁻²),best hour shift,r at that shift
APR,352.99,353.00,-0.01,0.78,0.0,1.0
AUG,389.84,389.47,0.37,3.29,0.0,1.0
SEP,381.68,381.64,0.04,0.56,0.0,1.0
OCT,345.25,345.22,0.03,0.83,0.0,1.0
NOV,308.12,308.15,-0.03,1.17,0.0,1.0
DEC,315.75,315.75,0.00,0.73,0.0,1.0


## 3. Liquid water path (row 10)

The sheet's source is "From Virendra (?)", which is not documented. Two ARM retrievals exist at the Pier, and both are shown:

* **MWRRET v1** (`epcmwrret1liljclouM1.c2`, `phys_lwp`; Turner et al. 2007, *IEEE TGRS* 45, 3680): the 2-channel MWR (23.8, 31.4 GHz), ~20-s samples.
* **MWRRET v2** (`epcmwrret2turnM1.c1`, `phys_lwp`): the 3-channel MWR (`mwr3c.b1`), ~1.3-s samples.

**Cloudy scenes only.** LWP is averaged only over samples taken under a detected cloud base. Clear-sky retrievals (LWP ≈ 0, often slightly negative) never enter.

* v1: `cloud_base_height` (ARSCL's ceilometer + lidar best estimate) with `qc_cloud_base_height` = 0. That QC field's bit 2 marks clear sky, and its bit 3 marks a 1-km fill on days without input.
* v2: `cbh_detected` > 0 and ceilometer `detection_status` 1–3. Neither field has a QC field. Fog with no detectable base (status 4) is not counted as cloud.

**Averaging.** `AVG_PERIOD_LWP` sets the averaging window in minutes. With a window, each value is the mean of the cloudy samples in one window, and every window counts once in the statistics. With `AVG_PERIOD_LWP = 0` there is no time averaging: every sample that passes the QC, cloud and rain filters enters the statistics on its own. Periods with more cloudy samples then carry more weight, and the counts `n` become sample counts. MWRRET v1 reports every ~20 s and v2 about every 1 s, so their `n` differ by ~20×; the means are still comparable because both sample regularly in time.

**Quality control.** `qc_phys_lwp` = 0 for both retrievals; for v2, `phys_qc_flag` = 0 as well.

**Rain.** Liquid on the radiometer's radome emits at these frequencies and makes the retrieved LWP far too high. Each retrieval is therefore shown twice: **rain-free**, which excludes values within ±`LWP_RAIN_WINDOW_MIN` of any rain at the co-located Parsivel (LD M1, QC = 0), and **including rain**. For an averaged value, the ± window is measured from either end of its averaging window. Rain-free MWRRET v1 is the primary definition. The last table sets all four side by side.

**Reading against the sheet:** the sheet's 56 ± 127 g m⁻² has a standard deviation larger than its mean, which suggests an all-sky mean that includes clear sky. A cloudy-scene mean is expected to be higher.

In [10]:
# =============================================================================
# Section 3: liquid water path (LWP) over cloudy scenes, with and without rain
# =============================================================================

# --- 1. Rain record: flags times when the radiometer's radome may be wet -----
# Pier Parsivel (LD M1) precipitation rate, 1-min samples, QC = 0 samples only.
ld_m1 = load_product("rain_ld_M1")
pier_rain = Q.to_series(Q.qc0_values(ld_m1, "precip_rate")[0])
del ld_m1  # only the rain-rate series is needed from here on

# --- 2. The two retrievals, and how each marks a sample as cloudy -------------
#   product      : product name in config.yaml (downloaded by download_data.py)
#   cloudy       : function -> True where a cloud base was detected above the MWR
#   require_zero : quality-flag fields that must equal 0, in addition to qc_phys_lwp
#   primary      : the definition carried into the summary table and figure
LWP_RETRIEVALS = {
    "MWRRET v1 (2-channel MWR, mwrret1liljclou.c2)": dict(
        product="lwp_mwrret1_M1", cloudy=Q.cloudy_mwrret1, require_zero=(), primary=True),
    "MWRRET v2 (3-channel MWR, mwrret2turn.c1)": dict(
        product="lwp_mwrret2_M1", cloudy=Q.cloudy_mwrret2, require_zero=("phys_qc_flag",), primary=False),
}

# What one value in the tables is, given AVG_PERIOD_LWP (minutes; 0 = no time averaging)
LWP_SAMPLE = ("cloudy sample (no time averaging)" if AVG_PERIOD_LWP == 0
              else f"{AVG_PERIOD_LWP:g}-min mean of cloudy samples")

LWP_TABLE = {}  # (retrieval, rain filter) -> "mean ± std" per season, for the side-by-side table in step 7
for label, spec in LWP_RETRIEVALS.items():
    display(Markdown(f"### {label}"))
    ds = load_product(spec["product"])

    # --- 3. Which QC tests fire on phys_lwp, and how often ---------------------
    print(qc.describe_qc(ds, "phys_lwp"))

    # --- 4. Cloudy-sample mask; then QC = 0 LWP of cloudy samples, averaged ----
    #        over AVG_PERIOD_LWP minutes (0 = keep every sample at its own time)
    cloudy = spec["cloudy"](ds)
    lwp, info = Q.lwp_cloudy_gm2(ds, "phys_lwp", cloudy, require_zero=spec["require_zero"],
                                 window_min=AVG_PERIOD_LWP)
    QC_INFO.extend(info["qc"])  # feeds the QC table in Section 11
    print(f"Cloudy samples: {100 * info['cloudy_fraction']:.1f} % of those passing QC")

    # --- 5. Flag values within ±LWP_RAIN_WINDOW_MIN of rain at the Pier --------
    #        Each value covers [t, t + AVG_PERIOD_LWP]; with 0 each sample is
    #        checked at its own time stamp against the 1-min rain record.
    near_rain = Q.near_rain(pier_rain, lwp.index, window_min=LWP_RAIN_WINDOW_MIN,
                            value_length_min=AVG_PERIOD_LWP)
    has_value = lwp.notna().to_numpy()
    print(f"Values ({LWP_SAMPLE}) near rain: {100 * near_rain[has_value].mean():.1f} %; mean LWP there "
          f"{lwp[near_rain].mean():.0f} g m⁻² vs {lwp[~near_rain].mean():.0f} g m⁻² rain-free")

    # --- 6. Seasonal mean ± std vs the sheet: rain-free, then including rain ---
    for rain_label, series, is_primary in [
        ("rain-free", lwp.where(~near_rain), spec["primary"]),
        ("including rain", lwp, False),
    ]:
        cmp = check(10, series, f"{label}: phys_lwp, cloudy scenes, {rain_label}, QC = 0",
                    LWP_SAMPLE, primary=is_primary)
        LWP_TABLE[(label.split(" (")[0], rain_label)] = cmp.set_index("season")["ours"]
    del ds, cloudy  # v2 holds ~23 million 1.3-s samples; free it before the next retrieval

# --- 7. All four side by side, with the sheet's values for reference -----------
lwp_side_by_side = pd.DataFrame(LWP_TABLE).T[S.SEASON_ORDER]
lwp_side_by_side.loc[("SHEET (Kavin)", "as given"), :] = S.table_values(TABLE, 10)["raw"].values
display(Markdown(f"**LWP (g m⁻²) over cloudy scenes: mean ± std; one value = {LWP_SAMPLE}**"))
display(lwp_side_by_side)

### MWRRET v1 (2-channel MWR, mwrret1liljclou.c2)

qc_phys_lwp: 1,251,788 samples, 0.0% QC value missing
  bit    1 [Bad          ]   1.8%  Data value does not represent a valid measurement or missing value, data has been set to missing_value.
  bit    2 [Indeterminate]  33.7%  cloud_base_height field is missing.
  bit    4 [Indeterminate]   2.2%  MWR tbsky(tbsky23 or tbsky31) failed one of the following: max, water on window, thermal stabilization(warm-up or overheat), or possible spike.
  bit    8 [Bad          ]   0.4%  MWR tbsky(tbsky23 or tbsky31) failed one of the following: missing, spike, invalid optical depth, min, > 200 K test.
Cloudy samples: 100.0 % of those passing QC
Values (cloudy sample (no time averaging)) near rain: 9.6 %; mean LWP there 372 g m⁻² vs 75 g m⁻² rain-free


**Row 10: MWRRET v1 (2-channel MWR, mwrret1liljclou.c2): phys_lwp, cloudy scenes, rain-free, QC = 0**  
One sample = cloudy sample (no time averaging).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,75 ± 94,722913,56 ± 127,33.6,0.74,False
Spring,107 ± 100,243904,92 ± 161,16.8,0.62,False
Summer,63 ± 83,227937,59 ± 118,6.3,0.71,True
Fall,55 ± 83,120099,42 ± 100,31.8,0.83,False
Winter,53 ± 92,130973,33 ± 92,59.9,1.00,False


**Row 10: MWRRET v1 (2-channel MWR, mwrret1liljclou.c2): phys_lwp, cloudy scenes, including rain, QC = 0**  *(sensitivity test)*  
One sample = cloudy sample (no time averaging).

...

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,75 ± 84,10343115,56 ± 127,33.2,0.66,False
Spring,104 ± 89,3800979,92 ± 161,13.1,0.55,False
Summer,61 ± 70,3743154,59 ± 118,4.2,0.60,True
Fall,44 ± 63,1235590,42 ± 100,5.2,0.63,True
Winter,58 ± 93,1563392,33 ± 92,77.2,1.01,False


**Row 10: MWRRET v2 (3-channel MWR, mwrret2turn.c1): phys_lwp, cloudy scenes, including rain, QC = 0**  *(sensitivity test)*  
One sample = cloudy sample (no time averaging).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,93 ± 130,11405874,56 ± 127,66.1,1.03,False
Spring,120 ± 114,4107604,92 ± 161,30.2,0.71,False
Summer,74 ± 115,3894065,59 ± 118,24.9,0.98,False
Fall,62 ± 127,1354938,42 ± 100,47.3,1.27,False
Winter,97 ± 172,2049267,33 ± 92,193.5,1.87,False


**LWP (g m⁻²) over cloudy scenes: mean ± std; one value = cloudy sample (no time averaging)**

season                           EPCAPE     Spring    Summer      Fall     Winter
MWRRET v1     rain-free         75 ± 94  107 ± 100   63 ± 83   55 ± 83    53 ± 92
              including rain  103 ± 168  127 ± 135  78 ± 132  76 ± 156  123 ± 244
MWRRET v2     rain-free         75 ± 84   104 ± 89   61 ± 70   44 ± 63    58 ± 93
              including rain   93 ± 130  120 ± 114  74 ± 115  62 ± 127   97 ± 172
SHEET (Kavin) as given        56 ± 127    92 ± 161  59 ± 118  42 ± 100    33 ± 92

### 3b. Kavin's data and code: the hourly MWRRET v2 file

Kavin's row-10 values come from `epcmwrret_turnM1_hourly_averages.nc`, which holds hourly means of MWRRET v2 `phys_lwp`. Step 5 below rebuilds it from the ARM files: each hour is the **mean of the QC = 0 samples with LWP > 0**, and `lwp_count` counts all QC = 0 samples. The file has **no cloud screen and no rain screen**. Clear-sky hours enter with small positive values, because negative noise is dropped but positive noise is kept, and wet-radome hours enter with large ones.

His code ("LWP Code") puts each month into one column of `yearlylwpbox = nan(500,13)`, with months in **Pacific time**. Two things in it change the numbers:

* **Zero padding.** Most months hold more than 500 hours. Assigning past row 500 makes MATLAB grow the array and fill the new elements of every *other* column with **0**, not NaN. Those zeros are kept by `'omitmissing'`, so they pull every mean down.
* **Column ↔ month offset.** Column *j* holds month *j*+1 (Feb 2023 is column 1). The sheet's seasonal values are reproduced by columns 2:4, 5:7, 8:10 and 11:13, which are **Mar–May, Jun–Aug, Sep–Nov and Dec–Feb**, not the sheet's AMJ/JAS/OND/FM+JF headers. The live script as shared uses 4:6, 7:9, 10:12 and 1:3 (May–Jul, Aug–Oct, Nov–Jan, Feb–Apr), which does *not* reproduce the sheet. The sheet was presumably made with an earlier version of the script.

Corrected on the same file: no padding zeros, the sheet's seasons, and optionally the same ±30-min rain screen as Section 3.

In [11]:
# =============================================================================
# Section 3b: LWP from Kavin's input file (hourly means of MWRRET v2)
# =============================================================================
# Inputs
#   KAVIN_DIR/epcmwrret_turnM1_hourly_averages.nc  hourly means of MWRRET v2 phys_lwp (g m-2), labelled
#                                                   by hour START (UTC), with lwp_count (K.read_kavin_lwp)
#   pier_rain (Section 3)                           Pier Parsivel rain rate (QC = 0), for the rain screen
#   lwp_mwrret2_M1                                  the ARM MWRRET v2 files, to rebuild the hourly file
# K.kavin_lwp_box builds Kavin's 13-column array (MATLAB zero padding included);
# K.kavin_lwp_stats averages sets of its columns.

# --- 1. Read the hourly file -------------------------------------------------------------------
KAVIN_LWP = K.read_kavin_lwp(KAVIN_DIR)
klwp = KAVIN_LWP["phys_lwp_gm2"]
print(f"{len(KAVIN_LWP):,} hours, {KAVIN_LWP.index[0]} to {KAVIN_LWP.index[-1]} UTC; {klwp.isna().sum()} hours "
      f"without LWP; smallest LWP {klwp.min():.4f} g m⁻² (none <= 0)")

# --- 2. Kavin's code exactly: the zero-padded array --------------------------------------------
LWP_BOX = K.kavin_lwp_box(klwp)  # (rows, 13): column j = month j+1 of 2023, then Jan and Feb 2024
lwp_kavin_sheet = K.kavin_lwp_stats(LWP_BOX, K.KAVIN_LWP_COLUMNS_SHEET)
check_stats(10, lwp_kavin_sheet, "Kavin's hourly MWRRET v2 file, Kavin's code: zero-padded array; columns 2:4, 5:7, "
            "8:10, 11:13 (Mar–May, Jun–Aug, Sep–Nov, Dec–Feb; Pacific time)", "hourly mean (all sky) or a padding zero")
lwp_kavin_shared = K.kavin_lwp_stats(LWP_BOX, K.KAVIN_LWP_COLUMNS_SHARED)
check_stats(10, lwp_kavin_shared, "Kavin's hourly MWRRET v2 file, the script as shared: zero-padded array; columns "
            "4:6, 7:9, 10:12, 1:3 (May–Jul, Aug–Oct, Nov–Jan, Feb–Apr)", "hourly mean (all sky) or a padding zero")
display(Markdown("Months in each column set, and how many padding zeros MATLAB added to it:"))
display(pd.DataFrame({
    "reproduces the sheet: months": lwp_kavin_sheet["months"], "hourly values": lwp_kavin_sheet["n"] - lwp_kavin_sheet["n_zero_pad"],
    "padding zeros": lwp_kavin_sheet["n_zero_pad"],
    "script as shared: months": lwp_kavin_shared["months"], "padding zeros (as shared)": lwp_kavin_shared["n_zero_pad"],
}))

# --- 3. Same file, corrected: no padding zeros, the sheet's seasons (UTC) --------------------------
check(10, klwp, "Kavin's hourly file, no padding zeros; sheet seasons", "hourly mean (all sky)", primary=False)

# --- 4. Same file, rain-free hours: no Pier rain within ±LWP_RAIN_WINDOW_MIN of the hour [t, t + 60 min] ---
near_rain_h = Q.near_rain(pier_rain, klwp.index, window_min=LWP_RAIN_WINDOW_MIN, value_length_min=60)
print(f"Hours near rain: {near_rain_h[klwp.notna().to_numpy()].mean():.1%}")
check(10, klwp.where(~near_rain_h), f"Kavin's hourly file, no padding zeros, rain-free hours (±{LWP_RAIN_WINDOW_MIN:g} min "
      "of Pier rain); sheet seasons", "hourly mean (all sky)", primary=False)

# --- 5. What the hourly file holds: rebuild it from the ARM MWRRET v2 files ---------------------
ds = load_product("lwp_mwrret2_M1")
lwp_v2 = Q.to_series(units.water_path_to_gm2(Q.qc0_values(ds, "phys_lwp")[0]))  # QC = 0 samples, g m-2
del ds
rebuild = {}
for label, samples in {"mean of all QC = 0 samples": lwp_v2,
                       "mean of the QC = 0 samples with LWP > 0": lwp_v2.where(lwp_v2 > 0)}.items():
    pair = pd.concat({"file": klwp, "rebuilt": samples.resample("1h").mean()}, axis=1).dropna()
    rebuild[label] = {"hours compared": len(pair),
                      "hours within 0.001 g m⁻² of the file": np.mean(np.abs(pair["file"] - pair["rebuilt"]) < 1e-3)}
counts = pd.concat({"file": KAVIN_LWP["lwp_count"], "rebuilt": lwp_v2.resample("1h").count()}, axis=1).dropna()
rebuild["lwp_count vs number of ALL QC = 0 samples"] = {"hours compared": len(counts),
                                                        "hours within 0.001 g m⁻² of the file": np.mean(counts["file"] == counts["rebuilt"])}
display(Markdown("Rebuilding Kavin's hourly file from the ARM MWRRET v2 files (fraction of hours that agree):"))
display(pd.DataFrame(rebuild).T.rename(columns={"hours within 0.001 g m⁻² of the file": "fraction agreeing"}).round(3))
del lwp_v2

# --- 6. Side by side, mean ± std in g m-2 ---------------------------------------------------------------
def pm(stats):
    """'mean ± std' per season, in SEASON_ORDER."""
    return [S.format_pm(m, s) for m, s in zip(stats["mean"], stats["std"])]


LWP_STEPS = pd.DataFrame({
    "sheet (row 10)": S.table_values(TABLE, 10)["raw"].values,
    "Kavin's code, columns that reproduce the sheet": pm(lwp_kavin_sheet),
    "Kavin's code as shared": pm(lwp_kavin_shared),
    "same file: no padding zeros, sheet seasons": pm(S.seasonal_stats(klwp)),
    "same file: no padding zeros, rain-free hours": pm(S.seasonal_stats(klwp.where(~near_rain_h))),
    "this check: MWRRET v1, cloudy, rain-free (Section 3, primary)": LWP_TABLE[("MWRRET v1", "rain-free")].values,
    "this check: MWRRET v2, cloudy, rain-free (Section 3)": LWP_TABLE[("MWRRET v2", "rain-free")].values,
}, index=S.SEASON_ORDER).T
display(Markdown("**LWP (g m⁻²), step by step from the sheet to this check** (Kavin's columns use his own month sets; see above)"))
display(LWP_STEPS)

7,913 hours, 2023-02-15 00:00:00 to 2024-02-14 23:00:00 UTC; 598 hours without LWP; smallest LWP 0.0037 g m⁻² (none <= 0)


**Row 10: Kavin's hourly MWRRET v2 file, Kavin's code: zero-padded array; columns 2:4, 5:7, 8:10, 11:13 (Mar–May, Jun–Aug, Sep–Nov, Dec–Feb; Pacific time)**  *(sensitivity test)*  
One sample = hourly mean (all sky) or a padding zero.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,56 ± 127,8457,56 ± 127,-0.7,1.00,True
Spring,92 ± 161,2093,92 ± 161,0.2,1.00,True
Summer,59 ± 118,2129,59 ± 118,-0.8,1.00,True
Fall,42 ± 100,1932,42 ± 100,0.9,1.00,True
Winter,33 ± 121,1782,33 ± 92,0.7,1.31,True


**Row 10: Kavin's hourly MWRRET v2 file, the script as shared: zero-padded array; columns 4:6, 7:9, 10:12, 1:3 (May–Jul, Aug–Oct, Nov–Jan, Feb–Apr)**  *(sensitivity test)*  
One sample = hourly mean (all sky) or a padding zero.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,56 ± 127,8457,56 ± 127,-0.7,1.00,True
Spring,74 ± 100,2173,92 ± 161,-19.7,0.62,False
Summer,53 ± 122,1967,59 ± 118,-10.1,1.04,False
Fall,35 ± 127,1962,42 ± 100,-17.8,1.27,False
Winter,66 ± 161,1881,33 ± 92,100.8,1.75,False


...

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,42 ± 61,6522,56 ± 127,-24.2,0.48,False
Spring,73 ± 79,1841,92 ± 161,-21.0,0.49,False
Summer,44 ± 57,1936,59 ± 118,-26.2,0.48,False
Fall,19 ± 29,1385,42 ± 100,-54.6,0.29,False
Winter,24 ± 43,1360,33 ± 92,-28.1,0.47,False


Rebuilding Kavin's hourly file from the ARM MWRRET v2 files (fraction of hours that agree):

,hours compared,fraction agreeing
mean of all QC = 0 samples,7315.0,0.671
mean of the QC = 0 samples with LWP > 0,7315.0,1.000
lwp_count vs number of ALL QC = 0 samples,7913.0,1.000


**LWP (g m⁻²), step by step from the sheet to this check** (Kavin's columns use his own month sets; see above)

,EPCAPE,Spring,Summer,Fall,Winter
sheet (row 10),56 ± 127,92 ± 161,59 ± 118,42 ± 100,33 ± 92
"Kavin's code, columns that reproduce the sheet",56 ± 127,92 ± 161,59 ± 118,42 ± 100,33 ± 121
Kavin's code as shared,56 ± 127,74 ± 100,53 ± 122,35 ± 127,66 ± 161
"same file: no padding zeros, sheet seasons",64 ± 134,90 ± 102,60 ± 125,34 ± 127,65 ± 173
"same file: no padding zeros, rain-free hours",42 ± 61,73 ± 79,44 ± 57,19 ± 29,24 ± 43
"this check: MWRRET v1, cloudy, rain-free (Section 3, primary)",75 ± 94,107 ± 100,63 ± 83,55 ± 83,53 ± 92
"this check: MWRRET v2, cloudy, rain-free (Section 3)",75 ± 84,104 ± 89,61 ± 70,44 ± 63,58 ± 93


## 4. Cloud-top height (row 15)

KAZR-ARSCL (`cloud_layer_top_height`, heights above ground; the Pier site is ~7 m above sea level) gives the **top of the lowest hydrometeor layer**, counted when that top is ≤ `LOW_CLOUD_MAX_M`. The sheet's ~900 m mean says Kavin's statistic is about the marine stratocumulus deck, not every cloud.
ARSCL distributes no QC field for these variables.

In [12]:
# =============================================================================
# Section 4: cloud-top height from KAZR-ARSCL (sheet row 15)
# =============================================================================
# ARSCL combines the Ka-band radar (KAZR, which sets cloud TOPS), the micropulse
# lidar and the ceilometer into up to 10 hydrometeor layers every 4 s, heights
# in m above ground. Q.lowest_layer_top_m takes the top of the LOWEST layer,
# keeps it only for low cloud (top <= LOW_CLOUD_MAX_M) and, if CT_CLOUD_SOURCES is
# set, only where the chosen instrument(s) saw that top, then averages the kept
# samples over AVG_PERIOD_CT minutes (0 = no averaging). ARSCL has no QC fields
# for these variables.

# --- 1. Load the combined campaign file (~8 million 4-s samples, 10 layers) ----
arscl = load_product("cloud_arscl_M1")
if CT_CLOUD_SOURCES is not None:
    # Attach the per-sample "which instrument saw the lowest top" code (cloud_source_flag
    # at the top gate, reduced from the daily files; same 4-s time stamps as arscl).
    arscl["cloud_top_source"] = SRC.load_arscl_top_source().reindex(time=arscl["time"], fill_value=-1)
# What one value in the tables is, given AVG_PERIOD_CT (minutes; 0 = no averaging)
CT_SAMPLE = "4-s sample (no time averaging)" if AVG_PERIOD_CT == 0 else f"{AVG_PERIOD_CT:g}-min mean"

# --- 2. Primary: top of the lowest layer, low cloud only ------------------------
cth, info = Q.lowest_layer_top_m(arscl, max_top_m=LOW_CLOUD_MAX_M, window_min=AVG_PERIOD_CT,
                                 cloud_sources=CT_CLOUD_SOURCES)
QC_INFO.extend(info["qc"])  # feeds the QC table in Section 11
if info["top_source_counts"]:  # which instruments saw the low-cloud tops (before the source selection)
    n_tops = sum(info["top_source_counts"].values())
    display(pd.Series({f"{k}: {Q.ARSCL_SOURCE_CODES.get(k, 'no layer')}": 100 * v / n_tops
                       for k, v in info["top_source_counts"].items()}, name="% of low-cloud tops").round(1))
check(15, cth, f"KAZR-ARSCL top of lowest layer, ≤ {LOW_CLOUD_MAX_M:g} m AGL", f"{CT_SAMPLE} (low cloud present)")

# --- 3. Sensitivity: single-layer low cloud only (no second layer above) -------
cth_single, _ = Q.lowest_layer_top_m(arscl, max_top_m=LOW_CLOUD_MAX_M, single_layer=True,
                                     window_min=AVG_PERIOD_CT, cloud_sources=CT_CLOUD_SOURCES)
check(15, cth_single, f"KAZR-ARSCL single-layer low cloud only, top ≤ {LOW_CLOUD_MAX_M:g} m", CT_SAMPLE,
      primary=False)

# --- 4. Sensitivity: no height cut (includes mid- and high-level cloud) --------
cth_any, _ = Q.lowest_layer_top_m(arscl, max_top_m=np.inf, window_min=AVG_PERIOD_CT,
                              cloud_sources=CT_CLOUD_SOURCES)
check(15, cth_any, "KAZR-ARSCL top of lowest layer, any height", f"{CT_SAMPLE} (any cloud)", primary=False)

# --- 5. Keep a 5-min "low cloud overhead" flag for Section 5 --------------------
# True for 5-min bins that held at least one low-cloud sample. Section 5 uses it
# to compute the PBL height only while a low cloud is overhead. Built on a fixed
# 5-min grid so it means the same thing whatever AVG_PERIOD_CT is (with 0, cth
# holds only the cloudy samples, so empty bins -> NaN -> False).
LOW_CLOUD_5MIN = cth.notna().resample("5min").max().astype("boolean").fillna(False).astype(bool)
del arscl  # free ~0.7 GB

**Row 15: KAZR-ARSCL top of lowest layer, ≤ 3500 m AGL**  
One sample = 4-s sample (no time averaging) (low cloud present).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,902 ± 615,2205853,920 ± 380,-2.0,1.62,True
Spring,814 ± 356,981951,890 ± 330,-8.6,1.08,True
Summer,616 ± 408,669637,774 ± 353,-20.4,1.16,False
Fall,1212 ± 774,160695,1057 ± 416,14.7,1.86,False
Winter,1482 ± 860,393570,1192 ± 397,24.3,2.17,False


**Row 15: KAZR-ARSCL single-layer low cloud only, top ≤ 3500 m**  *(sensitivity test)*  
One sample = 4-s sample (no time averaging).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,829 ± 527,1755150,920 ± 380,-9.9,1.39,True
Spring,815 ± 361,854644,890 ± 330,-8.5,1.09,True
Summer,604 ± 374,612332,774 ± 353,-22.0,1.06,False
Fall,1114 ± 683,113216,1057 ± 416,5.3,1.64,True
Winter,1504 ± 820,174958,1192 ± 397,26.2,2.06,False


**Row 15: KAZR-ARSCL top of lowest layer, any height**  *(sensitivity test)*  
One sample = 4-s sample (no time averaging) (any cloud).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,4246 ± 4351,3735117,920 ± 380,361.5,11.45,False
Spring,1906 ± 2983,1124470,890 ± 330,114.2,9.04,False
Summer,2902 ± 3757,956681,774 ± 353,274.9,10.64,False
Fall,8001 ± 4252,666466,1057 ± 416,657.0,10.22,False
Winter,5676 ± 4006,987500,1192 ± 397,376.2,10.09,False


## 5. Boundary-layer height (rows 17–19)

**Row 17, "PBLH-Thermo (Zhang)":** ARM's `pblhtthermo` VAP (Data Discovery: "Planetary Boundary Layer Height Best Estimate using Machine Learning"; `pbl_height` from combined Raman-lidar and TROPoe potential-temperature profiles, km above ground, 10-min samples, no QC field).
The sheet's seasonal cycle (summer lowest, winter highest) tracks the cloud-top heights in row 15. Three readings are computed: all times, times when KAZR sees a low cloud, and EPCAPE Low Cloud Periods. As for the rain rows, the **primary is the reading that best reproduces the sheet**: PBL height while a low cloud is overhead. The all-times mean is the first table.

**Rows 18–19, "PBLH-MPL" and "BL-Depth (Miller)":** ARM Data Discovery lists no MPL-based or Miller boundary-layer product for EPCAPE as of 2026-10-06. The MPL products are cloud masks and raw backscatter only. These two rows cannot be re-derived. For context, radiosonde PBL heights (PBLHTSONDE, QC = 0, converted to m above ground) are shown next to them.

In [13]:
# =============================================================================
# Section 5: PBL height from PBLHTTHERMO (sheet row 17)
# =============================================================================
# Input: pblh_thermo_M1 = epcpblhtthermoM1.c1, variable pbl_height (km above ground),
# from combined Raman-lidar and TROPoe potential-temperature profiles. The VAP gives
# ONE value every 10 minutes (on :00, :10, ... :50; no data 11-31 Oct 2023) and no
# QC field. "10-min sample" in the tables below means exactly that: each table value
# is one of these 10-min retrievals, used as delivered, with NO extra time averaging.

# --- 1. Load and convert to metres (Q.pblh_thermo_m: QC = 0 no-op, km -> m) ------
thermo = load_product("pblh_thermo_M1")
pblh, info = Q.pblh_thermo_m(thermo)
QC_INFO.extend(info["qc"])  # feeds the QC table in Section 11

# --- 2. All times (sensitivity) ---------------------------------------------------
# The sheet gives no definition beyond "PBLH-Thermo (Zhang)". The all-times mean is
# computed first; the primary is the reading that best reproduces the sheet (PBL
# height while a low cloud is overhead), as for the rain rows. Which reading the
# published table should use is a decision for the authors.
check(17, pblh, "PBLHTTHERMO pbl_height, all times", "10-min sample", primary=False)

# --- 3. Only while KAZR-ARSCL sees a low cloud (primary) ------------------------
# LOW_CLOUD_5MIN (Section 4) is True for 5-min bins holding at least one low-cloud
# (top <= LOW_CLOUD_MAX_M) sample. Each 10-min PBL time stamp is matched to the
# nearest 5-min bin within 5 min; because both grids fall on whole 5-/10-minute
# marks, that is the bin STARTING at the PBL time stamp. No match -> not cloudy.
cloudy = (LOW_CLOUD_5MIN.reindex(pblh.index, method="nearest", tolerance=pd.Timedelta("5min"))
          .astype("boolean").fillna(False).astype(bool))
check(17, pblh.where(cloudy), "PBLHTTHERMO when KAZR-ARSCL sees a low cloud (±5 min) [best-matching reading]",
      "10-min sample")

# --- 4. Only during EPCAPE Low Cloud Periods (sensitivity) ----------------------
# Hourly flag from the UCSD Library file (single-layer cloud, base and top < 3 km,
# cloud base within 150 m of the LCL). Each 10-min PBL value takes the flag of the
# hour it falls in (time stamp floored to the hour).
lcp = SRC.load_low_cloud_periods()
in_lcp = lcp.reindex(pblh.index.floor("1h")).fillna(False).to_numpy(dtype=bool)
check(17, pblh.where(in_lcp), "PBLHTTHERMO during EPCAPE Low Cloud Periods", "10-min sample", primary=False)

**Row 17: PBLHTTHERMO pbl_height, all times**  *(sensitivity test)*  
One sample = 10-min sample.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,656 ± 503,48617,899 ± 305,-27.0,1.65,False
Spring,773 ± 345,13102,886 ± 256,-12.8,1.35,False
Summer,492 ± 274,13244,764 ± 206,-35.6,1.33,False
Fall,459 ± 511,10174,1014 ± 343,-54.7,1.49,False
Winter,874 ± 679,12097,1124 ± 442,-22.2,1.54,False


**Row 17: PBLHTTHERMO when KAZR-ARSCL sees a low cloud (±5 min) [best-matching reading]**  
One sample = 10-min sample.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,804 ± 411,18053,899 ± 305,-10.6,1.35,False
Spring,779 ± 290,7453,886 ± 256,-12.1,1.13,False
Summer,618 ± 234,5766,764 ± 206,-19.1,1.14,False
Fall,920 ± 472,1266,1014 ± 343,-9.3,1.38,True
Winter,1114 ± 595,3568,1124 ± 442,-0.9,1.35,True


**Row 17: PBLHTTHERMO during EPCAPE Low Cloud Periods**  *(sensitivity test)*  
One sample = 10-min sample.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,635 ± 224,2004,899 ± 305,-29.4,0.73,False
Spring,725 ± 199,744,886 ± 256,-18.2,0.78,False
Summer,587 ± 220,1146,764 ± 206,-23.2,1.07,False
Fall,462 ± 209,42,1014 ± 343,-54.4,0.61,False
Winter,569 ± 214,72,1124 ± 442,-49.4,0.48,False


,sheet_row,quantity,definition,season,ours_mean,ours_std,n,table_mean,table_std,ours,table,diff_pct,std_ratio,close,sample,primary
0,17,Boundary Layer Height,PBLHTTHERMO during EPCAPE Low Cloud Periods,EPCAPE,634.569361,223.642125,2004,899.0,305.0,635 ± 224,899 ± 305,-29.413864,0.733253,False,10-min sample,False
1,17,Boundary Layer Height,PBLHTTHERMO during EPCAPE Low Cloud Periods,Spring,724.600806,198.829213,744,886.0,256.0,725 ± 199,886 ± 256,-18.216613,0.776677,False,10-min sample,False
2,17,Boundary Layer Height,PBLHTTHERMO during EPCAPE Low Cloud Periods,Summer,586.574171,220.231871,1146,764.0,206.0,587 ± 220,764 ± 206,-23.223276,1.069087,False,10-min sample,False
3,17,Boundary Layer Height,PBLHTTHERMO during EPCAPE Low Cloud Periods,Fall,462.071429,208.848340,42,1014.0,343.0,462 ± 209,1014 ± 343,-54.430826,0.608887,False,10-min sample,False
4,17,Boundary Layer Height,PBLHTTHERMO during EPCAPE Low Cloud Periods,Winter,568.791667,214.070670,72,1124.0,442.0,569 ± 214,1124 ± 442,-49.395759,0.484323,False,10-min sample,False


In [14]:
# =============================================================================
# Section 5 (continued): radiosonde PBL heights, and rows 18-19
# =============================================================================
# Input: pblh_sonde_M1 = epcpblhtsonde1mcfarlM1.c1 (PBLHTSONDE VAP), ONE FILE PER LAUNCH
# (~3-4 launches a day from the Scripps Pier). Each file holds one PBL height per method,
# in m above SEA level, each with its own qc_ field:
#   pbl_height_liu_liang              Liu & Liang (2010) potential-temperature method
#   pbl_height_heffter                Heffter (1980) inversion method
#   pbl_height_bulk_richardson_pt25   bulk Richardson number, critical value 0.25
# Q.sonde_launch reduces each file to one row: these heights with QC = 0, converted
# to m above GROUND (minus the 7-m site altitude), plus the launch's lowest ~2 hPa
# p, T and RH (used for the LCL in Section 6). The index is the launch time.

# --- 1. One row per launch -------------------------------------------------------------
launches = SRC.read_per_launch("pblh_sonde_M1", Q.sonde_launch)
SONDE_METHODS = {"liu_liang_m_agl": "Liu & Liang (2010)", "heffter_m_agl": "Heffter (1980)",
                 "bulk_richardson_pt25_m_agl": "bulk Richardson, Ri_c = 0.25"}

# --- 2. QC bookkeeping and (hidden) comparisons for each method --------------------------
# n_finite = launches with a value before QC (<method>_raw_m_agl); n_kept = after QC = 0.
for col, name in SONDE_METHODS.items():
    QC_INFO.append({"variable": f"pbl_height_{col.replace('_m_agl', '')}", "has_qc": True, "n": len(launches),
                    "n_finite": int(launches[col.replace("_m_agl", "_raw_m_agl")].notna().sum()),
                    "n_kept": int(launches[col].notna().sum())})
    check(17, launches[col], f"PBLHTSONDE {name}, QC = 0, m AGL", "radiosonde launch", primary=False, show=False)

# --- 3. Rows 18-19 have no identifiable data source ---------------------------------------
NOT_CHECKED[18] = "PBLH-MPL: no MPL-based PBL-height product for EPCAPE in ARM Data Discovery (2026-10-06); source unknown"
NOT_CHECKED[19] = "BL-Depth (Miller): source not identified (not an ARM product for EPCAPE)"

# --- 4. Context table: the sheet's three PBL rows next to every estimate available here --
context = {"sheet row 17 (PBLH-Thermo)": S.table_values(TABLE, 17)["raw"],
           "sheet row 18 (PBLH-MPL)": S.table_values(TABLE, 18)["raw"],
           "sheet row 19 (BL-Depth Miller)": S.table_values(TABLE, 19)["raw"],
           "PBLHTTHERMO, all times": S.seasonal_stats(pblh).pipe(lambda d: [S.format_pm(m, s) for m, s in zip(d["mean"], d["std"])])}
for col, name in SONDE_METHODS.items():
    st = S.seasonal_stats(launches[col])
    context[f"sondes: {name}"] = [S.format_pm(m, s) for m, s in zip(st["mean"], st["std"])]
display(Markdown("PBL heights for context (m above ground):"))
display(pd.DataFrame(context, index=S.SEASON_ORDER).T)

pblh_sonde_M1: 1098 launches read from /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/arm_subset/pblh_sonde_M1


PBL heights for context (m above ground):

,EPCAPE,Spring,Summer,Fall,Winter
sheet row 17 (PBLH-Thermo),899 ± 305,886 ± 256,764 ± 206,1014 ± 343,1124 ± 442
sheet row 18 (PBLH-MPL),790 ±.274,765 ± 255,728 ± 248,844 ± 313,970 ± 288
sheet row 19 (BL-Depth Miller),703 ± 275,796 ± 255,679 ± 261,626 + 297,649 ± 272
"PBLHTTHERMO, all times",656 ± 503,773 ± 345,492 ± 274,459 ± 511,874 ± 679
sondes: Liu & Liang (2010),285 ± 200,367 ± 207,261 ± 158,178 ± 138,278 ± 244
sondes: Heffter (1980),724 ± 627,910 ± 494,518 ± 300,484 ± 620,1031 ± 1031
"sondes: bulk Richardson, Ri_c = 0.25",251 ± 264,305 ± 247,185 ± 156,153 ± 198,378 ± 410


## 6. Lifting condensation level (row 20)

Primary: ARM's SONDEPARAM VAP, surface-parcel `lcl`. It is required to have `data_quality` = 0 for the whole launch, the QC = 0 equivalent because the VAP has no `qc_lcl`.
Cross-checks from the same launches:

* the exact LCL of Romps (2017, *JAS* 74, 3891), from the median p, T and RH of the lowest ~2 hPa (~15 m) of each sonde profile, as height above the launch level
* the 125 m K⁻¹ × (T − T_d) rule of Lawrence (2005, *BAMS* 86, 225), with T_d from the Magnus formula (Alduchov & Eskridge 1996)
* SONDEPARAM's mixed-layer-parcel LCL, which brackets the sheet's values from above

The VAP documentation does not say whether `lcl` is above ground or above sea level. The difference is ~7–13 m.

In [15]:
# =============================================================================
# Section 6: lifting condensation level (sheet row 20), one value per radiosonde launch
# =============================================================================
# Inputs
#   sondeparam_M1 = epcsondeparamM1.c1 (SONDEPARAM VAP), one file per launch:
#     lcl(time, parcel_type)  LCL in km for three parcels: 1 surface, 2 most unstable,
#                             3 mixed layer. Same value at every sample of the launch.
#                             Appears to be above SEA level (launch level ~13 m).
#     data_quality(time)      bit mask of input checks on the profile; 0 = all passed.
#                             There is no qc_lcl, so data_quality = 0 is our QC = 0.
#   launches (Section 5)    = pblh_sonde_M1 = epcpblhtsonde1mcfarlM1.c1, one file per launch:
#     p_sfc_hpa, t_sfc_c, rh_sfc_pct  medians of atm_pres, air_temp, rh over the lowest
#                             ~2 hPa (~15 m) of the profile; no QC field for these.
#     alt_m                   site altitude (7 m above sea level)
# Q.sondeparam_launch reduces each SONDEPARAM file to one row:
#   lcl_sondeparam_m  surface parcel, data_quality = 0  (primary)
#   lcl_raw_m         surface parcel, NO quality screen (only the -9999 fill removed)
#   lcl_mixed_layer_m mixed-layer parcel, data_quality = 0

# --- 1. One row per launch from the SONDEPARAM files ---------------------------
sp = SRC.read_per_launch("sondeparam_M1", Q.sondeparam_launch)
QC_INFO.append({"variable": "lcl (data_quality == 0)", "has_qc": True, "n": len(sp),
                "n_finite": int(sp["lcl_raw_m"].notna().sum()), "n_kept": int(sp["lcl_sondeparam_m"].notna().sum())})

# --- 2. Primary: SONDEPARAM surface-parcel LCL, data_quality = 0 ----------------
check(20, sp["lcl_sondeparam_m"], "SONDEPARAM surface-parcel LCL (data_quality = 0)", "radiosonde launch")

# --- 3. Same, with NO quality filter (every launch with a finite LCL) -----------
print(f"Launches failing data_quality: {int((~sp['data_quality_ok']).sum())} of {len(sp)}")
check(20, sp["lcl_raw_m"], "SONDEPARAM surface-parcel LCL, NO QC filter", "radiosonde launch", primary=False)

# --- 4. Sensitivity: mixed-layer parcel (brackets the sheet from above) ---------
check(20, sp["lcl_mixed_layer_m"], "SONDEPARAM mixed-layer-parcel LCL (data_quality = 0)", "radiosonde launch",
      primary=False)

# --- 5. Independent check: Romps (2017) exact LCL from each sonde's lowest ~2 hPa -
# Height above the launch level; inputs converted to Pa, K and a 0-1 fraction.
lcl_romps = pd.Series(
    Q.lcl_romps2017_m(launches["p_sfc_hpa"] * 100.0, launches["t_sfc_c"] + 273.15, launches["rh_sfc_pct"] / 100.0),
    index=launches.index, name="lcl_romps_m")
check(20, lcl_romps, "Romps (2017) exact LCL from the lowest ~2 hPa of each sonde (m above launch level)",
      "radiosonde launch", primary=False)

# --- 6. Rule of thumb: 125 m per K of dew-point depression (Lawrence 2005) -----
td_c = Q.dewpoint_magnus_c(launches["t_sfc_c"], launches["rh_sfc_pct"])  # Magnus dew point, degC
check(20, pd.Series(Q.lcl_lawrence2005_m(launches["t_sfc_c"], td_c), index=launches.index),
      "125 m/K × (T − Td) (Lawrence 2005)", "radiosonde launch", primary=False)

# --- 7. Same launches, two methods: how well do SONDEPARAM and Romps agree? ----
# Launches are matched by time (nearest within 10 min).
pair = pd.merge_asof(sp[["lcl_sondeparam_m"]].sort_index(), lcl_romps.sort_index().to_frame(),
                     left_index=True, right_index=True, tolerance=pd.Timedelta("10min"), direction="nearest").dropna()
d = pair["lcl_sondeparam_m"] - pair["lcl_romps_m"]
print(f"SONDEPARAM - Romps over {len(pair)} launches: median {d.median():.0f} m, "
      f"5th-95th percentile {d.quantile(0.05):.0f} to {d.quantile(0.95):.0f} m")

sondeparam_M1: 1098 launches read from /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/arm_subset/sondeparam_M1


**Row 20: SONDEPARAM surface-parcel LCL (data_quality = 0)**  
One sample = radiosonde launch.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,435 ± 396,1081,637 ± 483,-31.7,0.82,False
Spring,413 ± 244,361,490 ± 261,-15.7,0.93,False
Summer,268 ± 146,368,361 ± 165,-25.7,0.88,False
Fall,649 ± 686,178,917 ± 676,-29.2,1.02,False
Winter,617 ± 432,174,787 ± 428,-21.6,1.01,False


Launches failing data_quality: 8 of 1098


**Row 20: SONDEPARAM surface-parcel LCL, NO QC filter**  *(sensitivity test)*  
One sample = radiosonde launch.

...

**Row 20: Romps (2017) exact LCL from the lowest ~2 hPa of each sonde (m above launch level)**  *(sensitivity test)*  
One sample = radiosonde launch.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,435 ± 440,1098,637 ± 483,-31.8,0.91,False
Spring,384 ± 254,361,490 ± 261,-21.7,0.97,False
Summer,249 ± 156,370,361 ± 165,-30.9,0.95,False
Fall,718 ± 773,183,917 ± 676,-21.7,1.14,False
Winter,625 ± 447,184,787 ± 428,-20.6,1.04,False


**Row 20: 125 m/K × (T − Td) (Lawrence 2005)**  *(sensitivity test)*  
One sample = radiosonde launch.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,432 ± 446,1098,637 ± 483,-32.3,0.92,False
Spring,380 ± 254,361,490 ± 261,-22.5,0.98,False
Summer,244 ± 153,370,361 ± 165,-32.4,0.93,False
Fall,719 ± 786,183,917 ± 676,-21.6,1.16,False
Winter,624 ± 452,184,787 ± 428,-20.7,1.06,False


SONDEPARAM - Romps over 1081 launches: median 16 m, 5th-95th percentile -77 to 79 m


## 7. Cloud-base height (row 21)

Ceilometer at the Scripps Pier: `first_cbh` with QC = 0, counted when `detection_status` is 1–3 (one to three bases detected) and the base is ≤ `LOW_CLOUD_MAX_M`. Heights are above the instrument.

In [16]:
# =============================================================================
# Section 7: cloud-base height from the ceilometer (sheet row 21)
# =============================================================================
# Inputs
#   cbh_ceil_M1 = epcceilM1.b1, Vaisala ceilometer at the Scripps Pier, 16-s samples:
#     first_cbh         lowest cloud base (m ABOVE THE INSTRUMENT, not sea level), 0-7700 m
#     qc_first_cbh      QC bits: missing, < 0 m, > 7700 m (all "Bad"); we keep QC = 0
#     detection_status  0 clear, 1/2/3 one/two/three cloud bases, 4 obscured with no base
#                       (fog, heavy rain: first_cbh is then a visibility, not a base)
#   LOW_CLOUD_MAX_M     keep bases <= this height (3000 m, the Low Cloud Period bound)
#   AVG_PERIOD_CBH      averaging window in minutes; 0 = every kept 16-s sample on its own
# Q.ceilometer_cbh_m keeps first_cbh where qc_first_cbh == 0, detection_status is
# 1-3 (a cloud base was found) and the base is <= LOW_CLOUD_MAX_M, then averages
# over AVG_PERIOD_CBH minutes (0 = no averaging).
ceil = load_product("cbh_ceil_M1")
# What one value in the tables is, given AVG_PERIOD_CBH (minutes; 0 = no averaging)
CBH_SAMPLE = "16-s sample (no time averaging)" if AVG_PERIOD_CBH == 0 else f"{AVG_PERIOD_CBH:g}-min mean"

# --- 1. Primary: low cloud bases only ------------------------------------------
cbh, info = Q.ceilometer_cbh_m(ceil, max_cbh_m=LOW_CLOUD_MAX_M, window_min=AVG_PERIOD_CBH)
QC_INFO.extend(info["qc"])  # feeds the QC table in Section 11
check(21, cbh, f"Ceilometer first_cbh ≤ {LOW_CLOUD_MAX_M:g} m, QC = 0", f"{CBH_SAMPLE} (cloud base detected)")

# --- 2. Sensitivity: every detected base, any height ---------------------------
cbh_all, _ = Q.ceilometer_cbh_m(ceil, max_cbh_m=np.inf, window_min=AVG_PERIOD_CBH)
check(21, cbh_all, "Ceilometer first_cbh, any height, QC = 0", CBH_SAMPLE, primary=False)
del ceil

**Row 21: Ceilometer first_cbh ≤ 3500 m, QC = 0**  
One sample = 16-s sample (no time averaging) (cloud base detected).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,633 ± 562,925113,661 ± 316,-4.3,1.78,True
Spring,551 ± 353,347964,619 ± 270,-11.0,1.31,False
Summer,417 ± 397,299771,554 ± 254,-24.7,1.56,False
Fall,858 ± 730,97940,836 ± 389,2.7,1.88,True
Winter,1028 ± 747,179438,929 ± 362,10.6,2.06,False


**Row 21: Ceilometer first_cbh, any height, QC = 0**  *(sensitivity test)*  
One sample = 16-s sample (no time averaging).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,1077 ± 1573,1013401,661 ± 316,62.9,4.98,False
Spring,595 ± 598,350899,619 ± 270,-3.9,2.21,True
Summer,910 ± 1584,332079,554 ± 254,64.3,6.24,False
Fall,1686 ± 2094,116127,836 ± 389,101.6,5.38,False
Winter,1793 ± 1929,214296,929 ± 362,93.0,5.33,False


## 8. Rain (rows 22–24)

The sheet calls these "Rain Accumulation" in mm/hr and gives no definition. Several readings are computed from the same QC = 0 rain-rate series (1-min samples) and ranked by how well they reproduce the sheet. The ranking uses the median |% difference| of the means; the median log₂ std ratio shows whether the spread matches as well.

* LD M1 = Parsivel² `precip_rate` at the Pier (row 22)
* "LDS1" = Parsivel² at **Mt. Soledad (S2)** (row 23). ARM has no S1 facility for EPCAPE.
* VDIS = 2D video disdrometer, VDISQUANTS `rain_rate` at the Pier (row 24). There is no QC field. The VAP writes NaN, not 0, when no drops are detected, so NaN is read as "no rain" on days that have a file. The VDIS also records much less than the Parsivel during several storms (e.g. 21 Aug 2023: LD 14.6 mm, VDIS 0 mm), so its totals are lower.

In [17]:
# =============================================================================
# Section 8: rain (sheet rows 22-24)
# =============================================================================
# Inputs (1-min rain rates, mm/h)
#   row 22  rain_ld_M1   = epcldM1.b1      Parsivel2 laser disdrometer, Scripps Pier;
#                          precip_rate with qc_precip_rate -> QC = 0
#   row 23  rain_ld_S2   = epcldS2.b1      Parsivel2 at Mt. Soledad (the sheet's "LDS1");
#                          precip_rate with qc_precip_rate -> QC = 0
#   row 24  rain_vdis_M1 = epcvdisquantsM1.c1  2D video disdrometer VAP, Scripps Pier;
#                          rain_rate, no qc_ field; NaN when no drops are seen
# Settings
#   RAIN_THRESHOLD_MM_H  a 1-min sample is "raining" at >= this rate
#   RAIN_MAX_GAP_MIN     raining samples closer than this belong to one event
#   RAIN_MIN_MM          minimum accumulation of an event / rain day / wet hour
# The sheet does not define its rain statistic, so seven readings (A-G) are computed
# for each instrument and ranked by how well they reproduce the sheet.
RAIN = {22: ("rain_ld_M1", "precip_rate", "LD M1"),
        23: ("rain_ld_S2", "precip_rate", "LD S2"),
        24: ("rain_vdis_M1", "rain_rate", "VDIS M1")}
RAIN_RANKINGS, RAIN_CMPS = {}, {}  # per row: candidate ranking, and the comparison tables
for row, (product, var, label) in RAIN.items():
    display(Markdown(f"### Row {row}: {TABLE.loc[row, 'quantity']}  ({product}, `{var}`)"))
    # --- 1. Load the rain rate, QC = 0 (where a qc_ field exists) -----------------------
    ds = load_product(product)
    values, info = Q.qc0_values(ds, var)
    QC_INFO.append(info)  # feeds the QC table in Section 11
    rate = Q.to_series(values)
    if product == "rain_vdis_M1":
        # VDISQUANTS writes NaN (not 0) whenever no drops are detected (~97 % of 1-min samples),
        # so on days that have a file NaN means "no rain". Days without a file have no rows at all.
        rate = rate.fillna(0.0)
    # --- 2. Building blocks for the candidate statistics --------------------------------
    # rain events: raining samples merged across gaps <= RAIN_MAX_GAP_MIN; per event the
    #   accumulation (mm), duration and mean intensity (mm/h)
    events = Q.rain_events(rate, threshold_mm_h=RAIN_THRESHOLD_MM_H, max_gap_min=RAIN_MAX_GAP_MIN,
                           min_accum_mm=RAIN_MIN_MM)
    # daily (UTC) and hourly accumulations on wet days / hours; days or hours with < 80 %
    #   of the expected samples are dropped so outages do not look dry
    daily = Q.daily_rain_mm(rate, min_day_mm=RAIN_MIN_MM)
    hourly = Q.hourly_rain_mm(rate, min_hour_mm=RAIN_MIN_MM)
    # the 1-min rates while raining
    wet = Q.raining_rates(rate, threshold_mm_h=RAIN_THRESHOLD_MM_H)
    # --- 3. The seven candidate readings of the sheet's "Rain Accumulation (mm/hr)" -----
    candidates = {
        "A rain rate while raining (mm/h)": (wet, "1-min sample while raining"),
        "B event-mean intensity (mm/h)": (events["mean_rate_mm_h"], "rain event"),
        "C event accumulation (mm)": (events["accumulation_mm"], "rain event"),
        "D daily accumulation on rain days (mm/day)": (daily, "rain day"),
        "E hourly accumulation in wet hours (mm per hour)": (hourly, "wet hour"),
        "F monthly mean of D (mm/day), std across months": (S.monthly_means(daily), "calendar month"),
        "G monthly mean of A (mm/h), std across months": (S.monthly_means(wet), "calendar month"),
    }
    # --- 4. Compare each with the sheet; the primary is chosen across all three rows below
    ranking, cmps = compare_candidates(row, candidates, label=f"{label}: ", mark_primary=False)
    RAIN_RANKINGS[row], RAIN_CMPS[row] = ranking, cmps
    print(f"{len(events)} rain events, {len(daily)} rain days; campaign total "
          f"{np.nansum(rate) * Q._step_hours(rate):.0f} mm (QC = 0 samples)")
    del ds

# --- 5. One definition for all three instruments --------------------------------------
# Kavin presumably used ONE method for all three instruments, so the primary
# definition is the candidate letter that best reproduces rows 22-24 together:
# the MEDIAN over the three rows of each row's median |% difference|, so one row
# that no definition reproduces (row 23) cannot decide the choice.
letters = pd.concat({r: rk.assign(letter=rk["definition"].str.split(": ").str[1].str[0])
                     for r, rk in RAIN_RANKINGS.items()}, names=["sheet_row"])
overall = letters.groupby("letter")["median_abs_diff_pct"].median().sort_values()
BEST_RAIN = overall.index[0]
# mark that letter primary in every row's comparisons (used by the summary and the figure)
for cmps in RAIN_CMPS.values():
    for c in cmps:
        c["primary"] = c["definition"].iloc[0].split(": ")[1].startswith(BEST_RAIN)
display(Markdown(f"Median over rows 22-24 of each row's median |% difference|, per candidate. **Primary: {BEST_RAIN}**"))
display(overall.round(1).to_frame("median over rows 22-24 of median |diff| (%)"))

### Row 22: Rain Accumulation - LDM1  (rain_ld_M1, `precip_rate`)

season,EPCAPE,Spring,Summer,Fall,Winter
LD M1: A rain rate while raining (mm/h),1.7 ± 2.9,0.6 ± 1.3,1.3 ± 2.4,2.0 ± 4.4,1.8 ± 2.9
LD M1: B event-mean intensity (mm/h),1.4 ± 1.9,1.6 ± 3.7,0.8 ± 1.6,2.0 ± 2.0,1.3 ± 1.6
LD M1: C event accumulation (mm),2.7 ± 6.0,0.63 ± 0.74,3.2 ± 9.6,1.8 ± 2.5,3.1 ± 6.4
LD M1: D daily accumulation on rain days (mm/day),4.0 ± 6.5,0.63 ± 0.71,3.5 ± 6.7,2.0 ± 2.6,5.6 ± 7.8
LD M1: E hourly accumulation in wet hours (mm per hour),1.1 ± 1.3,0.42 ± 0.36,1.1 ± 1.5,1.1 ± 1.6,1.2 ± 1.3
"LD M1: F monthly mean of D (mm/day), std across months",3.4 ± 3.3,0.73 ± 0.47,3.9 ± 6.0,2.7 ± 2.3,5.6 ± 1.4
"LD M1: G monthly mean of A (mm/h), std across months",1.50 ± 0.92,0.58 ± 0.23,0.98 ± 0.66,2.3 ± 1.0,1.99 ± 0.55
SHEET (Kavin),3.1 ± 3.4,0.2 ± 0.1,3.6 ± 5.5,1.7 ± 1.7,5.8 ± 1.9


,definition,median_abs_diff_pct,max_abs_diff_pct,median_abs_log2_std_ratio,seasons_compared
0,"LD M1: F monthly mean of D (mm/day), std across months",10.52,265.74,0.39,5
1,LD M1: C event accumulation (mm),12.43,216.65,0.83,5
2,LD M1: D daily accumulation on rain days (mm/day),20.19,213.49,0.94,5
3,LD M1: A rain rate while raining (mm/h),62.69,199.60,1.23,5
4,"LD M1: G monthly mean of A (mm/h), std across months",65.76,191.61,1.80,5
5,LD M1: E hourly accumulation in wet hours (mm per hour),70.61,111.46,1.37,5
6,LD M1: B event-mean intensity (mm/h),77.19,680.76,0.84,5


109 rain events, 76 rain days; campaign total 302 mm (QC = 0 samples)


### Row 23: Rain Accumulation - LDS1  (rain_ld_S2, `precip_rate`)

...

season,EPCAPE,Spring,Summer,Fall,Winter
VDIS M1: A rain rate while raining (mm/h),1.7 ± 3.2,0.7 ± 1.3,0.7 ± 1.2,3.2 ± 5.4,2.0 ± 3.4
VDIS M1: B event-mean intensity (mm/h),1.4 ± 1.6,0.51 ± 0.93,1.0 ± 1.9,2.9 ± 2.3,1.4 ± 1.5
VDIS M1: C event accumulation (mm),2.0 ± 3.1,0.8 ± 1.1,1.1 ± 2.2,2.3 ± 1.8,2.2 ± 3.5
VDIS M1: D daily accumulation on rain days (mm/day),3.3 ± 3.9,0.9 ± 1.1,1.5 ± 2.5,2.8 ± 2.0,4.4 ± 4.5
VDIS M1: E hourly accumulation in wet hours (mm per hour),1.0 ± 1.1,0.46 ± 0.55,0.53 ± 0.46,1.6 ± 1.1,1.0 ± 1.1
"VDIS M1: F monthly mean of D (mm/day), std across months",2.6 ± 2.1,1.01 ± 0.61,2.5 ± 3.4,2.4 ± 1.6,4.0 ± 1.4
"VDIS M1: G monthly mean of A (mm/h), std across months",1.53 ± 0.95,0.67 ± 0.25,0.76 ± 0.43,3.01 ± 0.31,2.02 ± 0.15
SHEET (Kavin),3.4 ± 3.3,.39 ± 0.19,2.3 ± 3.6,2.6 ± 1.7,7.0 ± 2.4


,definition,median_abs_diff_pct,max_abs_diff_pct,median_abs_log2_std_ratio,seasons_compared
0,"VDIS M1: F monthly mean of D (mm/day), std across months",22.89,159.15,0.66,5
1,VDIS M1: D daily accumulation on rain days (mm/day),34.96,133.53,0.55,5
2,VDIS M1: C event accumulation (mm),51.14,114.93,0.54,5
3,VDIS M1: B event-mean intensity (mm/h),57.86,79.31,0.93,5
4,"VDIS M1: G monthly mean of A (mm/h), std across months",66.98,72.73,2.44,5
5,VDIS M1: A rain rate while raining (mm/h),68.64,81.60,1.59,5
6,VDIS M1: E hourly accumulation in wet hours (mm per hour),71.89,85.49,1.53,5


86 rain events, 53 rain days; campaign total 175 mm (QC = 0 samples)


Median over rows 22-24 of each row's median |% difference|, per candidate. **Primary: F**

,median over rows 22-24 of median |diff| (%)
letter,
F,22.9
D,35.0
C,51.1
B,58.4
A,62.7
G,65.8
E,70.6


## 9. FM-120 fog monitor: on-site cloud and haze sampling hours (rows 31–32)

The DMT FM-120 fog monitor sat on the Russell instrument van at Mt. Soledad, next to the GCVI inlet. It measured droplets 2–50 µm across. The data come from Lauren Robinson and Rachel Chang (Dalhousie) as daily files of **5-min averages** of 1-s data, in `data/processed/Daily Averaged FM-120 FIles/` (see `readme_FM120.md`). Rows exist only while the instrument ran, and there is no QC field.

**Sorting each 5-min interval into cloud or haze** follows the operational definitions in Lynn Russell's EPCAPE BAMS paper. They use the Mt. Soledad **visibility** record (`msd_visibility_*.csv`, ~1-s samples in metres, 2023-02-24 to 2024-02-14) together with the FM-120 LWC:

* **cloud:** visibility < `FM120_CLOUD_VIS_MAX_M` (1 km) **and** LWC > `FM120_CLOUD_LWC_MIN_GM3` (0.01 g m⁻³);
* **haze:** visibility < `FM120_HAZE_VIS_MAX_M` (5 km) **and** LWC < `FM120_HAZE_LWC_MAX_GM3` (0.01 g m⁻³).

The two classes cannot overlap: cloud needs LWC above 0.01 g m⁻³ and haze needs it below. Two kinds of interval fall in between, and their hours are reported for context:
* visibility 1–5 km with LWC ≥ 0.01 g m⁻³ is *neither* cloud nor haze;
* visibility < 1 km with LWC < 0.01 g m⁻³ counts as *haze*.

An interval's visibility is the **median** of its 1-s samples (`FM120_VIS_STAT`), so "below 1 km" means below 1 km for more than half the interval. Samples of 0 m and spikes above the sensor's 48,270 m ceiling (30 statute miles) are dropped. An interval needs at least `FM120_VIS_MIN_SAMPLES` valid samples. Hours are 5 min per classified interval. The FM-120 ran for eight days (16–23 Feb 2023) before the visibility record starts; those intervals cannot be classified.

**Validation of the file reading:** the cell first reproduces Table 1 of the PI's report (DOE/SC-ARM-24-023): monthly hours of operation, and hours in cloud with the PI's definition (N > 1 cm⁻³ and LWC > 0.05 g m⁻³).

**Sensitivity tests:**
* the mean instead of the median visibility of each interval;
* the reading that reproduces the sheet's haze row to within ~2 % in every season: 1 km < visibility < 5 km with **no** LWC condition, using the 5-min mean visibility;
* the PI's cloud definition;
* **filter set 2** (Andrew, 2026-10-08). Cloud is unchanged. Haze is (1 km < visibility < 5 km and LWC < 0.01 g m⁻³) or (visibility < 1 km and LWC < 0.01 g m⁻³). Joined, those are visibility < 5 km and LWC < 0.01 g m⁻³ except a visibility of exactly 1 km, so with these settings they equal the BAMS-paper haze definition apart from that one value.

Section 9b re-runs Kavin's own code for these hours. Section 9c re-runs his activated fractions (rows 27 and 30), and Section 9d derives droplet number and effective diameter (rows 25, 26, 28, 29). Section 9e derives the cloud and haze LWC rows (54, 55).

In [18]:
# =============================================================================
# Section 9: FM-120 on-site cloud and haze sampling hours (sheet rows 31-32)
# =============================================================================
# Inputs
#   data/processed/Daily Averaged FM-120 FIles/YYYY-MM-DD.csv  (config.yaml: fm120.folder)
#     DMT FM-120 fog monitor at Mt. Soledad (Lauren Robinson and Rachel Chang, Dalhousie).
#     5-min averages of 1-s data, one row per interval the instrument ran, labelled by the
#     interval START (2023-02-16 16:40 to 2024-02-15 17:00 UTC); no QC field. Used here:
#       lwc_gm3  liquid water content (g m-3);  nd_cm3  droplet number, 2-50 um (# cm-3)
#   data/processed/msd_visibility_*.csv  (config.yaml: fm120.visibility_files)
#     Mt. Soledad visibility, ~1-s samples in metres, 2023-02-24 to 2024-02-14 UTC.
#   FM120_CLOUD_VIS_MAX_M, FM120_CLOUD_LWC_MIN_GM3  cloud = visibility < 1 km AND LWC > 0.01 g m-3
#   FM120_HAZE_VIS_MAX_M, FM120_HAZE_LWC_MAX_GM3    haze  = visibility < 5 km AND LWC < 0.01 g m-3
#     (the operational definitions of Lynn Russell's EPCAPE BAMS paper)
#   FM120_VIS_STAT, FM120_VIS_MIN_SAMPLES           how an interval's visibility is formed
# Each 5-min interval classified as cloud (or haze) adds 5 min to that season's hours.

# --- 1. Load the FM-120 record (sources.load_fm120 handles the file quirks) -------------
fm = SRC.load_fm120()
print(f"FM-120: {len(fm):,} 5-min intervals, {fm.index[0]} to {fm.index[-1]} UTC")
month = fm.index.to_period("M")  # calendar month of each interval, for the validation table

# --- 2. Validation: reproduce Table 1 of DOE/SC-ARM-24-023 (Chang 2024) ------------------
# Monthly hours of operation and hours in cloud with the PI's definition
# (N > 1 cm-3 and LWC > 0.05 g m-3). Matching it confirms the files, the 5-min interval
# length and the time handling. The haze thresholds are set to infinity so nothing is haze.
CHANG_TABLE1 = {  # month: (hours of operation, hours in cloud), copied from the report
    "2023-02": (80.67, 0.25), "2023-03": (665.8, 35.8), "2023-04": (601.4, 70.4), "2023-05": (591.8, 15.9),
    "2023-06": (711.7, 7.50), "2023-07": (735.6, 77.3), "2023-08": (681.7, 37.6), "2023-09": (705.3, 7.50),
    "2023-10": (551.5, 29.1), "2023-11": (711.6, 9.92), "2023-12": (675.9, 2.00), "2024-01": (438.6, 9.58),
    "2024-02": (353.1, 4.08),
}
chang = Q.fm120_classify(fm, cloud_nd_min_cm3=1.0, cloud_lwc_min_gm3=0.05,
                         haze_nd_min_cm3=np.inf, haze_lwc_min_gm3=np.inf)
table1 = pd.DataFrame({
    "operating h (this check)": Q.interval_hours(chang["operating"]).groupby(month).sum(),
    "operating h (Chang Table 1)": pd.Series({pd.Period(k, "M"): v[0] for k, v in CHANG_TABLE1.items()}),
    "cloud h (this check)": Q.interval_hours(chang["cloud"]).groupby(month).sum(),
    "cloud h (Chang Table 1)": pd.Series({pd.Period(k, "M"): v[1] for k, v in CHANG_TABLE1.items()}),
}).round(2)
table1.loc["total"] = table1.sum()
display(Markdown("**Validation against DOE/SC-ARM-24-023, Table 1 (cloud = N > 1 cm⁻³ and LWC > 0.05 g m⁻³)**"))
display(table1)

# --- 3. Visibility on the FM-120 intervals -------------------------------------------------
# ~1-s samples -> one median and one mean per 5-min interval [t, t + 5 min), matching the
# FM-120 interval labels. Zeros and spikes above 48,270 m are dropped; intervals with fewer
# than FM120_VIS_MIN_SAMPLES valid samples get NaN (and cannot be classified).
vis_1s = SRC.load_visibility_msd()
vis = Q.visibility_on_intervals(vis_1s, fm.index, interval_min=5.0, min_samples=FM120_VIS_MIN_SAMPLES)
del vis_1s  # ~25 million samples; only the 5-min statistics are needed
print(f"FM-120 intervals with usable visibility: {vis['vis_median_m'].notna().sum():,} of {len(fm):,}")

# --- 4. The sheet's rows 31-32 ---------------------------------------------------------
# They are attributed to "Fig. 5", not Kavin, so they are read separately. Their campaign
# cell is a formula (=sum(F:I)), which the reader returns as NaN, so the four seasons are added.
FIG5 = S.read_table_rows(TABLE_XLSX, author_contains="Fig. 5")


def sheet_hours(row):
    """The sheet's hours for one row, with the campaign value = sum of the four seasons."""
    v = S.table_values(FIG5, row).copy()
    v.loc["EPCAPE", "mean"] = v.loc[["Spring", "Summer", "Fall", "Winter"], "mean"].sum()
    v["raw"] = [f"{x:g}" for x in v["mean"]]
    return v


def compare_hours(row, mask, definition, primary):
    """Hours per season of the True intervals in `mask`, next to sheet row `row`."""
    hours = S.seasonal_sums(Q.interval_hours(mask))  # season -> hours
    n_intervals = S.seasonal_sums(mask.astype(float))  # season -> number of 5-min intervals
    ours = pd.DataFrame({"mean": hours, "std": np.nan, "median": np.nan, "n": n_intervals})
    cmp = S.compare_to(ours, sheet_hours(row), row, FIG5.loc[row, "quantity"], definition, close_pct=CLOSE_PCT)
    cmp = cmp.assign(sample="5-min interval (summed to hours)", primary=primary)
    SUMMARY.append(cmp)
    tag = "" if primary else "  *(sensitivity test)*"
    display(Markdown(f"**Row {row}: {definition}**{tag}  \nHours = number of 5-min intervals × 5 min."))
    display(S.show(cmp)[["ours", "n", "table", "diff_pct", "close"]])
    return cmp


def classify(vis_stat, **thresholds):
    """fm120_classify_visibility on the `vis_stat` ("median" or "mean") interval visibility.

    Thresholds not given default to the settings (the BAMS-paper definitions)."""
    kw = dict(cloud_vis_max_m=FM120_CLOUD_VIS_MAX_M, cloud_lwc_min_gm3=FM120_CLOUD_LWC_MIN_GM3,
              haze_vis_max_m=FM120_HAZE_VIS_MAX_M, haze_lwc_max_gm3=FM120_HAZE_LWC_MAX_GM3)
    kw.update(thresholds)
    return Q.fm120_classify_visibility(fm, vis[f"vis_{vis_stat}_m"], **kw)


# Text of the definitions, for the table titles
CLOUD_DEF = f"visibility < {FM120_CLOUD_VIS_MAX_M / 1e3:g} km and LWC > {FM120_CLOUD_LWC_MIN_GM3:g} g m⁻³"
HAZE_DEF = f"visibility < {FM120_HAZE_VIS_MAX_M / 1e3:g} km and LWC < {FM120_HAZE_LWC_MAX_GM3:g} g m⁻³"

# --- 5. Primary: the BAMS-paper definitions, hours per season ------------------------------
cls = classify(FM120_VIS_STAT)
compare_hours(31, cls["cloud"], f"FM-120 cloud: {FM120_VIS_STAT} {CLOUD_DEF}", primary=True)
compare_hours(32, cls["haze"], f"FM-120 haze: {FM120_VIS_STAT} {HAZE_DEF}", primary=True)

# --- 6. Sensitivity tests ------------------------------------------------------------
# 6a. interval visibility from the other statistic (mean <-> median), same thresholds
other_stat = "mean" if FM120_VIS_STAT == "median" else "median"
cls_other = classify(other_stat)
compare_hours(31, cls_other["cloud"], f"FM-120 cloud: {other_stat} {CLOUD_DEF}", primary=False)
compare_hours(32, cls_other["haze"], f"FM-120 haze: {other_stat} {HAZE_DEF}", primary=False)
# 6b. the haze reading that reproduces the sheet's row 32 to within ~2 % in every season:
#     1 km < visibility < 5 km with NO LWC condition, on the 5-min MEAN visibility (found by
#     trying both statistics, with and without the LWC < 0.01 condition and the 1-km lower
#     bound). With the mean visibility the paper's cloud definition (6a when the primary is the
#     median) also matches row 31 to within ~1 %. An inference about how "Fig. 5" was made.
cls_sheet = classify("mean", haze_vis_min_m=FM120_CLOUD_VIS_MAX_M, haze_lwc_max_gm3=np.inf)
compare_hours(32, cls_sheet["haze"], f"FM-120 haze: {FM120_CLOUD_VIS_MAX_M / 1e3:g} km < mean visibility < "
              f"{FM120_HAZE_VIS_MAX_M / 1e3:g} km, no LWC condition [best match to the sheet]", primary=False)
# 6c. the PI's documented cloud definition (no visibility): N > 1 cm-3 and LWC > 0.05 g m-3
compare_hours(31, chang["cloud"], "FM-120 cloud, PI definition: N > 1 cm⁻³ and LWC > 0.05 g m⁻³ "
              "(DOE/SC-ARM-24-023)", primary=False)

# 6d. Andrew's filter set 2 (settings FM120_HAZE2_*): cloud unchanged; haze = (1 km < visibility < 5 km and
#     LWC < 0.01 g m-3) OR (visibility < 1 km and LWC < 0.01 g m-3). The two sets are joined (a row in either
#     one is haze), so together they are visibility < 5 km and LWC < 0.01 g m-3 except a visibility of exactly
#     1 km: with these settings, the BAMS-paper haze definition apart from that one value.
#     Same interval visibility as the primary (FM120_VIS_STAT of the screened 1-s samples).
vis_set2 = vis[f"vis_{FM120_VIS_STAT}_m"].reindex(fm.index)
lwc_set2 = fm["lwc_gm3"]
SET2 = {
    "cloud": cls["cloud"],  # unchanged: visibility < 1 km and LWC > 0.01 g m-3
    "haze": cls["operating"] & (
        ((vis_set2 > FM120_CLOUD_VIS_MAX_M) & (vis_set2 < FM120_HAZE2_VIS_MAX_M) & (lwc_set2 < FM120_HAZE2_LWC_MAX_GM3))
        | ((vis_set2 < FM120_CLOUD_VIS_MAX_M) & (lwc_set2 < FM120_HAZE2_LWC_MAX_GM3))),
}
SET2_HAZE_DEF = (f"({FM120_CLOUD_VIS_MAX_M / 1e3:g} km < {FM120_VIS_STAT} visibility < {FM120_HAZE2_VIS_MAX_M / 1e3:g} km "
                 f"and LWC < {FM120_HAZE2_LWC_MAX_GM3:g} g m⁻³) or ({FM120_VIS_STAT} visibility < "
                 f"{FM120_CLOUD_VIS_MAX_M / 1e3:g} km and LWC < {FM120_HAZE2_LWC_MAX_GM3:g} g m⁻³)")
compare_hours(31, SET2["cloud"], f"FM-120 cloud, filter set 2 (cloud unchanged): {FM120_VIS_STAT} {CLOUD_DEF}",
              primary=False)
compare_hours(32, SET2["haze"], f"FM-120 haze, filter set 2: {SET2_HAZE_DEF}", primary=False)

# --- 7. Context: hours per season in each class (primary definitions) -------------------------
vis_primary = vis[f"vis_{FM120_VIS_STAT}_m"].reindex(fm.index)
context = pd.DataFrame({
    "FM-120 operating": S.seasonal_sums(Q.interval_hours(cls["operating"])),
    "... with usable visibility": S.seasonal_sums(Q.interval_hours(cls["has_vis"])),
    f"cloud ({CLOUD_DEF})": S.seasonal_sums(Q.interval_hours(cls["cloud"])),
    f"haze ({HAZE_DEF})": S.seasonal_sums(Q.interval_hours(cls["haze"])),
    "  of which visibility < 1 km": S.seasonal_sums(Q.interval_hours(cls["haze"] & (vis_primary < FM120_CLOUD_VIS_MAX_M))),
    "neither (visibility < 5 km, too wet for haze, too clear for cloud)": S.seasonal_sums(Q.interval_hours(cls["low_vis_neither"])),
    "sheet 31 (cloud)": sheet_hours(31)["mean"],
    "sheet 32 (haze)": sheet_hours(32)["mean"],
}).round(0)
display(Markdown(f"Hours per season, for context ({FM120_VIS_STAT} visibility of each interval):"))
display(context.T)

# --- 8. The other FM-120 rows are derived in the next sections -----------------------------
# 9b: Kavin's code for these hours; 9c: activated fractions (rows 27, 30); 9d: droplet number and
# effective diameter (rows 25, 26, 28, 29); 9e: liquid water content (rows 54, 55).

FM-120: 90,009 5-min intervals, 2023-02-16 16:40:00 to 2024-02-15 17:00:00 UTC


**Validation against DOE/SC-ARM-24-023, Table 1 (cloud = N > 1 cm⁻³ and LWC > 0.05 g m⁻³)**

,operating h (this check),operating h (Chang Table 1),cloud h (this check),cloud h (Chang Table 1)
2023-02,78.67,80.67,0.25,0.25
2023-03,665.83,665.80,35.75,35.80
2023-04,599.58,601.40,70.42,70.40
2023-05,591.83,591.80,15.92,15.90
2023-06,711.67,711.70,7.50,7.50
2023-07,735.58,735.60,77.33,77.30
2023-08,681.67,681.70,37.58,37.60
2023-09,705.33,705.30,7.50,7.50
2023-10,551.50,551.50,29.08,29.10
2023-11,711.50,711.60,9.92,9.92


FM-120 intervals with usable visibility: 87,866 of 90,009


**Row 31: FM-120 cloud: median visibility < 1 km and LWC > 0.01 g m⁻³**  
Hours = number of 5-min intervals × 5 min.

...

,ours,n,table,diff_pct,close
season,,,,,
EPCAPE,480,5757,470,2.1,True
Spring,137,1644,134,2.2,True
Summer,217,2605,212,2.4,True
Fall,66.8,802,66,1.3,True
Winter,58.8,706,58,1.4,True


**Row 32: FM-120 haze, filter set 2: (1 km < median visibility < 5 km and LWC < 0.01 g m⁻³) or (median visibility < 1 km and LWC < 0.01 g m⁻³)**  *(sensitivity test)*  
Hours = number of 5-min intervals × 5 min.

,ours,n,table,diff_pct,close
season,,,,,
EPCAPE,395,4737,418,-5.6,True
Spring,77.6,931,85,-8.7,True
Summer,161,1929,172,-6.5,True
Fall,56,672,49,14.3,False
Winter,100,1205,112,-10.3,False


Hours per season, for context (median visibility of each interval):

,EPCAPE,Spring,Summer,Fall,Winter
FM-120 operating,7484.0,1903.0,2123.0,1939.0,1519.0
... with usable visibility,7322.0,1851.0,2085.0,1925.0,1461.0
cloud (visibility < 1 km and LWC > 0.01 g m⁻³),480.0,137.0,217.0,67.0,59.0
haze (visibility < 5 km and LWC < 0.01 g m⁻³),395.0,78.0,161.0,56.0,100.0
of which visibility < 1 km,58.0,4.0,24.0,11.0,20.0
"neither (visibility < 5 km, too wet for haze, too clear for cloud)",76.0,9.0,31.0,4.0,31.0
sheet 31 (cloud),470.0,134.0,212.0,66.0,58.0
sheet 32 (haze),418.0,85.0,172.0,49.0,112.0


### 9b. Kavin's code for the cloud and haze hours

His code ("Hours Calculations (Big File)") reads the same FM-120 and visibility records. It differs from Section 9 in the ways below; `kavin.py` reproduces each of them:

* **Visibility of a row** = the **mean** of the raw ~1-s samples from the row's time to the **next row's** time, both ends included. Across an FM-120 gap the window therefore spans the whole gap. Zeros and the ~10⁸ m spikes are **not** removed.
* Rows before the second visibility file starts read the first file. The row at its start (2023-07-19 04:00 UTC) is set to **0 m**, and the last row keeps its initial 0 m.
* **cloud** = visibility < 1 km and LWC > 0.01 g m⁻³; **haze** = 1 km < visibility < 5 km, with **no LWC condition**. This differs from the BAMS-paper text, which requires LWC < 0.01 g m⁻³ for haze and has no 1 km lower bound.
* Seasons are **calendar quarters** of the UTC time stamp. Every row counts, including 15 Feb 2024.
* The sheet holds the hours **truncated** to whole hours, not rounded, and its campaign cell is the sum of the four truncated seasons.

In [19]:
# =============================================================================
# Section 9b: FM-120 cloud and haze hours with Kavin's code (sheet rows 31-32)
# =============================================================================
# Inputs
#   fm (Section 9)                  FM-120 5-min rows: the same data as Kavin's monthly files
#   msd_visibility_*.csv            the two raw visibility files, read separately and unscreened,
#                                   as Kavin reads them (SRC.load_visibility_msd(separate=True))
# K.kavin_interval_visibility reproduces his avgvis; K.kavin_fm120_hours his hour counts.

# --- 1. Visibility per FM-120 row, as Kavin's code computes it ----------------------------------
vis_files = SRC.load_visibility_msd(separate=True)
avgvis_kavin = K.kavin_interval_visibility(fm.index, vis_files)
del vis_files  # ~25 million samples
spans_gap = (pd.Series(fm.index).diff().shift(-1) > pd.Timedelta(minutes=5)).sum()
print(f"Rows whose visibility window spans an FM-120 gap (next row > 5 min later): {spans_gap}")

# --- 2. Hours per calendar quarter, counted as in his code ----------------------------------------
HOURS_KAVIN = K.kavin_fm120_hours(fm["lwc_gm3"], avgvis_kavin)
for row, kind, definition in (
    (31, "cloud", "Kavin's code: raw mean visibility, row to next row, < 1 km and LWC > 0.01 g m⁻³; calendar quarters"),
    (32, "haze", "Kavin's code: 1 km < raw mean visibility, row to next row, < 5 km, no LWC condition; calendar quarters"),
):
    ours = pd.DataFrame({"mean": HOURS_KAVIN[f"{kind}_h"], "std": np.nan, "median": np.nan, "n": HOURS_KAVIN[f"{kind}_n"]})
    cmp = S.compare_to(ours, sheet_hours(row), row, FIG5.loc[row, "quantity"], definition, close_pct=CLOSE_PCT)
    SUMMARY.append(cmp.assign(sample="5-min FM-120 row (summed to hours)", primary=False))
    display(Markdown(f"**Row {row}: {definition}**  *(Kavin's code on the same data)*"))
    display(S.show(cmp)[["ours", "n", "table", "diff_pct", "close"]])

# --- 3. The sheet holds these hours truncated to whole hours ----------------------------------
seasons4 = S.SEASON_ORDER[1:]  # Spring, Summer, Fall, Winter
truncated = pd.DataFrame({
    "cloud, Kavin's code (h)": HOURS_KAVIN["cloud_h"].round(2),
    "cloud, truncated": np.floor(HOURS_KAVIN["cloud_h"]),
    "sheet 31": sheet_hours(31)["mean"],
    "haze, Kavin's code (h)": HOURS_KAVIN["haze_h"].round(2),
    "haze, truncated": np.floor(HOURS_KAVIN["haze_h"]),
    "sheet 32": sheet_hours(32)["mean"],
})
# the sheet's campaign cell = sum of the four truncated seasons
truncated.loc["EPCAPE", ["cloud, truncated", "haze, truncated"]] = [
    np.floor(HOURS_KAVIN.loc[seasons4, "cloud_h"]).sum(), np.floor(HOURS_KAVIN.loc[seasons4, "haze_h"]).sum()]
display(Markdown("Truncated to whole hours (campaign = sum of the truncated seasons), Kavin's code reproduces the sheet:"))
display(truncated.T)

# --- 4. From Kavin's counting to Section 9's definitions, on the same files ------------------------
cls_kavin_thresholds = classify("mean", haze_vis_min_m=FM120_CLOUD_VIS_MAX_M, haze_lwc_max_gm3=np.inf)
HOURS_STEPS = pd.DataFrame({
    "cloud: Kavin's code (calendar quarters)": HOURS_KAVIN["cloud_h"],
    "cloud: 5-min mean visibility, screened (sheet seasons)": S.seasonal_sums(Q.interval_hours(cls_kavin_thresholds["cloud"])),
    f"cloud: BAMS-paper definition, {FM120_VIS_STAT} visibility (Section 9)": S.seasonal_sums(Q.interval_hours(cls["cloud"])),
    "haze: Kavin's code (calendar quarters)": HOURS_KAVIN["haze_h"],
    "haze: 1–5 km, no LWC condition, 5-min mean visibility, screened": S.seasonal_sums(Q.interval_hours(cls_kavin_thresholds["haze"])),
    f"haze: BAMS-paper definition, {FM120_VIS_STAT} visibility (Section 9)": S.seasonal_sums(Q.interval_hours(cls["haze"])),
}).T
display(Markdown("**Hours step by step, on the same FM-120 and visibility files**"))
display(HOURS_STEPS[S.SEASON_ORDER].round(1))

Rows whose visibility window spans an FM-120 gap (next row > 5 min later): 65


**Row 31: Kavin's code: raw mean visibility, row to next row, < 1 km and LWC > 0.01 g m⁻³; calendar quarters**  *(Kavin's code on the same data)*

,ours,n,table,diff_pct,close
season,,,,,
EPCAPE,473,5672,470,0.6,True
Spring,135,1619,134,0.7,True
Summer,213,2555,212,0.4,True
Fall,66.2,794,66,0.3,True
Winter,58.7,704,58,1.1,True


**Row 32: Kavin's code: 1 km < raw mean visibility, row to next row, < 5 km, no LWC condition; calendar quarters**  *(Kavin's code on the same data)*

,ours,n,table,diff_pct,close
season,,,,,
EPCAPE,420,5040,418,0.5,True
Spring,85.2,1022,85,0.2,True
Summer,173,2074,172,0.5,True
Fall,49.8,598,49,1.7,True
Winter,112,1346,112,0.1,True


Truncated to whole hours (campaign = sum of the truncated seasons), Kavin's code reproduces the sheet:

,EPCAPE,Spring,Summer,Fall,Winter
"cloud, Kavin's code (h)",472.67,134.92,212.92,66.17,58.67
"cloud, truncated",470.00,134.00,212.00,66.00,58.00
sheet 31,470.00,134.00,212.00,66.00,58.00
"haze, Kavin's code (h)",420.00,85.17,172.83,49.83,112.17
"haze, truncated",418.00,85.00,172.00,49.00,112.00
sheet 32,418.00,85.00,172.00,49.00,112.00


**Hours step by step, on the same FM-120 and visibility files**

,EPCAPE,Spring,Summer,Fall,Winter
cloud: Kavin's code (calendar quarters),472.7,134.9,212.9,66.2,58.7
"cloud: 5-min mean visibility, screened (sheet seasons)",472.5,134.8,212.8,66.2,58.7
"cloud: BAMS-paper definition, median visibility (Section 9)",479.8,137.0,217.1,66.8,58.8
haze: Kavin's code (calendar quarters),420.0,85.2,172.8,49.8,112.2
"haze: 1–5 km, no LWC condition, 5-min mean visibility, screened",420.2,85.2,173.0,49.8,112.2
"haze: BAMS-paper definition, median visibility (Section 9)",394.8,77.6,160.8,56.0,100.5


### 9c. Cloud and haze activated fraction (rows 27 and 30)

The activated fractions come from `ET_Dcrit_nativetime.csv`, produced by Abbey Williams' code and passed on by Kavin, with one row per sample. Kavin's code ("Critical Diameters & % Activated Code from Abbey") takes the rows of one class (`ClrCldHaz_class` 1 = cloud, 2 = haze) that have both `Dta` and `Dta_critical`. The activated fraction is the share of those rows with **`Dta` > `Dta_critical`**. It counts **samples, not time**. His code computes only the campaign value; here the same count is applied within each season.

**What this does and does not check.** It repeats Kavin's arithmetic on Abbey's file. It is **not** an independent check of `Dta` or `Dta_critical`: neither is defined in the file, and the code that made them was not shared. `Dta_critical` exists only for **June–September and December 2023**, so "spring" means June and "fall" means December.

In [20]:
# =============================================================================
# Section 9c: cloud and haze activated fraction (sheet rows 27 and 30)
# =============================================================================
# Inputs
#   KAVIN_DIR/ET_Dcrit_nativetime.csv  from Abbey Williams' code (via Kavin), one row per sample
#                                      (K.read_dcrit_native): Dta and Dta_critical (diameters,
#                                      presumably nm; not defined in the file), ClrCldHaz_class
#                                      (1 = cloud, 2 = haze, empty = neither)
# K.activated_flags: 1 where Dta > Dta_critical, 0 where Dta <= Dta_critical, for one class's rows
# with both diameters; the mean of the flags over a season is that season's activated fraction.
#
# LIMITED DATA: Dta_critical is present in only 12,282 of the file's 114,528 rows, all of them in
# Jun-Sep and Dec 2023 (95 % in Jun-Aug; see the coverage table printed below). So:
#   - "spring" is June alone (941 cloud and 19 haze samples);
#   - "fall" is December alone (138 cloud samples, no haze);
#   - winter has no samples, and fall and winter have no haze value;
#   - the campaign value is in effect a summer (Jun-Aug) value.
# The sheet's seasonal activated fractions inherit all of this. They should not be read as
# seasonal climatologies.

# --- 1. Read the file; when are both diameters available? ---------------------------------
DCRIT = K.read_dcrit_native(KAVIN_DIR)
has_both = DCRIT["Dta"].notna() & DCRIT["Dta_critical"].notna()
both_rows = DCRIT[has_both]
print(f"{len(DCRIT):,} rows, {DCRIT.index.min()} to {DCRIT.index.max()}; both diameters in {has_both.sum():,}; "
      f"repeated time stamps {DCRIT.index.duplicated().sum():,}; fully repeated rows "
      f"{DCRIT.reset_index().duplicated().sum():,}")
display(Markdown("Samples with both diameters, by month and class:"))
display(both_rows.groupby([both_rows.index.to_period("M"),
                           both_rows["ClrCldHaz_class"].map(K.CLASS_CODES).fillna("no class")]).size().unstack(fill_value=0))

# --- 2. Activated fraction per season (rows 27 and 30) ------------------------------------------
#        The sheet gives no std for a fraction, and the std of 0/1 flags means little, so std is left out.
DCRIT_DEDUP = DCRIT.reset_index().drop_duplicates().set_index("time")  # fully repeated rows counted once
for row, class_code in ((27, 1), (30, 2)):
    kind = K.CLASS_CODES[class_code]
    check_stats(row, S.seasonal_stats(K.activated_flags(DCRIT, class_code)).assign(std=np.nan),
                f"{kind}: fraction of {kind}-class samples with Dta > Dta_critical (Abbey Williams' "
                "ET_Dcrit_nativetime.csv; Kavin's count, applied per season)", "native-time sample", primary=True)
    # sensitivity: the file's fully repeated rows counted once
    check_stats(row, S.seasonal_stats(K.activated_flags(DCRIT_DEDUP, class_code)).assign(std=np.nan),
                f"{kind}: as above, fully repeated rows counted once", "native-time sample", primary=False)

114,528 rows, 2023-03-10 22:38:16 to 2024-01-31 07:42:09; both diameters in 12,282; repeated time stamps 24,601; fully repeated rows 471


Samples with both diameters, by month and class:

ClrCldHaz_class,cloud,haze,no class
time,,,
2023-06,941,19,7
2023-07,7697,286,166
2023-08,2475,80,73
2023-09,378,0,22
2023-12,138,0,0


**Row 27: cloud: fraction of cloud-class samples with Dta > Dta_critical (Abbey Williams' ET_Dcrit_nativetime.csv; Kavin's count, applied per season)**  
One sample = native-time sample.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.853,11629,0.85,0.3,NaN,True
Spring,0.943,941,0.94,0.3,NaN,True
Summer,0.852,10550,0.85,0.2,NaN,True
Fall,0.319,138,0.32,-0.4,NaN,True
Winter,–,0,n/a,NaN,NaN,False


...

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.853,11616,0.85,0.3,NaN,True
Spring,0.943,941,0.94,0.3,NaN,True
Summer,0.852,10537,0.85,0.2,NaN,True
Fall,0.319,138,0.32,-0.4,NaN,True
Winter,–,0,n/a,NaN,NaN,False


**Row 30: haze: fraction of haze-class samples with Dta > Dta_critical (Abbey Williams' ET_Dcrit_nativetime.csv; Kavin's count, applied per season)**  
One sample = native-time sample.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.579,385,0.58,-0.1,NaN,True
Spring,1,19,1,0.0,NaN,True
Summer,0.557,366,0.56,-0.5,NaN,True
Fall,–,0,n/a,NaN,NaN,False
Winter,–,0,n/a,NaN,NaN,False


**Row 30: haze: as above, fully repeated rows counted once**  *(sensitivity test)*  
One sample = native-time sample.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.579,385,0.58,-0.1,NaN,True
Spring,1,19,1,0.0,NaN,True
Summer,0.557,366,0.56,-0.5,NaN,True
Fall,–,0,n/a,NaN,NaN,False
Winter,–,0,n/a,NaN,NaN,False


### 9d. Cloud and haze droplet number concentration and effective diameter (rows 25, 26, 28, 29)

*Superseded as the primary for these rows by Section 10e, which reproduces them from Abbey Williams' 15-min file. This section's native-time selections remain as sensitivity tests.*

**Values.** The FM-120 5-min files hold the droplet number concentration (`Number Conc`, 2–50 µm, cm⁻³) and `ED` in µm.
* The README calls `ED` the "Equivalent Diameter". Step 1 recomputes the **effective diameter**, Σ N D³ / Σ N D², from the 30 size bins and checks that it matches the column.
* `ED` is undefined when no droplet was counted, so it is set to NaN where N = 0 (2 rows).
* There is no QC field.

**Which rows are cloud and which are haze.** These follow Abbey Williams' classification, the `ClrCldHaz_class` column of `ET_Dcrit_nativetime.csv` (1 = cloud, 2 = haze). Kavin's live script selects it on lines 11–12 (`indexcld`, `indexhaz`).
* The classified samples are ~1–4 s apart. An FM-120 5-min row counts as **cloud (haze)** when more than half of the classified samples inside it are class 1 (2).
* Each FM-120 row then counts once, as in the other FM-120 rows.
* No row mixes cloud and haze samples, so "more than half" and "any" give the same rows (checked below).

**Coverage.** The classes exist only from 2023-03-10 to 2024-01-31, mostly while the GCVI sampled. So far fewer FM-120 rows are classified than the visibility-based hours of Section 9, and winter is March 2023 plus a few hours of January 2024. The coverage table lists the classified hours per month.

**Kavin's own code for these rows was not shared.** No mapping tried here reproduces the sheet exactly, so these are comparisons, not reproductions. Two sensitivity tests bracket the choice:
1. weighting each FM-120 row by its number of native samples (per native sample);
2. classifying FM-120 rows by Kavin's visibility/LWC definitions from Section 9b instead of Abbey's classes.

In [21]:
# =============================================================================
# Section 9d: cloud and haze droplet number and effective diameter (rows 25, 26, 28, 29)
# =============================================================================
# Inputs
#   fm (Section 9)        FM-120 5-min rows (interval START, UTC): nd_cm3 = number concentration of
#                         2-50 um droplets (# cm-3); ed_um = effective diameter (um); bin01..bin30_cm3 =
#                         dN per size bin (edges SRC.FM120_BIN_EDGES_UM, from readme_FM120.md). No QC field.
#   DCRIT (Section 9c)    Abbey Williams' native-time samples with ClrCldHaz_class (1 = cloud, 2 = haze),
#                         the classes Kavin's live script selects on lines 11-12 (indexcld, indexhaz)
#   avgvis_kavin (9b)     Kavin's visibility per FM-120 row, for sensitivity test (2)
# Primary: FM-120 rows in which most classified native samples are cloud (haze); each row counts once.

# --- 1. What the ED column is: the effective diameter from the size bins ---------------------------
#        D_eff = sum(N_i D_i^3) / sum(N_i D_i^2), D_i = bin mid-point (um). The 5-min ED column is a mean
#        of 1-s values, so it need not equal D_eff of the 5-min mean distribution exactly.
mid_um = 0.5 * (SRC.FM120_BIN_EDGES_UM[:-1] + SRC.FM120_BIN_EDGES_UM[1:])
dn_cm3 = fm[[f"bin{k:02d}_cm3" for k in range(1, 31)]].to_numpy()  # (rows, 30)
with np.errstate(invalid="ignore", divide="ignore"):
    d_eff_bins_um = (dn_cm3 * mid_um ** 3).sum(axis=1) / (dn_cm3 * mid_um ** 2).sum(axis=1)  # (rows,)
droplets = (fm["nd_cm3"] > 10).to_numpy()  # enough droplets for a stable ratio
ed_ratio = fm["ed_um"].to_numpy()[droplets] / d_eff_bins_um[droplets]
print(f"ED column / effective diameter from the bins (rows with N > 10 cm-3, n = {droplets.sum():,}): "
      f"median {np.median(ed_ratio):.3f}, r = {np.corrcoef(fm['ed_um'].to_numpy()[droplets], d_eff_bins_um[droplets])[0, 1]:.3f}")
ed_um = fm["ed_um"].where(fm["nd_cm3"] > 0)  # no droplets -> no diameter (the file writes 0)

# --- 2. Abbey's classes on the FM-120 rows -----------------------------------------------------------
abbey_cloud = K.abbey_class_on_intervals(DCRIT, fm.index, 1)
abbey_haze = K.abbey_class_on_intervals(DCRIT, fm.index, 2)
classified = DCRIT["ClrCldHaz_class"].dropna()
any_cloud = fm.index.isin(classified.index[classified == 1].floor("5min"))  # rows with ANY cloud sample
any_haze = fm.index.isin(classified.index[classified == 2].floor("5min"))
print(f"FM-120 rows classed cloud: {abbey_cloud.sum():,} (with any cloud sample: {any_cloud.sum():,}); "
      f"haze: {abbey_haze.sum():,} (any: {any_haze.sum():,}); rows with both kinds of sample: {(any_cloud & any_haze).sum()}")
display(Markdown("Classified FM-120 hours per month (Abbey Williams' classes):"))
display(pd.DataFrame({"cloud (h)": Q.interval_hours(abbey_cloud).groupby(fm.index.to_period("M")).sum(),
                      "haze (h)": Q.interval_hours(abbey_haze).groupby(fm.index.to_period("M")).sum()}).round(1).T)

# --- 3. Seasonal mean ± std vs the sheet: primary, then two sensitivity tests --------------------
#   rows:  sheet row, FM-120 values, label, class code, Abbey mask, visibility mask (Kavin's 9b definitions)
vis_cloud = (avgvis_kavin < 1000) & (fm["lwc_gm3"] > 0.01)
vis_haze = (avgvis_kavin > 1000) & (avgvis_kavin < 5000)
DROPLET_ROWS = [
    (25, fm["nd_cm3"], "number concentration (cm⁻³)", 1, abbey_cloud, vis_cloud),
    (26, ed_um, "effective diameter (µm)", 1, abbey_cloud, vis_cloud),
    (28, fm["nd_cm3"], "number concentration (cm⁻³)", 2, abbey_haze, vis_haze),
    (29, ed_um, "effective diameter (µm)", 2, abbey_haze, vis_haze),
]
for row, values, label, code, abbey_mask, vis_mask in DROPLET_ROWS:
    kind = K.CLASS_CODES[code]
    display(Markdown(f"#### Row {row}: {kind} droplet {label}"))
    cmps = [
        check(row, values.where(abbey_mask), f"FM-120 {label}, 5-min rows classed {kind} by Abbey Williams' "
              "ClrCldHaz_class (most native samples)", "FM-120 5-min mean", primary=False),  # 10e is primary
        # (1) each native sample counts once: rows weighted by how many class samples they hold
        check(row, K.abbey_class_per_sample(DCRIT, values, code), f"FM-120 {label}, per {kind}-class native "
              "sample", "native-time sample", primary=False, show=False),
        # (2) rows classed by Kavin's visibility/LWC definitions (Section 9b) instead of Abbey's classes
        check(row, values.where(vis_mask), f"FM-120 {label}, 5-min rows classed {kind} by Kavin's visibility/LWC "
              "definitions (Section 9b)", "FM-120 5-min mean", primary=False, show=False),
    ]
    side = pd.concat({c["definition"].iloc[0]: c.set_index("season")["ours"] for c in cmps}, axis=1).T
    side.loc["SHEET (Kavin)"] = cmps[0].set_index("season")["table"]
    display(Markdown("All three selections next to the sheet (mean ± std):"))
    display(side[S.SEASON_ORDER])
del dn_cm3

ED column / effective diameter from the bins (rows with N > 10 cm-3, n = 9,091): median 0.990, r = 0.976
FM-120 rows classed cloud: 3,302 (with any cloud sample: 3,302); haze: 812 (any: 812); rows with both kinds of sample: 0


Classified FM-120 hours per month (Abbey Williams' classes):

time,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02
cloud (h),0.0,3.5,63.0,19.8,10.2,87.5,33.9,12.6,28.2,13.9,2.0,0.5,0.0
haze (h),0.0,3.2,5.3,9.8,3.8,22.4,10.2,4.0,6.8,2.2,0.0,0.0,0.0


#### Row 25: cloud droplet number concentration (cm⁻³)

**Row 25: FM-120 number concentration (cm⁻³), 5-min rows classed cloud by Abbey Williams' ClrCldHaz_class (most native samples)**  *(sensitivity test)*  
One sample = FM-120 5-min mean.

...

#### Row 29: haze droplet effective diameter (µm)

**Row 29: FM-120 effective diameter (µm), 5-min rows classed haze by Abbey Williams' ClrCldHaz_class (most native samples)**  *(sensitivity test)*  
One sample = FM-120 5-min mean.

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,10.5 ± 4.3,812,12 ± 5,-12.6,0.86,False
Spring,10.4 ± 4.5,227,14 ± 6,-25.5,0.75,False
Summer,10.9 ± 3.6,439,12 ± 4,-9.2,0.89,True
Fall,8.5 ± 3.2,107,9.7 ± 2.6,-11.9,1.21,False
Winter,11.6 ± 9.1,39,11 ± 6,5.3,1.51,True


All three selections next to the sheet (mean ± std):

season,EPCAPE,Spring,Summer,Fall,Winter
"FM-120 effective diameter (µm), 5-min rows classed haze by Abbey Williams' ClrCldHaz_class (most native samples)",10.5 ± 4.3,10.4 ± 4.5,10.9 ± 3.6,8.5 ± 3.2,11.6 ± 9.1
"FM-120 effective diameter (µm), per haze-class native sample",8.4 ± 3.7,10.1 ± 4.0,8.3 ± 3.5,8.4 ± 3.3,7.4 ± 3.7
"FM-120 effective diameter (µm), 5-min rows classed haze by Kavin's visibility/LWC definitions (Section 9b)",8.2 ± 3.8,9.3 ± 4.0,8.8 ± 3.5,7.1 ± 2.8,7.0 ± 4.1
SHEET (Kavin),12 ± 5,14 ± 6,12 ± 4,9.7 ± 2.6,11 ± 6


### 9e. Cloud and haze liquid water content (rows 54, 55)

*Superseded as the primary for these rows by Section 10e (Abbey Williams' 15-min file); kept as sensitivity tests.*

The FM-120 LWC (g m⁻³) averaged over the rows of each class. Kavin's code for these rows wasn't shared, and the sheet leaves the campaign cell empty. Four selections of cloud (haze) rows are ranked by how well they reproduce the sheet, and the best match becomes the primary:
1. **Kavin's visibility/LWC classes** (Section 9b): cloud = his row visibility < 1 km and LWC > 0.01 g m⁻³; haze = 1–5 km, no LWC condition. His live script loads LWC in the same section that computes these hours.
2. **Abbey Williams' classes** on the FM-120 rows (Section 9d).
3. **Abbey's classes, per native sample** (rows weighted by their number of class samples).
4. **The BAMS-paper definitions** of Section 9 (screened median visibility).

The cloud definitions in 1 and 4 select on LWC > 0.01 g m⁻³ themselves, so their mean LWC is conditioned on that threshold.

In [22]:
# =============================================================================
# Section 9e: cloud and haze liquid water content (sheet rows 54, 55)
# =============================================================================
# Inputs
#   fm (Section 9)                     FM-120 5-min rows: lwc_gm3 (liquid water content, g m-3); no QC field
#   vis_cloud, vis_haze (9d)           rows classed by Kavin's visibility/LWC definitions (Section 9b)
#   abbey_cloud, abbey_haze (9d)       rows classed by Abbey Williams' ClrCldHaz_class (majority)
#   DCRIT (9c)                         Abbey's native-time samples, for the per-sample weighting
#   cls (Section 9)                    rows classed by the BAMS-paper definitions (FM120_* settings)
# compare_candidates ranks the selections by the median |% difference| from the sheet over the
# columns the sheet fills (its campaign cell is empty) and marks the best one primary.
for row, kind, code, vis_mask, abbey_mask in ((54, "cloud", 1, vis_cloud, abbey_cloud),
                                               (55, "haze", 2, vis_haze, abbey_haze)):
    display(Markdown(f"#### Row {row}: {kind} liquid water content (g m⁻³)"))
    lwc_gm3 = fm["lwc_gm3"]
    compare_candidates(row, {
        "rows classed by Kavin's visibility/LWC definitions (Section 9b)": (lwc_gm3.where(vis_mask), "FM-120 5-min mean"),
        "rows classed by Abbey Williams' ClrCldHaz_class (most native samples)": (lwc_gm3.where(abbey_mask), "FM-120 5-min mean"),
        "per Abbey-class native sample": (K.abbey_class_per_sample(DCRIT, lwc_gm3, code), "native-time sample"),
        f"rows classed by the BAMS-paper definitions ({FM120_VIS_STAT} visibility; Section 9)": (lwc_gm3.where(cls[kind]), "FM-120 5-min mean"),
    }, label=f"FM-120 {kind} LWC (g m⁻³), ", mark_primary=False)  # Section 10e is primary
del fm, avgvis_kavin

#### Row 54: cloud liquid water content (g m⁻³)

season,EPCAPE,Spring,Summer,Fall,Winter
"FM-120 cloud LWC (g m⁻³), rows classed by Kavin's visibility/LWC definitions (Section 9b)",0.14 ± 0.18,0.17 ± 0.17,0.12 ± 0.16,0.19 ± 0.27,0.07 ± 0.10
"FM-120 cloud LWC (g m⁻³), rows classed by Abbey Williams' ClrCldHaz_class (most native samples)",0.16 ± 0.19,0.17 ± 0.16,0.15 ± 0.19,0.17 ± 0.24,0.31 ± 0.21
"FM-120 cloud LWC (g m⁻³), per Abbey-class native sample",0.22 ± 0.19,0.22 ± 0.16,0.25 ± 0.25,0.18 ± 0.19,0.28 ± 0.20
"FM-120 cloud LWC (g m⁻³), rows classed by the BAMS-paper definitions (median visibility; Section 9)",0.14 ± 0.18,0.17 ± 0.17,0.12 ± 0.16,0.19 ± 0.27,0.07 ± 0.10
SHEET (Kavin),None,0.16 ± 0.15,0.12 ± 0.17,0.17 ± 0.23,0.12 ± 0.16


,definition,median_abs_diff_pct,max_abs_diff_pct,median_abs_log2_std_ratio,seasons_compared
0,"FM-120 cloud LWC (g m⁻³), rows classed by the BAMS-paper definitions (median visibilit...",7.23,45.46,0.21,4
1,"FM-120 cloud LWC (g m⁻³), rows classed by Kavin's visibility/LWC definitions (Section 9b)",8.01,45.17,0.22,4
2,"FM-120 cloud LWC (g m⁻³), rows classed by Abbey Williams' ClrCldHaz_class (most native...",16.21,157.11,0.12,4
3,"FM-120 cloud LWC (g m⁻³), per Abbey-class native sample",74.57,131.82,0.29,4


#### Row 55: haze liquid water content (g m⁻³)

season,EPCAPE,Spring,Summer,Fall,Winter
"FM-120 haze LWC (g m⁻³), rows classed by Kavin's visibility/LWC definitions (Section 9b)",0.011 ± 0.030,0.0049 ± 0.0061,0.008 ± 0.015,0.006 ± 0.026,0.021 ± 0.050
"FM-120 haze LWC (g m⁻³), rows classed by Abbey Williams' ClrCldHaz_class (most native samples)",0.013 ± 0.018,0.0093 ± 0.0095,0.013 ± 0.014,0.0086 ± 0.0061,0.052 ± 0.050
"FM-120 haze LWC (g m⁻³), per Abbey-class native sample",0.016 ± 0.017,0.012 ± 0.010,0.009 ± 0.013,0.0103 ± 0.0073,0.037 ± 0.018
"FM-120 haze LWC (g m⁻³), rows classed by the BAMS-paper definitions (median visibility; Section 9)",0.0027 ± 0.0029,0.0032 ± 0.0029,0.0034 ± 0.0029,0.0020 ± 0.0026,0.0016 ± 0.0026
SHEET (Kavin),None,0.0077 ± 0.0081,0.0089 ± 0.0091,0.0062 ± 0.0145,0.021 ± 0.033


,definition,median_abs_diff_pct,max_abs_diff_pct,median_abs_log2_std_ratio,seasons_compared
0,"FM-120 haze LWC (g m⁻³), rows classed by Kavin's visibility/LWC definitions (Section 9b)",9.02,36.21,0.68,4
1,"FM-120 haze LWC (g m⁻³), rows classed by Abbey Williams' ClrCldHaz_class (most native ...",40.45,148.85,0.63,4
2,"FM-120 haze LWC (g m⁻³), per Abbey-class native sample",59.46,74.42,0.70,4
3,"FM-120 haze LWC (g m⁻³), rows classed by the BAMS-paper definitions (median visibility...",64.42,92.16,2.06,4


## 10. GCVI–AMS cloud and haze residuals (rows 33–42)

At Mt. Soledad the HR-ToF-AMS sampled either ambient air or the residuals of droplets collected by the ground-based counterflow virtual impactor (GCVI).
The distributed AMS file (`AMS_CE_corrected_msptof.nc`, CE-corrected NR-PM1) does not label which inlet was sampled. GCVI sampling periods are identified from the 1-s GCVI record (`EPCAPE_CVI_EF.csv`): EF finite and > 1, segments ≥ 15 min.
Behind a CVI, concentrations are enhanced by the enhancement factor EF ≈ 5, so ambient-equivalent residual concentrations are *measured / EF* (Shingler et al. 2012, *AMT* 5, 1259). EF is averaged over each 2-min AMS V-mode period.

**Structure.** The sheet splits GCVI sampling into *cloud* and *haze*.
* **10a–10b** check the split-free quantities as context:
  * the total sampling hours, cloud + haze;
  * the residual concentrations over all GCVI periods, against the **hour-weighted combination** of the sheet's cloud and haze rows.
* **10c** splits the GCVI-AMS samples into cloud and haze with Abbey Williams' classes, the same FM-120 rows as Section 9d. It compares each sheet row separately and is the primary definition for rows 33–42.
* **10d** checks whether `AMS_ambient.nc` is the source of the residual rows.
For Fall, the haze hours used are 50 h instead of H34's "5.0×10²". That cell is a typo for 5.0×10¹: Kavin's code gives 49.5 h (Section 10e).
Neither the AMS file nor the GCVI file has a QC field. The AMS README says flagged data were already removed.

In [23]:
# =============================================================================
# Section 10a: GCVI-AMS sampling hours (sheet rows 33-34)
# =============================================================================
# Inputs (UCSD Library, "Aerosol Quantification Products", data/processed/Aerosol Quantification Products/)
#   AMS_CE_corrected_msptof.nc   HR-ToF-AMS at Mt. Soledad: NR-PM1 organics, sulfate, nitrate,
#                                ammonium, chloride (ug m-3), CE-corrected, ~5.5-min samples,
#                                time = START of each 2-min V-mode sample. No inlet flag and no
#                                QC field (the README says flagged data were already removed).
#   bb2743661p_36_1_CVI_EF.csv   1-s GCVI record: enhancement factor EF (~5 when sampling
#                                droplets), cut size, flows. No QC field.
#   GCVI_MIN_SEGMENT_MIN         "minimum sampling 15 min" from the sheet
# Steps: find continuous GCVI sampling segments (EF finite and > 1, gaps <= 60 s), keep those
# >= 15 min, and pick the AMS samples whose whole 2-min V-mode period lies inside one.

# --- 1. Load both records and pair them ------------------------------------------------
ams = SRC.load_ams_soledad()
gcvi = SRC.load_gcvi()
segments = Q.gcvi_segments(gcvi)  # start, end, duration_min, mean_EF, median_cut_um
resid, used = Q.gcvi_ams_residuals(ams, gcvi, segments, min_duration_min=GCVI_MIN_SEGMENT_MIN)
#   resid: one row per AMS sample behind the GCVI, measured and / EF concentrations
#   used : the >= 15-min segments that contain at least one AMS sample
long_segments = segments[segments["duration_min"] >= GCVI_MIN_SEGMENT_MIN]
ams_step_h = pd.Series(ams.index).diff().median() / pd.Timedelta(hours=1)  # AMS cadence, hours
print(f"GCVI segments: {len(segments)}; >= {GCVI_MIN_SEGMENT_MIN:g} min: {len(long_segments)}; "
      f"with >= 1 AMS sample: {len(used)}.  AMS samples behind the GCVI: {len(resid)} "
      f"(AMS cadence {60 * ams_step_h:.1f} min).  Median EF {resid['EF'].median():.2f}")

# --- 2. The sheet's hours (cloud row 33, haze row 34) ---------------------------------------
# Kavin's hours. The fall haze cell "5.0*10^2" is a typo for 5.0*10^1: Kavin's AMS code gives 49.5 h
# (Section 10e), so 50 h is used.
h_cloud = S.table_values(TABLE, 33)["mean"]
h_haze = S.table_values(TABLE, 34)["mean"].copy()
h_haze["Fall"] = 50.0

# --- 3. Hours per season, three ways, next to the sheet's ----------------------------------
#   all GCVI segments >= 15 min; only those in which the AMS sampled (the comparable one);
#   and the number of AMS samples behind the GCVI times the AMS cadence
HOURS = pd.DataFrame({
    f"GCVI sampling, segments ≥ {GCVI_MIN_SEGMENT_MIN:g} min": S.seasonal_sums(long_segments.set_index("start")["duration_min"] / 60),
    "… segments in which the AMS sampled": S.seasonal_sums(used.set_index("start")["duration_min"] / 60),
    "AMS samples behind the GCVI × cadence": S.seasonal_sums(pd.Series(ams_step_h, index=resid.index)),
    "sheet row 33 (cloud)": h_cloud,
    "sheet row 34 (haze; Fall = 50, typo corrected)": h_haze,
    "sheet 33 + 34": h_cloud + h_haze,
}).round(0)
display(Markdown("Sampling hours:"))
display(HOURS.T)

# --- 4. Add the hours comparison to the summary (cloud + haze together) ---------------------
# The total (cloud + haze) is compared with rows 33 + 34, as context; Section 10c splits it.
hours_ref = pd.DataFrame({"mean": h_cloud + h_haze, "std": np.nan,
                          "raw": [f"{v:.0f} (33+34)" for v in h_cloud + h_haze]}, index=S.SEASON_ORDER)
SUMMARY.append(S.compare_to(
    pd.DataFrame({"mean": HOURS["… segments in which the AMS sampled"], "std": np.nan, "median": np.nan,
                  "n": S.seasonal_sums(pd.Series(1.0, index=used["start"]))}),
    hours_ref, "33+34", "GCVI-AMS sampling hours (cloud + haze)",
    f"hours of GCVI segments ≥ {GCVI_MIN_SEGMENT_MIN:g} min with AMS data", close_pct=CLOSE_PCT
).assign(sample="sum of hours", primary=False))

GCVI segments: 1835; >= 15 min: 353; with >= 1 AMS sample: 236.  AMS samples behind the GCVI: 5502 (AMS cadence 5.6 min).  Median EF 4.96


Sampling hours:

,EPCAPE,Spring,Summer,Fall,Winter
"GCVI sampling, segments ≥ 15 min",766.0,212.0,365.0,131.0,58.0
… segments in which the AMS sampled,605.0,155.0,318.0,109.0,23.0
AMS samples behind the GCVI × cadence,512.0,131.0,253.0,105.0,23.0
sheet row 33 (cloud),352.0,106.0,185.0,47.0,14.0
"sheet row 34 (haze; Fall = 50, typo corrected)",236.0,52.0,101.0,50.0,35.0
sheet 33 + 34,588.0,158.0,286.0,97.0,49.0


In [24]:
# =============================================================================
# Section 10b: cloud + haze residual concentrations (sheet rows 35-42)
# =============================================================================
# Inputs
#   resid (Section 10a)  AMS samples behind the GCVI: <species>_ugm3 as measured, and
#                        <species>_resid_ugm3 = measured / EF (ambient-equivalent; Shingler
#                        et al. 2012), EF averaged over each 2-min V-mode period
#   h_cloud, h_haze      the sheet's sampling hours (rows 33, 34), used as weights
# The sheet splits cloud (rows 35-38) from haze (39-42); without FM-120 data we can only
# check both together, so each species is compared with the HOUR-WEIGHTED combination of
# its cloud and haze rows.
SPECIES = {"sulfate": (35, 39), "nitrate": (36, 40), "ammonium": (37, 41), "organics": (38, 42)}
w_cloud = h_cloud / (h_cloud + h_haze)  # per season: fraction of the sampling hours that were cloud


def hour_weighted(row_cloud, row_haze):
    """Mixture of the sheet's cloud and haze rows, weighted by their sampling hours.

    mean = w_c m_c + w_h m_h ;  var = w_c (s_c² + m_c²) + w_h (s_h² + m_h²) - mean²"""
    c, h = S.table_values(TABLE, row_cloud), S.table_values(TABLE, row_haze)
    mean = w_cloud * c["mean"] + (1 - w_cloud) * h["mean"]
    second = w_cloud * (c["std"] ** 2 + c["mean"] ** 2) + (1 - w_cloud) * (h["std"] ** 2 + h["mean"] ** 2)
    std = np.sqrt(second - mean ** 2)
    return pd.DataFrame({"mean": mean, "std": std,
                         "raw": [S.format_pm(m, s) for m, s in zip(mean, std)]}, index=S.SEASON_ORDER)


context = {}  # extra rows for the context table at the end
for species, (rc, rh) in SPECIES.items():
    # --- 1. Our / EF concentration vs the hour-weighted sheet value ------------------------
    check(f"{rc}+{rh}", resid[f"{species}_resid_ugm3"],
          f"AMS {species} behind the GCVI / EF, all GCVI periods (cloud + haze), µg m⁻³",
          "AMS sample (~5.5 min)", reference=hour_weighted(rc, rh),
          quantity=f"{species} residual, cloud + haze (hour-weighted rows {rc} & {rh})",
          primary=False)  # context: Section 10c compares each cloud and haze row separately
    # --- 2. Context: the same samples before dividing by EF, and the sheet's separate rows --
    st_raw = S.seasonal_stats(resid[f"{species}_ugm3"])
    context[f"{species}: as measured behind GCVI (not / EF)"] = [S.format_pm(m, s) for m, s in zip(st_raw["mean"], st_raw["std"])]
    context[f"{species}: sheet cloud (row {rc})"] = S.table_values(TABLE, rc)["raw"].values
    context[f"{species}: sheet haze (row {rh})"] = S.table_values(TABLE, rh)["raw"].values
display(Markdown("For context: concentrations before dividing by EF, and the sheet's separate cloud and haze rows (µg m⁻³):"))
display(pd.DataFrame(context, index=S.SEASON_ORDER).T)

**Row 35+39: AMS sulfate behind the GCVI / EF, all GCVI periods (cloud + haze), µg m⁻³**  *(sensitivity test)*  
One sample = AMS sample (~5.5 min).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.23 ± 0.26,5213,0.26 ± 0.27,-12.0,0.97,False
Spring,0.11 ± 0.10,1259,0.16 ± 0.20,-30.1,0.52,False
Summer,0.30 ± 0.29,2607,0.33 ± 0.27,-7.5,1.07,True
Fall,0.21 ± 0.27,1109,0.23 ± 0.28,-6.2,0.95,True
Winter,0.051 ± 0.057,238,0.12 ± 0.19,-56.7,0.30,False


**Row 36+40: AMS nitrate behind the GCVI / EF, all GCVI periods (cloud + haze), µg m⁻³**  *(sensitivity test)*  
One sample = AMS sample (~5.5 min).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.30 ± 0.39,5499,0.35 ± 0.38,-12.2,1.02,False
Spring,0.16 ± 0.17,1404,0.26 ± 0.32,-39.2,0.54,False
Summer,0.33 ± 0.42,2720,0.35 ± 0.39,-8.0,1.08,True
Fall,0.48 ± 0.45,1131,0.50 ± 0.41,-4.5,1.10,True
Winter,0.088 ± 0.088,244,0.15 ± 0.23,-40.4,0.38,False


**Row 37+41: AMS ammonium behind the GCVI / EF, all GCVI periods (cloud + haze), µg m⁻³**  *(sensitivity test)*  
One sample = AMS sample (~5.5 min).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.16 ± 0.16,5136,0.18 ± 0.17,-10.4,0.96,False
Spring,0.096 ± 0.092,1256,0.14 ± 0.18,-33.1,0.51,False
Summer,0.18 ± 0.17,2554,0.19 ± 0.16,-2.8,1.09,True
Fall,0.21 ± 0.19,1118,0.22 ± 0.19,-4.9,1.00,True
Winter,0.049 ± 0.047,208,0.09 ± 0.13,-42.0,0.36,False


**Row 38+42: AMS organics behind the GCVI / EF, all GCVI periods (cloud + haze), µg m⁻³**  *(sensitivity test)*  
One sample = AMS sample (~5.5 min).

,ours,n,table,diff_pct,std_ratio,close
season,,,,,,
EPCAPE,0.42 ± 0.54,5013,0.45 ± 0.52,-5.5,1.03,True
Spring,0.23 ± 0.20,1109,0.31 ± 0.43,-26.6,0.47,False
Summer,0.48 ± 0.63,2624,0.51 ± 0.59,-6.8,1.06,True
Fall,0.54 ± 0.51,1080,0.51 ± 0.43,4.7,1.18,True
Winter,0.107 ± 0.092,200,0.19 ± 0.34,-44.6,0.27,False


For context: concentrations before dividing by EF, and the sheet's separate cloud and haze rows (µg m⁻³):

,EPCAPE,Spring,Summer,Fall,Winter
sulfate: as measured behind GCVI (not / EF),1.1 ± 1.3,0.54 ± 0.51,1.5 ± 1.5,1.1 ± 1.3,0.25 ± 0.28
sulfate: sheet cloud (row 35),0.36 ± 0.29,0.21 ± 0.22,0.44 ± 0.27,0.35 ± 0.35,0.11 ± 0.04
sulfate: sheet haze (row 39),0.10 ± 0.12,0.045 ± 0.047,0.12 ± 0.11,0.11. ± 0.10,0.12 ± 0.22
nitrate: as measured behind GCVI (not / EF),1.5 ± 1.9,0.79 ± 0.86,1.6 ± 2.1,2.4 ± 2.2,0.44 ± 0.44
nitrate: sheet cloud (row 36),0.43 ± 0.42,0.34 ± 0.36,0.45 ± 0.44,0.57 ± 0.40,0.12 ± 0.07
nitrate: sheet haze (row 40),0.22 ± 0.27,0.10 ± 0.09,0.18 ± 0.17,0.43 ± 0.40,0.16 ± 0.27
ammonium: as measured behind GCVI (not / EF),0.80 ± 0.82,0.47 ± 0.45,0.91 ± 0.85,1.03 ± 0.95,0.24 ± 0.23
ammonium: sheet cloud (row 37),0.24 ± 0.18,0.19 ± 0.20,0.25 ± 0.16,0.28 ± 0.20,0.071 ± 0.036
ammonium: sheet haze (row 41),0.093 ± 0.110,0.047 ± 0.047,0.079 ± 0.065,0.16 ± 0.16,0.091 ± 0.154
organics: as measured behind GCVI (not / EF),2.1 ± 2.7,1.1 ± 1.0,2.4 ± 3.1,2.7 ± 2.5,0.53 ± 0.45


### 10c. Cloud and haze split with Abbey Williams' classes (rows 33–42)

*Superseded as the primary for these rows by Section 10e, which uses Abbey's 15-min classes as Kavin did. This section's native-time mappings remain as sensitivity tests.*

Each AMS sample taken behind the GCVI (Section 10a) is classed **cloud** or **haze** from Abbey Williams' classification, applied to the FM-120 rows as in Section 9d. The sample takes the class of the FM-120 5-min row that holds the middle of its 2-min V-mode period.
* **Hours (rows 33, 34):** the class's FM-120 rows that lie inside a GCVI-AMS segment (≥ 15 min with AMS data), × 5 min.
* **Residuals (rows 35–42):** mean ± std of the CE-corrected AMS concentration ÷ EF over the class's samples, as in Section 10b.

**Two sensitivity tests**, ranked by how well they reproduce the sheet:
1. **Whole GCVI segments** take the class held by the majority of Abbey's samples in them, and the segment's full duration counts as hours.
2. **Kavin's visibility/LWC classes** (Section 9b) on the FM-120 rows, instead of Abbey's classes.

**Where Abbey's samples sit in time.** None of them falls inside the 2-min V-mode windows. They fall between them, in the rest of each ~5.5-min AMS cycle. That is consistent with "ET" in the file name meaning the AMS **event-trigger** (single-particle) mode, which is my inference and isn't documented in the files. Classifying each AMS sample from Abbey's samples between it and the next AMS sample gives the same result as the FM-120 rows (checked while building this section).

In [25]:
# =============================================================================
# Section 10c: GCVI-AMS cloud / haze split with Abbey Williams' classes (sheet rows 33-42)
# =============================================================================
# Inputs
#   resid, used (Section 10a)     AMS samples behind the GCVI (measured, and / EF = <species>_resid_ugm3),
#                                 and the GCVI sampling segments >= 15 min holding at least one AMS sample
#   abbey_cloud, abbey_haze (9d)  FM-120 5-min rows classed cloud / haze from Abbey Williams' ClrCldHaz_class
#                                 (the classes on lines 11-12 of Kavin's live script)
#   vis_cloud, vis_haze (9d)      FM-120 rows classed by Kavin's visibility/LWC definitions (Section 9b)
#   DCRIT (9c)                    Abbey's native-time samples, for the whole-segment sensitivity test
#   SPECIES (10b)                 species -> (cloud row, haze row)
# Primary: each AMS sample takes the class of the FM-120 row holding the middle of its 2-min V-mode
# period (K.class_at); hours = the class's FM-120 rows inside GCVI-AMS segments x 5 min.

# --- 0. Where Abbey's samples sit relative to the AMS V-mode windows ---------------------------
abbey_times = DCRIT.index[DCRIT["ClrCldHaz_class"].notna()]  # sorted, as in the file
in_vmode = abbey_times.searchsorted(resid.index + pd.Timedelta(minutes=2)) - abbey_times.searchsorted(resid.index)
print(f"Abbey's classified samples inside the 2-min V-mode windows of the {len(resid):,} GCVI-AMS samples: "
      f"{int(in_vmode.sum())} (they lie between the windows)")

# --- 1. Three ways of classing the GCVI-AMS samples and time --------------------------------------
used_segments = used.reset_index(drop=True)


def class_rows_hours(row_mask):
    """Hours per season of the True FM-120 rows that lie inside a GCVI-AMS segment (each row = 5 min)."""
    t = row_mask.index[row_mask.to_numpy()]
    return S.seasonal_sums(pd.Series(5 / 60, index=t[K.segment_of(t, used_segments) >= 0]))


SPLITS = {}  # name -> {"cloud": (boolean mask over resid rows, hours per season), "haze": (...)}
PRIMARY_SPLIT = "Abbey's classes, FM-120 rows"
for name, rows_by_kind in {PRIMARY_SPLIT: {"cloud": abbey_cloud, "haze": abbey_haze},
                           "Kavin's visibility/LWC classes, FM-120 rows": {"cloud": vis_cloud, "haze": vis_haze}}.items():
    SPLITS[name] = {kind: (K.class_at(resid.index, rows), class_rows_hours(rows)) for kind, rows in rows_by_kind.items()}
segment_of_sample = K.segment_of(resid.index, used_segments)  # which GCVI-AMS segment each AMS sample is in
SPLITS["Abbey's classes, whole GCVI segments (majority)"] = {}
for kind, code in (("cloud", 1), ("haze", 2)):
    is_kind = K.segment_majority_class(DCRIT, used_segments, code)  # one value per segment
    SPLITS["Abbey's classes, whole GCVI segments (majority)"][kind] = (
        np.isin(segment_of_sample, np.flatnonzero(is_kind)),
        S.seasonal_sums(used_segments.loc[is_kind].set_index("start")["duration_min"] / 60))


def cell(cmp):
    """'ours (diff %)' per season from a comparison table."""
    c = cmp.set_index("season")
    return [f"{o} ({d:+.0f} %)" if np.isfinite(d) else str(o) for o, d in zip(c["ours"], c["diff_pct"])]


# --- 2. Hours (rows 33 and 34); fall haze = 50 h (the sheet's typo corrected; Section 10e) ------
hour_refs = {33: h_cloud, 34: h_haze}
HOURS_SPLIT = {}
for row, kind in ((33, "cloud"), (34, "haze")):
    ref = pd.DataFrame({"mean": hour_refs[row], "std": np.nan,
                        "raw": [f"{v:.0f}" for v in hour_refs[row]]}, index=S.SEASON_ORDER)
    for name, split in SPLITS.items():
        mask, hours = split[kind]
        ours = pd.DataFrame({"mean": hours, "std": np.nan, "median": np.nan,
                             "n": S.seasonal_sums(pd.Series(1.0, index=resid.index[mask])).astype(int)})
        cmp = S.compare_to(ours, ref, row, TABLE.loc[row, "quantity"], f"GCVI-AMS {kind} hours: {name}",
                           close_pct=CLOSE_PCT).assign(sample="sum of hours", primary=False)  # Section 10e is primary
        SUMMARY.append(cmp)
        HOURS_SPLIT[(row, name)] = cell(cmp)
    HOURS_SPLIT[(row, "SHEET (Kavin)")] = ref["raw"].tolist()
display(Markdown("**GCVI-AMS cloud (row 33) and haze (row 34) sampling hours: this check (difference from the sheet)**"))
display(pd.DataFrame(HOURS_SPLIT, index=S.SEASON_ORDER).T)

# --- 3. Residual concentrations (rows 35-42), µg m-3 ambient-equivalent (measured / EF) ----------
RESID_SPLIT, RANK = {}, []
for kind, col in (("cloud", 0), ("haze", 1)):
    for species, rows in SPECIES.items():
        row = rows[col]
        for name, split in SPLITS.items():
            cmp = check(row, resid[f"{species}_resid_ugm3"][split[kind][0]],
                        f"AMS {species} behind the GCVI / EF, {kind} samples: {name}", "AMS sample (~5.5 min)",
                        primary=False, show=False)  # Section 10e is primary
            RESID_SPLIT[(kind, species, name)] = cell(cmp)
            RANK.append({"class": kind, "classification": name, "species": species,
                         "median |diff| over seasons (%)": cmp["diff_pct"].abs().median()})
        RESID_SPLIT[(kind, species, "SHEET (Kavin)")] = S.table_values(TABLE, row)["raw"].tolist()
display(Markdown("**Cloud (rows 35–38) and haze (rows 39–42) residuals, µg m⁻³: this check (difference from the sheet)**"))
display(pd.DataFrame(RESID_SPLIT, index=S.SEASON_ORDER).T)
display(Markdown("How well each classification reproduces the sheet (median over the four species of the median |difference| over the five columns):"))
display(pd.DataFrame(RANK).groupby(["class", "classification"])["median |diff| over seasons (%)"].median().round(1).unstack(0))

Abbey's classified samples inside the 2-min V-mode windows of the 5,502 GCVI-AMS samples: 0 (they lie between the windows)


**GCVI-AMS cloud (row 33) and haze (row 34) sampling hours: this check (difference from the sheet)**

EPCAPE        Spring        Summer          Fall        Winter
33 Abbey's classes, FM-120 rows                      246 (-30 %)  76.2 (-28 %)   124 (-33 %)  41.7 (-11 %)   3.5 (-75 %)
   Kavin's visibility/LWC classes, FM-120 rows        359 (+2 %)    105 (-1 %)    195 (+5 %)  53.8 (+14 %)  4.75 (-66 %)
   Abbey's classes, whole GCVI segments (majority)   400 (+14 %)   119 (+13 %)   218 (+18 %)  62.7 (+33 %)    0 (-100 %)
   SHEET (Kavin)                                             352           106           185            47            14
34 Abbey's classes, FM-120 rows                     52.2 (-78 %)  11.7 (-77 %)  31.6 (-69 %)  8.92 (-82 %)    0 (-100 %)
   Kavin's visibility/LWC classes, FM-120 rows       152 (-36 %)  30.4 (-42 %)   96.8 (-4 %)  22.3 (-55 %)   2.5 (-93 %)
   Abbey's classes, whole GCVI segments (majority)  98.6 (-58 %)  19.7 (-62 %)    28 (-72 %)  30.7 (-39 %)  20.1 (-43 %)
   SHEET (Kavin)                                             236            52           101            50            35

**Cloud (rows 35–38) and haze (rows 39–42) residuals, µg m⁻³: this check (difference from the sheet)**

EPCAPE                 Spring                 Summer                   Fall                 Winter
cloud sulfate  Abbey's classes, FM-120 rows                        0.36 ± 0.31 (-1 %)    0.16 ± 0.10 (-25 %)    0.49 ± 0.31 (+12 %)     0.35 ± 0.35 (-0 %)   0.107 ± 0.037 (-2 %)
               Kavin's visibility/LWC classes, FM-120 rows        0.32 ± 0.29 (-11 %)    0.14 ± 0.11 (-33 %)     0.43 ± 0.30 (-3 %)    0.31 ± 0.33 (-12 %)  0.092 ± 0.047 (-17 %)
               Abbey's classes, whole GCVI segments (majority)    0.29 ± 0.29 (-20 %)    0.13 ± 0.11 (-39 %)    0.38 ± 0.31 (-14 %)    0.27 ± 0.32 (-22 %)                      –
               SHEET (Kavin)                                              0.36 ± 0.29            0.21 ± 0.22            0.44 ± 0.27            0.35 ± 0.35            0.11 ± 0.04
      nitrate  Abbey's classes, FM-120 rows                        0.44 ± 0.45 (+2 %)    0.25 ± 0.18 (-28 %)    0.53 ± 0.53 (+17 %)     0.56 ± 0.43 (-2 %)   0.117 ± 0.075 (-3 %)
               Kavin's visibility/LWC classes, FM-120 rows        0.38 ± 0.43 (-11 %)    0.20 ± 0.19 (-42 %)     0.44 ± 0.49 (-2 %)     0.56 ± 0.46 (-1 %)    0.13 ± 0.11 (+12 %)
               Abbey's classes, whole GCVI segments (majority)    0.37 ± 0.43 (-14 %)    0.18 ± 0.18 (-48 %)     0.44 ± 0.47 (-3 %)    0.50 ± 0.46 (-12 %)                      –
               SHEET (Kavin)                                              0.43 ± 0.42            0.34 ± 0.36            0.45 ± 0.44            0.57 ± 0.40            0.12 ± 0.07
      ammonium Abbey's classes, FM-120 rows                        0.24 ± 0.18 (-2 %)  0.137 ± 0.093 (-28 %)    0.29 ± 0.18 (+15 %)     0.28 ± 0.21 (-1 %)   0.070 ± 0.036 (-2 %)
               Kavin's visibility/LWC classes, FM-120 rows        0.21 ± 0.17 (-11 %)  0.119 ± 0.096 (-37 %)     0.25 ± 0.18 (+1 %)     0.26 ± 0.20 (-6 %)   0.076 ± 0.042 (+7 %)
               Abbey's classes, whole GCVI segments (majority)    0.20 ± 0.18 (-18 %)  0.108 ± 0.096 (-43 %)     0.23 ± 0.18 (-7 %)    0.23 ± 0.20 (-17 %)                      –
               SHEET (Kavin)                                             0.24 ± 0.18             0.19 ± 0.20            0.25 ± 0.16            0.28 ± 0.20          0.071 ± 0.036
      organics Abbey's classes, FM-120 rows                        0.62 ± 0.65 (+2 %)    0.30 ± 0.21 (-28 %)    0.80 ± 0.80 (+17 %)     0.72 ± 0.52 (-1 %)   0.159 ± 0.085 (+6 %)
               Kavin's visibility/LWC classes, FM-120 rows         0.57 ± 0.62 (-7 %)    0.29 ± 0.21 (-31 %)     0.67 ± 0.72 (-3 %)     0.71 ± 0.56 (-2 %)    0.19 ± 0.11 (+27 %)
               Abbey's classes, whole GCVI segments (majority)    0.53 ± 0.61 (-12 %)    0.27 ± 0.21 (-36 %)    0.62 ± 0.71 (-10 %)    0.63 ± 0.56 (-13 %)                      –
               SHEET (Kavin)                                              0.61 ± 0.59            0.42 ± 0.49            0.69 ± 0.66            0.73 ± 0.46            0.15 ± 0.08
haze  sulfate  Abbey's classes, FM-120 rows                       0.11 ± 0.16 (+14 %)  0.035 ± 0.029 (-22 %)    0.16 ± 0.18 (+31 %)  0.048 ± 0.059 (-56 %)                      –
               Kavin's visibility/LWC classes, FM-120 rows      0.065 ± 0.097 (-35 %)  0.027 ± 0.036 (-41 %)    0.09 ± 0.11 (-28 %)  0.031 ± 0.044 (-71 %)  0.012 ± 0.034 (-90 %)
               Abbey's classes, whole GCVI segments (majority)  0.087 ± 0.094 (-13 %)  0.050 ± 0.059 (+11 %)    0.11 ± 0.11 (-11 %)     0.11 ± 0.10 (+3 %)  0.056 ± 0.058 (-54 %)
               SHEET (Kavin)                                             0.10 ± 0.12           0.045 ± 0.047            0.12 ± 0.11           0.11. ± 0.10            0.12 ± 0.22
      nitrate  Abbey's classes, FM-120 rows                       0.20 ± 0.21 (-10 %)  0.063 ± 0.039 (-37 %)    0.24 ± 0.24 (+34 %)    0.18 ± 0.15 (-59 %)                      –
               Kavin's visibility/LWC classes, FM-120 rows        0.12 ± 0.14 (-44 %)  0.051 ± 0.040 (-49 %)    0.14 ± 0.16 (-21 %)    0.14 ± 0.13 (-68 %)  0.073 ±

How well each classification reproduces the sheet (median over the four species of the median |difference| over the five columns):

class,cloud,haze
classification,,
"Abbey's classes, FM-120 rows",2.6,31.3
"Abbey's classes, whole GCVI segments (majority)",15.3,7.2
"Kavin's visibility/LWC classes, FM-120 rows",9.0,44.8


### 10d. Is `AMS_ambient.nc` the source of the cloud and haze residual rows?

`data/processed/AMS/AMS_ambient.nc` holds the same AMS species (plus size-resolved mass), and its attributes say only "EPCAPE AMS measurements of size-resolved submicron aerosol composition". This cell compares it with the CE-corrected file used above:
* which samples it holds;
* how its values relate to the CE-corrected file;
* whether its samples in cloud and haze periods reproduce the sheet's rows 35–42.

In [26]:
# =============================================================================
# Section 10d: AMS_ambient.nc vs the CE-corrected AMS file and the sheet's rows 35-42
# =============================================================================
# Inputs
#   AMS/AMS_ambient.nc (SRC.load_ams_ambient)  AMS NR-PM1 species (ug m-3), UTC
#   ams, resid (Section 10a)                   the CE-corrected AMS file and its GCVI samples
#   abbey_cloud, abbey_haze, vis_cloud, vis_haze (9d)  FM-120 rows by class

# --- 1. Which samples does the ambient file hold? ---------------------------------------------
amb = SRC.load_ams_ambient()
ce_only = ams.index.difference(amb.index)
print(f"AMS_ambient.nc: {len(amb):,} samples, {amb.index[0]} to {amb.index[-1]}; "
      f"{amb.index.isin(ams.index).mean():.1%} of its time stamps are in the CE-corrected file.")
print(f"CE-corrected samples missing from it: {len(ce_only):,}. Of the {len(resid):,} GCVI-AMS samples of Section 10a, "
      f"{resid.index.isin(ce_only).sum():,} are missing from it -> it holds the ambient-inlet samples only.")

# --- 2. How do its values relate to the CE-corrected file? -------------------------------------
#        ratio ambient / CE-corrected for every sample and species, summarised per month
ratio = pd.DataFrame({s: amb[f"{s}_ugm3"] / ams.loc[amb.index, f"{s}_ugm3"] for s in SRC.AMS_SPECIES})
by_month = ratio.groupby(ratio.index.to_period("M"))
display(Markdown("Ratio AMS_ambient / CE-corrected file per month (all species): one constant per month, the "
                 "monthly collection efficiency, so AMS_ambient.nc is **not** CE-corrected"))
display(pd.DataFrame({"min": by_month.min().min(axis=1), "median": by_month.median().median(axis=1),
                      "max": by_month.max().max(axis=1), "samples": by_month.size()}).round(3).T)

# --- 3. Could rows 35-42 come from it? Its samples in cloud and haze periods ----------------------
print(f"Ambient samples in FM-120 rows of Abbey's classes: cloud {K.class_at(amb.index, abbey_cloud).sum()}, "
      f"haze {K.class_at(amb.index, abbey_haze).sum()} (Abbey's classes exist only while the GCVI sampled)")
AMB_TEST = {}
for kind, col, rows_mask in (("cloud", 0, vis_cloud), ("haze", 1, vis_haze)):
    in_kind = K.class_at(amb.index, rows_mask)
    for species, rows in SPECIES.items():
        for label, values in (("AMS_ambient.nc as stored", amb[f"{species}_ugm3"]),
                              ("CE-corrected", ams.loc[amb.index, f"{species}_ugm3"])):
            cmp = check(rows[col], values[in_kind], f"AMS ambient {species} ({label}), {kind} rows of Kavin's "
                        "visibility/LWC classes", "AMS sample (~5.5 min)", primary=False, show=False)
            AMB_TEST[(kind, species, label)] = cell(cmp) + [f"{cmp['diff_pct'].abs().median():.0f}"]
display(Markdown("Ambient AMS in Kavin's visibility/LWC cloud and haze rows vs the sheet (µg m⁻³; difference; last column = median |diff| %):"))
display(pd.DataFrame(AMB_TEST, index=S.SEASON_ORDER + ["median |diff| %"]).T)
del amb, ratio

AMS_ambient.nc: 52,100 samples, 2023-02-19 02:27:54 to 2024-02-14 23:54:09; 100.0% of its time stamps are in the CE-corrected file.
CE-corrected samples missing from it: 7,965. Of the 5,502 GCVI-AMS samples of Section 10a, 5,493 are missing from it -> it holds the ambient-inlet samples only.


Ratio AMS_ambient / CE-corrected file per month (all species): one constant per month, the monthly collection efficiency, so AMS_ambient.nc is **not** CE-corrected

time,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02
min,0.7,0.6,0.9,0.9,0.6,0.4,0.5,0.4,0.5,0.4,0.4,0.6,0.6
median,0.7,0.6,0.9,0.9,0.6,0.4,0.5,0.4,0.5,0.4,0.4,0.6,0.6
max,0.7,0.6,0.9,0.9,0.6,0.4,0.5,0.4,0.5,0.4,0.4,0.6,0.6
samples,565.0,6535.0,5360.0,3041.0,5365.0,5289.0,3805.0,4676.0,4251.0,6683.0,5479.0,41.0,1010.0


Ambient samples in FM-120 rows of Abbey's classes: cloud 0, haze 0 (Abbey's classes exist only while the GCVI sampled)


Ambient AMS in Kavin's visibility/LWC cloud and haze rows vs the sheet (µg m⁻³; difference; last column = median |diff| %):

EPCAPE                 Spring                 Summer                   Fall                 Winter median |diff| %
cloud sulfate  AMS_ambient.nc as stored  0.037 ± 0.063 (-90 %)  0.149 ± 0.081 (-29 %)    0.34 ± 0.14 (-22 %)  0.025 ± 0.017 (-93 %)  0.026 ± 0.029 (-76 %)              76
               CE-corrected                0.07 ± 0.12 (-81 %)     0.22 ± 0.12 (+5 %)    0.69 ± 0.29 (+56 %)  0.060 ± 0.041 (-83 %)  0.044 ± 0.048 (-60 %)              60
      nitrate  AMS_ambient.nc as stored  0.062 ± 0.055 (-85 %)  0.045 ± 0.019 (-87 %)  0.159 ± 0.040 (-65 %)  0.040 ± 0.035 (-93 %)  0.066 ± 0.057 (-45 %)              85
               CE-corrected              0.111 ± 0.097 (-74 %)  0.061 ± 0.015 (-82 %)  0.319 ± 0.081 (-29 %)  0.099 ± 0.088 (-83 %)   0.109 ± 0.094 (-9 %)              74
      ammonium AMS_ambient.nc as stored  0.030 ± 0.034 (-88 %)  0.022 ± 0.012 (-88 %)  0.145 ± 0.059 (-42 %)  0.016 ± 0.012 (-94 %)  0.027 ± 0.022 (-62 %)              88
               CE-corrected              0.054 ± 0.066 (-77 %)  0.029 ± 0.013 (-85 %)    0.29 ± 0.12 (+16 %)  0.037 ± 0.028 (-87 %)  0.045 ± 0.036 (-36 %)              77
      organics AMS_ambient.nc as stored    0.26 ± 0.29 (-58 %)  0.120 ± 0.049 (-71 %)  0.528 ± 0.017 (-23 %)    0.18 ± 0.14 (-76 %)    0.27 ± 0.32 (+81 %)              71
               CE-corrected                0.46 ± 0.49 (-25 %)  0.178 ± 0.087 (-58 %)  1.056 ± 0.034 (+53 %)    0.43 ± 0.34 (-40 %)   0.45 ± 0.53 (+201 %)              53
haze  sulfate  AMS_ambient.nc as stored   0.20 ± 0.19 (+102 %)   0.23 ± 0.25 (+420 %)   0.38 ± 0.25 (+221 %)  0.096 ± 0.091 (-13 %)    0.16 ± 0.13 (+32 %)             102
               CE-corrected               0.38 ± 0.41 (+281 %)   0.37 ± 0.40 (+721 %)   0.87 ± 0.59 (+625 %)   0.22 ± 0.19 (+104 %)   0.26 ± 0.22 (+119 %)             281
      nitrate  AMS_ambient.nc as stored    0.15 ± 0.16 (-30 %)    0.12 ± 0.23 (+24 %)     0.20 ± 0.13 (+9 %)    0.14 ± 0.16 (-67 %)     0.15 ± 0.16 (-9 %)              24
               CE-corrected                0.28 ± 0.30 (+28 %)    0.17 ± 0.26 (+73 %)   0.45 ± 0.33 (+152 %)    0.35 ± 0.37 (-20 %)    0.24 ± 0.27 (+51 %)              51
      ammonium AMS_ambient.nc as stored    0.12 ± 0.11 (+27 %)   0.19 ± 0.18 (+297 %)   0.19 ± 0.14 (+138 %)  0.069 ± 0.071 (-57 %)   0.097 ± 0.084 (+6 %)              57
               CE-corrected               0.22 ± 0.23 (+141 %)   0.28 ± 0.25 (+490 %)   0.43 ± 0.33 (+444 %)     0.16 ± 0.16 (+3 %)    0.16 ± 0.14 (+77 %)             141
      organics AMS_ambient.nc as stored   0.57 ± 0.59 (+185 %)   0.38 ± 0.46 (+358 %)   0.95 ± 0.63 (+401 %)    0.51 ± 0.46 (+66 %)   0.48 ± 0.56 (+130 %)             185
               CE-corrected                 1.1 ± 1.2 (+436 %)   0.57 ± 0.55 (+590 %)    2.2 ± 1.6 (+1051 %)     1.2 ± 1.1 (+301 %)   0.80 ± 0.93 (+283 %)             436

### 10e. Kavin's AMS code and Abbey Williams' 15-min file (rows 25, 26, 28, 29, 33–42, 54, 55)

Kavin's `AMSforAndrew.mlx` works on Abbey Williams' **15-min merged file**, `EPCAPE_15mindata.nc`. Its `ClrCldHaz_class` is 1 (cloud) or 2 (haze) only while the CVI is on (`CVIon_class` = 1); it is 0 when the CVI is off. His code:

* **Hours** = number of 15-min intervals of the class ÷ 4, whether or not the AMS reported, by calendar quarter. The sheet holds them **rounded**, so the fall haze "5.0×10²" is 49.5 h, a typo for 5.0×10¹.
* **Residuals** = mean ± std over the class's intervals of `AMS_mass_15min_EF`, a 5 × N matrix. Its rows are nitrate, sulfate, organics, ammonium and chloride (Abbey); that's the "vector 5 units long".
  * The matrix comes from `EPCAPE_15min_data.mat`, which wasn't provided. The `AMS_mass` of the NetCDF reproduces every residual value, and step 2 shows what it is.
  * The script as shared averages `AMS_massfrac` (mass fraction) for spring only. The sheet's spring values are concentrations.

**Not in the AMS script, but the same file and classes:** they also reproduce the FM-120 rows.
* Droplet number (25, 28): `FM120_Nd`.
* LWC (54, 55): `FM120_LWC`.
* Effective diameter (26, 29): Σ N D³ / Σ N D² of `FM120_DSD`, with **each bin's upper edge** (`FM120_diam` = 3, 4, …, 50 µm) as its diameter.

**This check with the same classes** keeps its own samples and gives each one the class of the 15-min interval that holds it. That makes these the primary definitions for these rows. The samples are:
* the AMS residual samples of Section 10a (CE-corrected ÷ EF, per 2-min V-mode period);
* the daily FM-120 5-min rows (`nd_cm3`, `ed_um`, `lwc_gm3`);
* hours from the class intervals in which the AMS reported.

Seasons are the sheet's.

In [27]:
# =============================================================================
# Section 10e: Kavin's AMS code (AMSforAndrew.mlx) and Abbey Williams' 15-min file
#              (sheet rows 25, 26, 28, 29, 33-42, 54, 55)
# =============================================================================
# Inputs
#   KAVIN_DIR/EPCAPE_15mindata.nc (K.read_kavin_15min)  Abbey's 15-min merged file: ClrCldHaz_class (1 = cloud,
#        2 = haze; set only while the CVI is on), CVIon_class, AMS_mass (5 species, ug m-3), AMS_massfrac,
#        FM120_Nd, FM120_LWC, FM120_DSD (dN per bin; bin diameters = upper edges); 15-min steps, labelled by START
#   ams, gcvi, resid (Section 10a)    CE-corrected AMS, the 1-s GCVI record, this check's AMS residual samples (/ EF)
#   SPECIES (10b), hour_refs, cell (10c)  species -> (cloud row, haze row); sheet hours; 'ours (diff %)' formatter
# Kavin's code: hours = class intervals / 4; averages = mean ± std over the class's 15-min intervals;
# calendar quarters (UTC). This check: own samples, each taking the class of its 15-min interval; sheet seasons.

# --- 1. Read the 15-min file; the class is set only while the CVI is on ----------------------------------------
F15, DSD15 = K.read_kavin_15min(KAVIN_DIR)
cls15 = F15["clr_cld_haz_class"]
print(f"{len(F15):,} 15-min intervals, {F15.index[0]} to {F15.index[-1]} UTC")
display(pd.crosstab(cls15.fillna(-1).rename("ClrCldHaz_class (-1 = empty)"), F15["cvi_on_class"].rename("CVIon_class")))

# --- 2. What the file's AMS_mass and FM-120 values are ----------------------------------------------------------
#   CVI off: AMS_mass vs the CE-corrected AMS file averaged over the same 15-min intervals
#   CVI on : AMS_mass vs this check's residuals (CE-corrected / EF, any segment length) averaged the same way
resid_any, _ = Q.gcvi_ams_residuals(ams, gcvi, Q.gcvi_segments(gcvi), min_duration_min=0)
what_is = {}
for species in SPECIES:
    k15 = F15[f"{species}_ugm3"]
    for state, reference, label in ((0, ams[f"{species}_ugm3"], "CVI off vs CE-corrected AMS"),
                                    (1, resid_any[f"{species}_resid_ugm3"], "CVI on vs CE-corrected AMS / EF")):
        ref15 = reference.resample("15min").mean().reindex(F15.index)
        ok = (F15["cvi_on_class"] == state) & k15.notna() & ref15.notna()
        what_is[(species, label)] = {"intervals": int(ok.sum()), "median ratio": (k15[ok] / ref15[ok]).median(),
                                     "correlation r": np.corrcoef(k15[ok], ref15[ok])[0, 1]}
display(Markdown("The file's AMS_mass against the AMS file of Sections 10a–10d, averaged over the same 15-min intervals:"))
display(pd.DataFrame(what_is).T.round(3))
fm5 = SRC.load_fm120()  # the daily FM-120 5-min rows (freed after Section 9e)
nd15 = fm5["nd_cm3"].resample("15min").mean().reindex(F15.index)
ok = nd15.notna() & F15["fm120_nd_cm3"].notna()
print(f"FM120_Nd vs the daily FM-120 5-min files averaged to 15 min: median ratio "
      f"{(F15['fm120_nd_cm3'][ok] / nd15[ok]).median():.3f}, r = {np.corrcoef(F15['fm120_nd_cm3'][ok], nd15[ok])[0, 1]:.3f}")
# Cloud/haze intervals with AMS data but no GCVI EF > 1 in the 1-s record: there the file's AMS_mass is the
# CE-corrected value NOT divided by EF (~5x a residual), and Kavin's averages include it.
ef_seconds = gcvi["EF"].where(gcvi["EF"] > 1).resample("15min").count().reindex(F15.index).fillna(0)
NO_EF = cls15.isin([1, 2]) & F15[[f"{s}_ugm3" for s in SPECIES]].notna().any(axis=1) & (ef_seconds == 0)
no_ef_months = ", ".join(f"{period.strftime('%b %Y')}: {n}" for period, n in
                         NO_EF[NO_EF].groupby(NO_EF[NO_EF].index.to_period("M")).size().items())
no_ef_ratio = (F15.loc[NO_EF, "sulfate_ugm3"] / ams["sulfate_ugm3"].resample("15min").mean().reindex(F15.index)[NO_EF]).median()
NO_EF_TEXT = (f"{int(NO_EF.sum())} cloud/haze intervals with AMS data but no GCVI EF record ({no_ef_months}), whose "
              f"AMS_mass is the CE-corrected value not divided by EF (ratio to it {no_ef_ratio:.2f})")
print(NO_EF_TEXT)

# --- 3. Hours (rows 33, 34) ----------------------------------------------------------------------------------------
HOURS15 = K.kavin_15min_hours(cls15)
has_ams = F15[[f"{s}_ugm3" for s in K.AMS_15MIN_SPECIES]].notna().any(axis=1)
hours15_table = {}
for row, kind, code in ((33, "cloud", 1), (34, "haze", 2)):
    ref = pd.DataFrame({"mean": hour_refs[row], "std": np.nan, "raw": [f"{v:g}" for v in hour_refs[row]]}, index=S.SEASON_ORDER)
    with_ams = (cls15 == code) & has_ams
    for stats, definition, primary, label in (
            (pd.DataFrame({"mean": HOURS15[f"{kind}_h"], "std": np.nan, "median": np.nan, "n": HOURS15[f"{kind}_n"]}),
             f"Kavin's AMS code: Abbey's 15-min {kind} intervals / 4; calendar quarters", False, "Kavin's code"),
            (pd.DataFrame({"mean": S.seasonal_sums(pd.Series(0.25, index=F15.index[with_ams])), "std": np.nan, "median": np.nan,
                           "n": S.seasonal_sums(pd.Series(1.0, index=F15.index[with_ams])).astype(int)}),
             f"Abbey's 15-min {kind} intervals in which the AMS reported, × 15 min; sheet seasons", True,
             "this check: class intervals with AMS data")):
        cmp = S.compare_to(stats, ref, row, TABLE.loc[row, "quantity"], definition, close_pct=CLOSE_PCT).assign(
            sample="15-min interval (summed to hours)", primary=primary)
        SUMMARY.append(cmp)
        hours15_table[(row, label)] = cell(cmp)
    # MATLAB's round() rounds halves away from zero (100.5 -> 101); Python's round() would give 100
    hours15_table[(row, "Kavin's code, rounded as in the sheet")] = [f"{np.floor(v + 0.5):.0f}" for v in HOURS15[f"{kind}_h"]]
    hours15_table[(row, "SHEET (Fall 34 = typo corrected)")] = ref["raw"].tolist()
display(Markdown("**Rows 33–34, hours: Kavin's code on Abbey's 15-min classes, and this check (difference from the sheet)**"))
display(pd.DataFrame(hours15_table, index=S.SEASON_ORDER).T)

# --- 4. Residual concentrations (rows 35-42), µg m-3 --------------------------------------------------------------
resid_class = K.class_of_times(resid.index, cls15)  # 15-min class of each AMS residual sample of Section 10a
resid15_table, rank15 = {}, []
for kind, col, code in (("cloud", 0, 1), ("haze", 1, 2)):
    for species, rows_ in SPECIES.items():
        row = rows_[col]
        k15 = F15[f"{species}_ugm3"].where(cls15 == code)
        kavin_stats = K.quarter_stats(k15, utc_offset_h=0)
        shared_stats = kavin_stats.copy()  # the script as shared: spring from AMS_massfrac
        shared_stats.loc["Spring"] = K.quarter_stats(F15[f"{species}_frac"].where(cls15 == code), utc_offset_h=0).loc["Spring"]
        for cmp, label in (
                (check_stats(row, kavin_stats, f"Kavin's AMS code: mean of Abbey's 15-min AMS_mass (already ÷ EF) over "
                             f"{kind} intervals; calendar quarters", "15-min interval", show=False), "Kavin's code"),
                (check_stats(row, shared_stats, "Kavin's AMS code as shared: spring from AMS_massfrac (mass fraction), "
                             "other seasons as above", "15-min interval", show=False), "Kavin's code as shared (spring = fraction)"),
                (check_stats(row, K.quarter_stats(k15.where(~NO_EF), utc_offset_h=0),
                             f"Kavin's AMS code without the {int(NO_EF.sum())} intervals lacking a GCVI EF record",
                             "15-min interval", show=False), "Kavin's code without the no-EF intervals"),
                (check(row, resid[f"{species}_resid_ugm3"][resid_class == code],
                       f"AMS {species} behind the GCVI / EF, {kind} samples: Abbey's 15-min classes", "AMS sample (~5.5 min)",
                       primary=True, show=False), "this check: own AMS samples, Abbey's 15-min classes")):
            resid15_table[(kind, species, label)] = cell(cmp)
            rank15.append({"class": kind, "calculation": label, "median |diff| (%)": cmp["diff_pct"].abs().median()})
        resid15_table[(kind, species, "SHEET (Kavin)")] = S.table_values(TABLE, row)["raw"].tolist()
display(Markdown("**Rows 35–42, residuals (µg m⁻³): Kavin's code and this check (difference from the sheet)**"))
display(pd.DataFrame(resid15_table, index=S.SEASON_ORDER).T)
display(pd.DataFrame(rank15).groupby(["class", "calculation"])["median |diff| (%)"].median().round(1).unstack(0))

# --- 5. FM-120 rows (25, 26, 28, 29, 54, 55) with the same 15-min classes ------------------------------------------
upper_um = DSD15.columns.to_numpy()
mid_um = upper_um - np.diff(np.r_[2.0, upper_um]) / 2  # bin mid-points (the first bin is 2-3 um; readme_FM120.md)
d_eff_upper = K.effective_diameter_um(DSD15, upper_um)  # Kavin: each bin's upper edge as its diameter
d_eff_mid = K.effective_diameter_um(DSD15, mid_um)
fm_class = K.class_of_times(fm5.index, cls15)  # 15-min class of each FM-120 5-min row
ed5 = fm5["ed_um"].where(fm5["nd_cm3"] > 0)
fm15_table = {}
for row, code, label, kavin_values, kavin_how, mine in (
        (25, 1, "number concentration (cm⁻³)", F15["fm120_nd_cm3"], "FM120_Nd", fm5["nd_cm3"]),
        (26, 1, "effective diameter (µm)", d_eff_upper, "D_eff from FM120_DSD with bin upper edges", ed5),
        (28, 2, "number concentration (cm⁻³)", F15["fm120_nd_cm3"], "FM120_Nd", fm5["nd_cm3"]),
        (29, 2, "effective diameter (µm)", d_eff_upper, "D_eff from FM120_DSD with bin upper edges", ed5),
        (54, 1, "liquid water content (g m⁻³)", F15["fm120_lwc_gm3"], "FM120_LWC", fm5["lwc_gm3"]),
        (55, 2, "liquid water content (g m⁻³)", F15["fm120_lwc_gm3"], "FM120_LWC", fm5["lwc_gm3"])):
    kind = K.CLASS_CODES[code]
    lines = [(check_stats(row, K.quarter_stats(kavin_values.where(cls15 == code), utc_offset_h=0),
                          f"Kavin's method: Abbey's 15-min file, {kavin_how}, mean over {kind} intervals; calendar quarters",
                          "15-min interval", show=False), "Kavin's method")]
    if row in (26, 29):
        lines.append((check_stats(row, K.quarter_stats(d_eff_mid.where(cls15 == code), utc_offset_h=0),
                                  f"Abbey's 15-min file, D_eff from FM120_DSD with bin mid-points, {kind} intervals; calendar "
                                  "quarters", "15-min interval", show=False), "same, bin mid-points"))
    lines.append((check(row, mine[fm_class == code], f"FM-120 5-min rows classed {kind} by Abbey's 15-min classes: {label}",
                        "FM-120 5-min mean", primary=True, show=False), "this check: FM-120 5-min rows, Abbey's 15-min classes"))
    for cmp, line_label in lines:
        fm15_table[(row, f"{kind} {label}", line_label)] = cell(cmp)
    fm15_table[(row, f"{kind} {label}", "SHEET (Kavin)")] = S.table_values(TABLE, row)["raw"].tolist()
display(Markdown("**FM-120 rows with Abbey's 15-min classes: Kavin's method and this check (difference from the sheet)**"))
display(pd.DataFrame(fm15_table, index=S.SEASON_ORDER).T)
del fm5, DSD15, resid_any, nd15

35,041 15-min intervals, 2023-02-15 00:00:00 to 2024-02-15 00:00:00 UTC


CVIon_class,0.0,0.5,1.0
ClrCldHaz_class (-1 = empty),,,
-1.0,0,4268,0
0.0,28419,0,0
1.0,0,0,1409
2.0,0,0,945


The file's AMS_mass against the AMS file of Sections 10a–10d, averaged over the same 15-min intervals:

intervals  median ratio  correlation r
sulfate  CVI off vs CE-corrected AMS        18462.0           1.0          1.000
         CVI on vs CE-corrected AMS / EF     1529.0           1.0          1.000
nitrate  CVI off vs CE-corrected AMS        18470.0           1.0          0.998
         CVI on vs CE-corrected AMS / EF     1532.0           1.0          1.000
ammonium CVI off vs CE-corrected AMS        18224.0           1.0          0.999
         CVI on vs CE-corrected AMS / EF     1527.0           1.0          1.000
organics CVI off vs CE-corrected AMS        18449.0           1.0          1.000
         CVI on vs CE-corrected AMS / EF     1520.0           1.0          1.000

FM120_Nd vs the daily FM-120 5-min files averaged to 15 min: median ratio 1.000, r = 1.000
43 cloud/haze intervals with AMS data but no GCVI EF record (Mar 2023: 12, Apr 2023: 31), whose AMS_mass is the CE-corrected value not divided by EF (ratio to it 1.00)


...

**Rows 35–42, residuals (µg m⁻³): Kavin's code and this check (difference from the sheet)**

EPCAPE                  Spring                Summer                  Fall                 Winter
cloud sulfate  Kavin's code                                           0.36 ± 0.29 (-1 %)      0.21 ± 0.22 (+1 %)    0.44 ± 0.27 (+0 %)    0.35 ± 0.35 (-0 %)   0.109 ± 0.035 (-1 %)
               Kavin's code as shared (spring = fraction)             0.36 ± 0.29 (-1 %)   0.187 ± 0.056 (-11 %)    0.44 ± 0.27 (+0 %)    0.35 ± 0.35 (-0 %)   0.109 ± 0.035 (-1 %)
               Kavin's code without the no-EF intervals               0.34 ± 0.28 (-4 %)   0.156 ± 0.098 (-26 %)    0.44 ± 0.27 (+0 %)    0.35 ± 0.35 (-0 %)   0.109 ± 0.035 (-1 %)
               this check: own AMS samples, Abbey's 15-min classes    0.34 ± 0.29 (-4 %)     0.16 ± 0.10 (-25 %)    0.44 ± 0.29 (+1 %)    0.35 ± 0.35 (+0 %)   0.107 ± 0.038 (-3 %)
               SHEET (Kavin)                                                 0.36 ± 0.29             0.21 ± 0.22           0.44 ± 0.27           0.35 ± 0.35            0.11 ± 0.04
      nitrate  Kavin's code                                           0.43 ± 0.42 (+1 %)      0.34 ± 0.36 (-1 %)    0.45 ± 0.44 (+1 %)    0.57 ± 0.40 (-1 %)   0.122 ± 0.075 (+2 %)
               Kavin's code as shared (spring = fraction)             0.43 ± 0.42 (+1 %)   0.282 ± 0.066 (-17 %)    0.45 ± 0.44 (+1 %)    0.57 ± 0.40 (-1 %)   0.122 ± 0.075 (+2 %)
               Kavin's code without the no-EF intervals               0.41 ± 0.39 (-5 %)     0.24 ± 0.17 (-28 %)    0.45 ± 0.44 (+1 %)    0.57 ± 0.40 (-1 %)   0.122 ± 0.075 (+2 %)
               this check: own AMS samples, Abbey's 15-min classes    0.41 ± 0.43 (-4 %)     0.25 ± 0.18 (-28 %)    0.45 ± 0.50 (+1 %)    0.56 ± 0.44 (-1 %)   0.111 ± 0.073 (-7 %)
               SHEET (Kavin)                                                 0.43 ± 0.42             0.34 ± 0.36           0.45 ± 0.44           0.57 ± 0.40            0.12 ± 0.07
      ammonium Kavin's code                                           0.24 ± 0.18 (-2 %)      0.19 ± 0.20 (-2 %)    0.25 ± 0.16 (+2 %)    0.28 ± 0.20 (-1 %)   0.071 ± 0.036 (-0 %)
               Kavin's code as shared (spring = fraction)             0.24 ± 0.18 (-2 %)   0.153 ± 0.019 (-19 %)    0.25 ± 0.16 (+2 %)    0.28 ± 0.20 (-1 %)   0.071 ± 0.036 (-0 %)
               Kavin's code without the no-EF intervals               0.22 ± 0.16 (-7 %)   0.135 ± 0.090 (-29 %)    0.25 ± 0.16 (+2 %)    0.28 ± 0.20 (-1 %)   0.071 ± 0.036 (-0 %)
               this check: own AMS samples, Abbey's 15-min classes    0.22 ± 0.17 (-7 %)   0.136 ± 0.093 (-29 %)    0.25 ± 0.17 (+2 %)    0.28 ± 0.21 (-1 %)   0.068 ± 0.036 (-5 %)
               SHEET (Kavin)                                                0.24 ± 0.18              0.19 ± 0.20           0.25 ± 0.16           0.28 ± 0.20          0.071 ± 0.036
      organics Kavin's code                                           0.61 ± 0.59 (+0 %)      0.42 ± 0.48 (+0 %)    0.69 ± 0.66 (-0 %)    0.73 ± 0.46 (+0 %)   0.154 ± 0.083 (+3 %)
               Kavin's code as shared (spring = fraction)             0.61 ± 0.59 (+0 %)   0.330 ± 0.069 (-21 %)    0.69 ± 0.66 (-0 %)    0.73 ± 0.46 (+0 %)   0.154 ± 0.083 (+3 %)
               Kavin's code without the no-EF intervals               0.58 ± 0.56 (-4 %)     0.30 ± 0.20 (-29 %)    0.69 ± 0.66 (-0 %)    0.73 ± 0.46 (+0 %)   0.154 ± 0.083 (+3 %)
               this check: own AMS samples, Abbey's 15-min classes    0.58 ± 0.63 (-4 %)     0.30 ± 0.21 (-28 %)    0.69 ± 0.74 (+0 %)    0.73 ± 0.55 (+0 %)   0.156 ± 0.084 (+4 %)
               SHEET (Kavin)                                                 0.61 ± 0.59             0.42 ± 0.49           0.69 ± 0.66           0.73 ± 0.46            0.15 ± 0.08
haze  sulfate  Kavin's code                                           0.10 ± 0.12 (+1 %)    0.044 ± 0.047 (-1 %)    0.12 ± 0.11 (-4 %)  0.107 ± 0.100 (-3 %)     0.12 ± 0.22 (-3 %)
               Kavin's code as shared (spring = fraction)             0.10 ± 0.12 (+1 %)  0.155 ± 0.077 (+246 %)    0.12 ±

class,cloud,haze
calculation,,
Kavin's code,1.0,0.8
Kavin's code as shared (spring = fraction),1.0,1.4
Kavin's code without the no-EF intervals,1.7,3.2
"this check: own AMS samples, Abbey's 15-min classes",4.3,3.2


**FM-120 rows with Abbey's 15-min classes: Kavin's method and this check (difference from the sheet)**

EPCAPE                  Spring                  Summer                  Fall                Winter
25 cloud number concentration (cm⁻³)  Kavin's method                                          210 ± 142 (+0 %)        219 ± 117 (-1 %)        181 ± 113 (+1 %)      337 ± 215 (-1 %)       102 ± 66 (+2 %)
                                      this check: FM-120 5-min rows, Abbey's 15-min classes   210 ± 145 (+0 %)        219 ± 119 (-1 %)        181 ± 116 (+1 %)      337 ± 218 (-1 %)       102 ± 71 (+2 %)
                                      SHEET (Kavin)                                                  210 ± 140               220 ± 120               180 ± 110             340 ± 210       1.0 * 10^2 ± 66
26 cloud effective diameter (µm)      Kavin's method                                         13.6 ± 4.4 (-3 %)       14.0 ± 4.0 (-0 %)       13.6 ± 4.2 (-3 %)     12.6 ± 5.2 (-3 %)     15.1 ± 5.2 (+1 %)
                                      same, bin mid-points                                   13.2 ± 4.3 (-6 %)       13.5 ± 4.0 (-3 %)       13.1 ± 4.2 (-6 %)     12.2 ± 5.2 (-6 %)     14.5 ± 5.1 (-3 %)
                                      this check: FM-120 5-min rows, Abbey's 15-min classes  12.7 ± 4.3 (-9 %)       13.3 ± 4.0 (-5 %)       12.7 ± 4.2 (-9 %)    11.3 ± 5.0 (-13 %)    13.1 ± 5.1 (-12 %)
                                      SHEET (Kavin)                                                     14 ± 4                  14 ± 4                  14 ± 4                13 ± 5                15 ± 5
28 haze number concentration (cm⁻³)   Kavin's method                                            33 ± 37 (-0 %)          25 ± 25 (-0 %)          33 ± 27 (-1 %)        29 ± 37 (-1 %)        77 ± 87 (-0 %)
                                      this check: FM-120 5-min rows, Abbey's 15-min classes     33 ± 42 (-0 %)          25 ± 30 (-0 %)          33 ± 33 (-1 %)        29 ± 44 (-1 %)        77 ± 92 (-0 %)
                                      SHEET (Kavin)                                                    33 ± 37                 25 ± 25                 33 ± 27               29 ± 28               77 ± 87
29 haze effective diameter (µm)       Kavin's method                                         12.1 ± 4.5 (+1 %)       13.7 ± 5.7 (-2 %)       12.3 ± 3.9 (+3 %)      9.7 ± 2.6 (-0 %)     10.6 ± 6.2 (-3 %)
                                      same, bin mid-points                                   11.8 ± 4.6 (-2 %)       13.5 ± 5.7 (-4 %)       12.0 ± 4.0 (+0 %)      9.3 ± 2.7 (-4 %)     10.2 ± 6.1 (-8 %)
                                      this check: FM-120 5-min rows, Abbey's 15-min classes  9.4 ± 3.8 (-22 %)      10.1 ± 4.5 (-28 %)       9.8 ± 3.3 (-19 %)     7.3 ± 2.6 (-24 %)     8.8 ± 5.7 (-20 %)
                                      SHEET (Kavin)                                                     12 ± 5                  14 ± 6                  12 ± 4             9.7 ± 2.6                11 ± 6
54 cloud liquid water content (g m⁻³) Kavin's method                                               0.14 ± 0.17      0.16 ± 0.15 (-2 %)      0.12 ± 0.17 (+3 %)    0.17 ± 0.23 (-2 %)    0.12 ± 0.16 (-3 %)
                                      this check: FM-120 5-min rows, Abbey's 15-min classes        0.14 ± 0.18      0.16 ± 0.16 (-2 %)      0.12 ± 0.17 (+3 %)    0.17 ± 0.24 (-2 %)    0.12 ± 0.17 (-3 %)
                                      SHEET (Kavin)                                                       None             0.16 ± 0.15             0.12 ± 0.17           0.17 ± 0.23           0.12 ± 0.16
55 haze liquid water content (g m⁻³)  Kavin's method                                             0.009 ± 0.013  0.0077 ± 0.0081 (-1 %)  0.0089 ± 0.0091 (-0 %)  0.006 ± 0.015 (+1 %)  0.021 ± 0.033 (-1 %)
                                      this check: FM-120 5-min rows, Abbey's 15-min classes      0.009 ± 0.015  0.0077 ± 0.0094 (-1 %)    0.009 ± 0.011 (-0 %)  0.006 ± 0.018 (+1 %)  0.021 ± 0.036 (-1 %)
                                      SHEET 

## 11. Summary

In [28]:
# =============================================================================
# Section 11a: summary table and saved outputs
# =============================================================================
# Input: SUMMARY, the comparison tables every section appended through check().
# Output files in OUT_DIR (data/processed/derived/seasonal_averages_check/):
#   seasonal_averages_check_all_definitions.csv  every definition, primary and sensitivity
#   seasonal_averages_check_primary.csv          one line per sheet row (primary definitions)
#   rows_not_checked.csv                         NOT_CHECKED, with the reason
#   settings.json                                every setting + the git revision

# --- 1. Stack every comparison; keep the primary definition of each row ------------------
summary = pd.concat(SUMMARY, ignore_index=True)
summary["sheet_row"] = summary["sheet_row"].astype(str)  # rows like "35+39" are labels, not numbers
primary = summary[summary["primary"]].copy()

# --- 2. One cell per season: "ours | sheet (difference of means)" ---------------------
primary["cell"] = [f"{o}  |  {t}  ({d:+.0f} %)" if np.isfinite(d) else f"{o}  |  {t}"
                   for o, t, d in zip(primary["ours"], primary["table"].astype(str), primary["diff_pct"])]
primary["row_key"] = primary["sheet_row"].str.extract(r"(\d+)", expand=False).astype(int)  # sort 7 before 10
WIDE = (primary.pivot(index=["row_key", "sheet_row", "quantity", "definition"], columns="season", values="cell")
        [S.SEASON_ORDER].sort_index().droplevel("row_key"))
display(Markdown("**Primary definition per row: this check | sheet (difference of means)**"))
display(WIDE)
display(Markdown("**Rows that could not be re-derived**"))
display(pd.Series(NOT_CHECKED, name="why").to_frame())

# --- 3. Save everything (derived/ is not in git: back it up) --------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT_DIR / "seasonal_averages_check_all_definitions.csv", index=False)
WIDE.to_csv(OUT_DIR / "seasonal_averages_check_primary.csv")
pd.Series(NOT_CHECKED, name="why").to_csv(OUT_DIR / "rows_not_checked.csv")
(OUT_DIR / "settings.json").write_text(json.dumps({**SETTINGS, "git_revision": GIT_REVISION}, indent=1, default=str))
print(f"Wrote tables and settings to {OUT_DIR}")

**Primary definition per row: this check | sheet (difference of means)**

,,season,EPCAPE,Spring,Summer,Fall,Winter
sheet_row,quantity,definition,,,,,
7,Shortwave (SW) Transmittance,"SW↓ / RADFLUX clear-sky SW↓, 5-min mean, SZA ≤ 80°, QC = 0 (Scripps Pier)",0.71 ± 0.32 | 0.58 ± 0.29 (+22 %),0.61 ± 0.34 | 0.50 ± 0.31 (+22 %),0.67 ± 0.31 | 0.56 ± 0.28 (+19 %),0.84 ± 0.25 | 0.66 ± 0.24 (+27 %),0.76 ± 0.32 | 0.61 ± 0.28 (+24 %)
8,Longwave (LW) Flux,"RADFLUX downwelling LW, every QC = 0 sample (no further averaging), QC = 0 (Scripps Pier, not Soledad)",347 ± 41 | 350 ± 30 (-1 %),355 ± 36 | 370 ± 10 (-4 %),381 ± 28 | 380 ± 10 (+0 %),332 ± 34 | 323 ± 5 (+3 %),318 ± 36 | 308 ± 3 (+3 %)
10,Liquid Water Path,"MWRRET v1 (2-channel MWR, mwrret1liljclou.c2): phys_lwp, cloudy scenes, rain-free, QC = 0",75 ± 94 | 56 ± 127 (+34 %),107 ± 100 | 92 ± 161 (+17 %),63 ± 83 | 59 ± 118 (+6 %),55 ± 83 | 42 ± 100 (+32 %),53 ± 92 | 33 ± 92 (+60 %)
15,Cloud Top Height,"KAZR-ARSCL top of lowest layer, ≤ 3500 m AGL",902 ± 615 | 920 ± 380 (-2 %),814 ± 356 | 890 ± 330 (-9 %),616 ± 408 | 774 ± 353 (-20 %),1212 ± 774 | 1057 ± 416 (+15 %),1482 ± 860 | 1192 ± 397 (+24 %)
17,Boundary Layer Height,PBLHTTHERMO when KAZR-ARSCL sees a low cloud (±5 min) [best-matching reading],804 ± 411 | 899 ± 305 (-11 %),779 ± 290 | 886 ± 256 (-12 %),618 ± 234 | 764 ± 206 (-19 %),920 ± 472 | 1014 ± 343 (-9 %),1114 ± 595 | 1124 ± 442 (-1 %)
20,Lifting Condensation Level,SONDEPARAM surface-parcel LCL (data_quality = 0),435 ± 396 | 637 ± 483 (-32 %),413 ± 244 | 490 ± 261 (-16 %),268 ± 146 | 361 ± 165 (-26 %),649 ± 686 | 917 ± 676 (-29 %),617 ± 432 | 787 ± 428 (-22 %)
21,Cloud Base Height,"Ceilometer first_cbh ≤ 3500 m, QC = 0",633 ± 562 | 661 ± 316 (-4 %),551 ± 353 | 619 ± 270 (-11 %),417 ± 397 | 554 ± 254 (-25 %),858 ± 730 | 836 ± 389 (+3 %),1028 ± 747 | 929 ± 362 (+11 %)
22,Rain Accumulation - LDM1,"LD M1: F monthly mean of D (mm/day), std across months",3.4 ± 3.3 | 3.1 ± 3.4 (+11 %),0.73 ± 0.47 | 0.2 ± 0.1 (+266 %),3.9 ± 6.0 | 3.6 ± 5.5 (+9 %),2.7 ± 2.3 | 1.7 ± 1.7 (+60 %),5.6 ± 1.4 | 5.8 ± 1.9 (-3 %)
23,Rain Accumulation - LDS1,"LD S2: F monthly mean of D (mm/day), std across months",4.4 ± 4.2 | 1.7 ± 1.7 (+156 %),0.94 ± 0.31 | 0.22 ± .14 (+327 %),4.2 ± 6.8 | 0.21 ± .11 (+1923 %),3.2 ± 3.5 | 2.8 ± n/a (+15 %),7.8 ± 2.0 | 3.2 ± 1.4 (+145 %)


**Rows that could not be re-derived**

,why
18,PBLH-MPL: no MPL-based PBL-height product for EPCAPE in ARM Data Discovery (2026-10-06...
19,BL-Depth (Miller): source not identified (not an ARM product for EPCAPE)


Wrote tables and settings to /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/processed/derived/seasonal_averages_check


In [29]:
# =============================================================================
# Section 11b: QC report
# =============================================================================
# Input: QC_INFO, one dict per screened variable appended by every section:
#   variable, has_qc (a qc_ field exists), n (all samples), n_finite (finite before QC),
#   n_kept (finite after QC = 0 and any extra screens such as the cloud mask).
# Variables with has_qc = False were used as distributed.
# Which variables had a QC field, and how many samples QC = 0 kept
QC_TABLE = Q.qc_report(QC_INFO)
display(QC_TABLE)
print("Used without a QC field (as distributed):", ", ".join(QC_TABLE.index[~QC_TABLE["has_qc"]]))

,has_qc,n,n_finite,n_kept,kept_pct_of_finite
variable,,,,,
downwelling_shortwave,True,524160,524154,524154,100.0
clearsky_downwelling_shortwave,False,524160,524160,524160,100.0
downwelling_longwave,True,524160,524151,522527,99.7
phys_lwp,True,1251788,1229607,799735,65.0
cloudy samples (for phys_lwp),False,1251788,799735,799735,100.0
phys_lwp,True,22047667,22047667,21316698,96.7
phys_qc_flag == 0 (for phys_lwp),True,22047667,21316698,21316698,100.0
cloudy samples (for phys_lwp),False,22047667,21316698,11405874,53.5
cloud_layer_top_height,False,7884000,3735117,2205853,59.1


Used without a QC field (as distributed): clearsky_downwelling_shortwave, cloudy samples (for phys_lwp), cloudy samples (for phys_lwp), cloud_layer_top_height, pbl_height, rain_rate


In [30]:
# --- Summary figure: COMMENTED OUT for now (not needed); remove the leading '# ' to draw it again ---
# # --- Summary figure: sheet vs this check, one panel per row ------------------
# # Input: `primary` (Section 11a), the primary definition of every checked row.
# # One panel per sheet row: x = campaign + four seasons; dots = means, whiskers = ±1 std.
# # Grey = the sheet (reference); violet = this check (categorical slot 7, so the
# # blue/orange/aqua instrument identities used in other EPCAPE figures stay unique).
# # Saved as OUT_DIR/seasonal_averages_check.png.
# SHEET_COLOR, OURS_COLOR = INK_3, "#4a3aa7"
# PANELS = {  # sheet row -> (short title, y label)
#     "7": ("SW transmittance", "ratio"), "8": ("LW↓ (Pier vs sheet's Soledad)", "W m⁻²"),
#     "10": ("Liquid water path", "g m⁻²"), "15": ("Cloud-top height", "m"),
#     "17": ("PBL height (thermo, low cloud)", "m"), "20": ("LCL", "m"), "21": ("Cloud-base height", "m"),
#     "22": ("Rain, LD M1", "rain"), "23": ("Rain, LD S2 (sheet: LDS1)", "rain"), "24": ("Rain, VDIS M1", "rain"),
#     "25": ("Cloud droplet number", "cm⁻³"), "26": ("Cloud droplet eff. diameter", "µm"),
#     "27": ("Cloud activated fraction (Kavin's file)", "fraction"),
#     "28": ("Haze droplet number", "cm⁻³"), "29": ("Haze droplet eff. diameter", "µm"),
#     "30": ("Haze activated fraction (Kavin's file)", "fraction"),
#     "31": ("FM-120 cloud hours", "h"), "32": ("FM-120 haze hours", "h"),
#     "33": ("GCVI-AMS cloud hours", "h"), "34": ("GCVI-AMS haze hours", "h"),
#     "35": ("Cloud residual SO₄", "µg m⁻³"), "36": ("Cloud residual NO₃", "µg m⁻³"),
#     "37": ("Cloud residual NH₄", "µg m⁻³"), "38": ("Cloud residual organics", "µg m⁻³"),
#     "39": ("Haze residual SO₄", "µg m⁻³"), "40": ("Haze residual NO₃", "µg m⁻³"),
#     "41": ("Haze residual NH₄", "µg m⁻³"), "42": ("Haze residual organics", "µg m⁻³"),
# }
# # y-axis units of the rain panels follow the rain definition chosen in Section 8
# RAIN_UNITS = {"A": "mm h⁻¹", "B": "mm h⁻¹ (event mean)", "C": "mm per event", "D": "mm day⁻¹",
#               "E": "mm per wet hour", "F": "mm day⁻¹ (monthly means)", "G": "mm h⁻¹ (monthly means)"}
# # --- 1. Layout: 3 columns, as many rows as needed ------------------------------------
# rows_to_plot = [r for r in PANELS if r in set(primary["sheet_row"])]
# ncol = 3
# nrow = int(np.ceil(len(rows_to_plot) / ncol))
# fig, axes = plt.subplots(nrow, ncol, figsize=(11, 2.5 * nrow), constrained_layout=True)
# x = np.arange(len(S.SEASON_ORDER))
# # --- 2. One panel per row: sheet (left, grey) and this check (right, violet) ---------------
# for ax, r in zip(axes.flat, rows_to_plot):
#     p = primary[primary["sheet_row"] == r].set_index("season").loc[S.SEASON_ORDER]
#     ax.errorbar(x - 0.12, p["table_mean"], yerr=p["table_std"], fmt="o", color=SHEET_COLOR, ms=5,
#                 elinewidth=1, capsize=0, label="Sheet (Kavin)")
#     ax.errorbar(x + 0.12, p["ours_mean"], yerr=p["ours_std"], fmt="o", color=OURS_COLOR, ms=5,
#                 elinewidth=1, capsize=0, label="This check (QC = 0)")
#     title, ylabel = PANELS[r]
#     ylabel = RAIN_UNITS.get(BEST_RAIN, "") if ylabel == "rain" else ylabel
#     ax.set_title(f"Row {r}: {title}", loc="left", fontsize=10)
#     ax.set_ylabel(ylabel, color=INK_2)
#     ax.set_xticks(x, ["Campaign", "Spr", "Sum", "Fall", "Win"])
#     ax.grid(axis="x", visible=False)
# # --- 3. Legend in the first empty panel (or below the grid), title, save -------------------
# handles, labels = axes.flat[0].get_legend_handles_labels()
# spare = list(axes.flat)[len(rows_to_plot):]
# for ax in spare:
#     ax.axis("off")
# if spare:  # legend in the first empty panel slot ...
#     spare[0].legend(handles, labels, loc="center", frameon=False, fontsize=10)
# else:  # ... or below the panels
#     fig.legend(handles, labels, loc="outside lower center", ncol=2, frameon=False)
# fig.suptitle("EPCAPE seasonal averages: sheet vs re-derived (mean ± 1 std; no whisker = no std given)",
#              x=0.01, ha="left", fontsize=12, color=INK)
# fig.savefig(OUT_DIR / "seasonal_averages_check.png", bbox_inches="tight")
# plt.show()

## 12. Findings to report back

*Written from the run recorded in this notebook (2026-10-08; QC = 0 throughout). Every number comes from the tables above. Percentages are (this check − sheet) / sheet, on the means.* **Section 13 collects every calculation, with its data, filters and thresholds, in one table, saved as `seasonal_averages_summary_table.xlsx`.** **Section 14 writes a copy of the sheet itself with this check's rows inserted in pink below Kavin's (`Table-EPCAPE_SeasonalAverages - with this check.xlsx`).**

**Kavin's own code and input files (Sections 1b, 2b, 3b, 9b, 9c, 10e)**

Re-running his live script on his files reproduces every value it can reach. The table lists what his calculation does that this check does not, and what the same data give once those quirks are corrected.

| Row | Reproduced from his files? | What his calculation does | Same data, corrected or compared |
|---|---|---|---|
| 7 SW transmittance | Summer 0.556 ± 0.280 and fall 0.657 ± 0.243, as in the sheet. The other seasons need Lubin's JAN, FEB, MAR and JUN files. | Averages Lubin's hourly ratios with **every daylight hour weighted equally**, low sun included. Drops day 31 and hour 0, averages in the day-1 −0.05 placeholders, and uses calendar-quarter seasons (winter = Jan–Mar). | Corrected cells: summer 0.563, fall 0.680. On 3,087 matched hours (r = 0.955 with Lubin's hours read as PST), RADFLUX's transmittance is a median **11 % above Lubin's**. On hours RADFLUX finds clear, Lubin's ratio is 0.94 at high sun and 0.77 at low sun, so **his clear-sky denominator is 7–30 % brighter than the clear sky** that was measured. That explains most of the gap; the rest is sampling (RADFLUX on Lubin's hours 0.637/0.818 vs this check's 0.666/0.839, summer/fall). |
| 8 LW↓ | Fall 323.0 ± 4.9, as in the sheet. The other seasons need the DJF, MAM and JJA files. | Mean of the **monthly median diurnal cycles**. The std is the **hour-to-hour spread of that cycle**, not day-to-day variability. Calendar quarters. | **Lubin's LW is the Pier SKYRAD record**: his monthly means agree with RADFLUX at the Pier within 0.4 W m⁻² in all six months. His statistic applied to RADFLUX gives 346/367/384/323/309 vs the sheet's 350/370/380/323/308 W m⁻² (all within 1.2 %). |
| 10 LWP | Campaign, spring, summer and fall to the sheet's precision; winter's mean (33) but not its std (121 vs 92). | Hourly **all-sky** means of QC = 0 MWRRET v2 LWP > 0, with no cloud or rain screen (the file is rebuilt exactly from the ARM files). MATLAB's array growth adds **1,142 zeros** (13.5 % of the values). The seasons that reproduce the sheet are **Mar–May, Jun–Aug, Sep–Nov and Dec–Feb**; the shared script's columns are May–Jul, Aug–Oct, Nov–Jan and Feb–Apr, and do not reproduce it. | Without the zeros, with the sheet's seasons: 64 ± 134 g m⁻² (+15 %); seasons 90/60/34/65. Without rain hours: 42 ± 61. This check, cloudy and rain-free: 75. |
| 27, 30 Activated fraction | Exactly: cloud 0.853/0.943/0.852/0.319; haze 0.579/1.000/0.557. | Share of the class's samples with `Dta` > `Dta_critical`, counted per sample, not per unit time. | Not an independent check: `Dta` and `Dta_critical` come from Abbey Williams' code, which wasn't shared. `Dta_critical` exists only in Jun–Sep and Dec 2023, so "spring" is June (941 samples), "fall" is December (138 samples) and haze spring is 19 samples. |
| 25–26, 28–29, 33–42, 54–55: Abbey Williams' 15-min file (`EPCAPE_15mindata.nc`; Kavin's `AMSforAndrew.mlx`) | Yes, to the sheet's rounding: hours 352.25/106.25/185.25/47/13.75 (cloud) and 236.25/51.75/100.5/49.5/34.5 (haze); every residual mean and std (median \|Δ\| ≤ 1 %); droplet N, D_eff and LWC (0–3 %). | Abbey's 15-min classes (`ClrCldHaz_class` 1 = cloud, 2 = haze, set only while the CVI is on). **Hours** = class intervals ÷ 4, whether or not the AMS reported, rounded. **Averages** = mean ± std over the class's intervals of: the 15-min `AMS_mass` (CE-corrected AMS, ÷ EF while the CVI is on); `FM120_Nd`; `FM120_LWC`; and D_eff of `FM120_DSD` with each bin's **upper edge** as its diameter. Calendar quarters. The shared script averages `AMS_massfrac` for spring, but the sheet's spring values are concentrations. **43 class intervals with no GCVI EF record (Mar 2023: 12, Apr 2023: 31) keep AMS values never divided by EF** (~5× too high). | Same classes, own samples: N and LWC are identical (the 15-min FM-120 values are the 5-min files averaged). D_eff with bin mid-points is 2–4 % lower, and the FM-120 ED column 7–22 % lower. Own AMS residual samples, which need an EF: spring cloud −25 to −29 % and winter haze −51 to −64 % (the intervals without EF); other seasons within 7 %. Hours with AMS data: cloud 254 h, haze 139 h; Kavin's count includes 28 % (cloud) and 41 % (haze) class intervals without AMS data. |
| 31–32 FM-120 hours | Exactly, once **truncated** to whole hours: 470 = 134 + 212 + 66 + 58; 418 = 85 + 172 + 49 + 112. | Visibility = mean of the raw samples from each row to the next row (65 windows span gaps), with no spike screening. Haze = 1–5 km with **no LWC condition**. Calendar quarters. | Same files with a screened 5-min mean: 472.5 / 420.2 h (≤ 0.2 h different in any season). The BAMS-paper text's haze definition (vis < 5 km and LWC < 0.01 g m⁻³) gives 395 h (−6 %), so the text and Fig. 5 disagree. |

**This check's own definitions** (independent data)

| Row | Quantity | Campaign: this check vs sheet | Remarks |
|---|---|---|---|
| 8 | LW↓ | 347 vs 350 W m⁻² (−1 %); seasons within 4 % | The same Pier instrument as Lubin's tables (Section 2b). The sheet's 3–10 W m⁻² stds are the spread of the median diurnal cycle; 1-min values spread by 28–41 W m⁻². |
| 10 | LWP, cloudy scenes | MWRRET v1, rain-free, every sample: 75 vs 56 g m⁻² (+34 %); seasons +6 to +60 % | MWRRET v2, rain-free: 75 (+33 %). Including rain: 103 (v1) and 93 (v2). Part of the gap is the sheet's padding zeros (−13.5 %). The rest is all-sky vs cloudy-only averaging: clear hours pull an all-sky mean down, and wet-radome hours push it up. |
| 15 | Cloud top | 902 vs 920 m (−2 %); summer −20 %, winter +24 % | KAZR-ARSCL lowest layer ≤ 3.5 km, every sample. The spread is 1.1–2.2× the sheet's. |
| 17 | PBL height | 804 vs 899 m (−11 %); seasons −1 to −19 % | Only when restricted to times with a low cloud overhead. All times: 656 m (−27 %; fall −55 %). The table should state the conditioning. |
| 21 | Cloud base | 633 vs 661 m (−4 %); summer −25 %, winter +11 % | Ceilometer ≤ 3.5 km, every sample. The spread is 1.3–2.1× the sheet's. |
| 22 | Rain, LD M1 | 3.4 vs 3.1 (+11 %) | Reproduced only as the **monthly mean of the daily accumulation on rain days, in mm day⁻¹, with std across months**. |
| 25, 26 | Cloud droplet N, effective diameter | N: 210 vs 210 cm⁻³ (seasons within 2 %). D_eff (FM-120 ED column): 12.7 vs 14 µm (−9 %) | FM-120 5-min rows with Abbey's 15-min cloud class (Section 10e). The sheet's diameter is D_eff of the 15-min mean distribution with bin upper edges; with mid-points it would be 13.2 µm. The older selections (Abbey's native-time classes, Kavin's visibility classes) are in Sections 9d–9e. |
| 28, 29 | Haze droplet N, effective diameter | N: 33 vs 33 cm⁻³ (seasons within 1 %; the sheet's fall std is 28, Kavin's code gives 37). D_eff (ED column): 9.4 vs 12 µm (−22 %) | Same, haze class. D_eff of the mean distribution with mid-points: 11.8 µm. For haze, the mean of the instrument's 1-s ED is much smaller than D_eff of the averaged distribution. |
| 54, 55 | Cloud / haze LWC | Cloud 0.16/0.12/0.17/0.12 and haze 0.0077/0.0090/0.0060/0.021 g m⁻³ (spring–winter): within 3 % in every season | FM-120 5-min rows with Abbey's 15-min classes (Section 10e); the sheet's campaign cell is empty. |
| 27, 30 | Activated fraction | 0.853 vs 0.85; 0.579 vs 0.58 | Kavin's arithmetic on Abbey Williams' file (Section 9c); see above. |
| 31–32 | FM-120 cloud / haze hours | BAMS-paper definitions: cloud 480 vs 470 h (+2 %); haze 395 vs 418 h (−6 %) | Kavin's code reproduces the sheet exactly (Section 9b). The FM-120 file reading reproduces the PI report's Table 1 (306.9 h). |
| 33–34 | GCVI-AMS cloud / haze hours | Intervals in which the AMS reported: cloud 254 vs 352 h (−28 %), haze 139 vs 236 h (−41 %) | Kavin's count of all class intervals reproduces the sheet exactly (Section 10e). This check counts only the intervals with AMS data, which are what the residual rows can use. |
| 35–38 | Cloud residual SO₄, NO₃, NH₄, Org | Own AMS samples with Abbey's 15-min classes: campaign −4 to −7 %; summer, fall and winter within 7 %; spring −25 to −29 % | Spring differs because Kavin's spring includes 31 April intervals (CVI on, no GCVI EF record) whose AMS values were never divided by EF. Without them his code gives the same as this check. |
| 39–42 | Haze residual SO₄, NO₃, NH₄, Org | Campaign −3 to −7 %; spring, summer and fall within 6 %; winter −51 to −64 % | Winter differs for the same reason: 12 March intervals without EF in Kavin's winter. |

**Systematic differences to ask about**

* **Row 7, SW transmittance (Dan Lubin, Kavin):** the cause is now located in the clear-sky denominator. On clear hours Lubin's transmittance is well below 1 (0.94 at high sun, 0.77 at low sun), so his Atwater & Ball model is brighter than the measured clear sky. Its inputs (aerosol, water vapour, surface pressure) are worth checking. Weighting every daylight hour equally also lowers the means, because low-sun hours have low ratios.
* **Row 10, LWP (Kavin):** the sheet includes 1,142 padding zeros and uses meteorological seasons. It is also an all-sky mean with rain hours. The definition the paper wants (all sky or cloudy only, rain screened or not) should be decided before the value is recomputed.
* **Row 20, LCL:** SONDEPARAM's surface-parcel LCL and an independent Romps (2017) calculation agree with each other but are 16–32 % below the sheet. The sheet presumably used a different starting level.
* **Row 23, rain at "LDS1":** no reading reproduces it; the record behind this row looks incomplete.
* **Row 24, rain, VDIS:** campaign −23 %, winter −42 %. The VDIS misses parts of several storms.
* **Rows 31–32:** the paper's text and the figure's numbers use different haze definitions (Section 9).
* **Rows 33–42 (Kavin, Abbey):** 43 CVI-on intervals (Mar 2023: 12, Apr 2023: 31) have AMS data but no GCVI EF, so their AMS_mass was not divided by EF. They raise the spring cloud residuals by ~30 % and the winter haze residuals by 2–3×. Should they be dropped? Two further questions:
  * Should the sampling hours count the class intervals without AMS data (28 % of cloud, 41 % of haze)?
  * Should spring be concentrations (as in the sheet) or mass fractions (as in the shared script)?
* **Rows 26, 29 (Kavin):** the effective diameter uses each bin's upper edge as its diameter. Bin mid-points lower it by 0.3–0.5 µm.
* **Rows 27 and 30:** `Dta_critical` exists only in Jun–Sep and Dec 2023, so their spring and fall values are June and December alone.

**`AMS_ambient.nc`** (Section 10d) is not where the cloud and haze residual values come from. It is the AMS record with the GCVI periods taken out: 5,493 of the 5,502 GCVI-AMS samples are missing from it. It is also not CE-corrected: its values are the CE-corrected file's times one constant per month (0.4–0.9), the collection efficiency. Abbey's classes exist only while the GCVI sampled, so no ambient sample falls in them. Ambient samples in Kavin's visibility cloud and haze rows miss the sheet by a median of 24–436 %. The residual rows come from the CE-corrected AMS samples behind the GCVI, divided by EF.

**Not checkable yet:** rows 18 (PBLH-MPL) and 19 (BL-Depth, Miller), which have no identified source. Every other row attributed to Kavin is now re-derived.

**Corrections to the sheet** (Section 0 and above):
* H34 "5.0×10²" is a typo for 5.0×10¹: Kavin's code gives 49.5 h. Rows 33–34 are rounded, not truncated.
* Rows 35–42 are mass concentrations (µg m⁻³), not mass fractions.
* B7's unit is a fraction, not %.
* B25/B28 should be cm⁻³.
* Rows 7–8 are **Pier** data, Lubin's included, and their seasons are calendar quarters (winter = Jan–Mar).
* Row 10's seasons are MAM/JJA/SON/DJF, and its values include padding zeros.
* Rows 22–24 are mm day⁻¹ (monthly means), not mm/hr.
* Rows 31–32 are truncated, not rounded.

## 13. Summary table: Kavin's calculations next to this check

One table for every row checked, built from the comparisons above and saved as **`seasonal_averages_summary_table.xlsx`** in `OUT_DIR`. Each sheet row has up to four kinds of line:

| Calculation | What it is |
|---|---|
| **Kavin (sheet value)** | the value in the sheet; *Notes* give his data, filters and thresholds where his code or the sheet states them |
| **Kavin's method, re-run here on his files** | his MATLAB code re-run in Python on his input files (rows 7, 8, 10, 27, 30, 31, 32) |
| **This check: same data as Kavin** | this check's calculation on the data he used (or, where marked, the data he most likely used) |
| **This check: other data** | this check on other data: the ARM products, where his data aren't known or as an independent check |

*Notes* lists the data, filters and thresholds of every line. *Primary* marks the line carried into the Section 11 summary. Δ columns are (this check − sheet) / sheet on the means.

The workbook's other sheets:
* *Numbers*: the same lines in long form (mean, std, n per season);
* *About*: the notebook, git revision and settings that produced it.

In [31]:
# =============================================================================
# Section 13: summary table -> OUT_DIR/seasonal_averages_summary_table.xlsx
# =============================================================================
# Inputs
#   summary (Section 11a)  every comparison above: one row per (definition, season), with ours, ours_mean,
#                          ours_std, n, table, diff_pct, primary
#   TABLE, FIG5            the sheet's values (rows attributed to Kavin; rows 31-32 attributed to "Fig. 5")
#   settings               the Settings cell, quoted in the notes so the table matches the run
# Each line names its calculation by a phrase of the definition printed in the sections above;
# pick() raises if the phrase matches no definition, or more than one, so a renamed definition
# cannot silently drop out of the table.
KAVIN = "Kavin (sheet value)"
RERUN = "Kavin's method, re-run here on his files"
SAME = "This check: same data as Kavin"
OTHER = "This check: other data"
UNITS = {7: "fraction (sheet: %)", 8: "W m⁻²", 10: "g m⁻²", 15: "m", 17: "m", 18: "m", 19: "m", 20: "m", 21: "m",
         22: "mm day⁻¹ (sheet: mm/hr)", 23: "mm day⁻¹ (sheet: mm/hr)", 24: "mm day⁻¹ (sheet: mm/hr)",
         25: "cm⁻³ (sheet: m-3)", 26: "µm", 27: "fraction", 28: "cm⁻³ (sheet: m-3)", 29: "µm", 30: "fraction",
         31: "h", 32: "h", 33: "h", 34: "h", **{r: "µg m⁻³ (sheet: 'mass fraction')" for r in range(35, 43)},
         54: "g m⁻³", 55: "g m⁻³"}
LINES = []  # (sheet row, calculation, data, (phrase, primary) or None for the sheet value, notes)


def kavin(row, data, notes):
    LINES.append((row, KAVIN, data, None, notes))


def ours(row, calculation, data, notes, phrase=None, primary=False):
    LINES.append((row, calculation, data, (phrase, primary), notes))


def pick(row, phrase=None, primary=False):
    """Comparison table (rows = seasons) of the ONE definition for `row` matching `phrase` and/or primary."""
    sub = summary[summary["sheet_row"] == str(row)]
    if primary:
        sub = sub[sub["primary"]]
    if phrase is not None:
        sub = sub[sub["definition"].str.contains(phrase, regex=False)]
    found = sub["definition"].unique()
    if len(found) != 1:
        raise ValueError(f"row {row}: {len(found)} definitions match {phrase!r} (primary={primary}): {list(found)[:3]}")
    return sub.set_index("season").loc[S.SEASON_ORDER]


def sheet_says(row):
    """The sheet's own location and method text for a row."""
    return f"Sheet: {TABLE.loc[row, 'location']}; {TABLE.loc[row, 'method'] or 'no method given'}"


win = Q.describe_window  # "5-min mean", "every QC = 0 sample (no further averaging)", ...

# --- Row 7: SW transmittance ------------------------------------------------------------------------
kavin(7, "Dan Lubin's hourly SW transmittance tables, <MON>_SW_Boxplot.csv (hours in PST)",
      "Sheet note: ratio of 5-min SKYRAD (radflux1long) SW↓ to the Atwater & Ball (1981) clear-sky model. Kavin's code: "
      "mean ± std over every (day, hour) cell of the season's three months; days 1–30 only (day 31 dropped), hours 1–23 "
      "(hour 0 dropped), day-1 −0.05 placeholders included; seasons = calendar quarters (winter = Jan–Mar). SKYRAD is at "
      "the Pier (M1), not Soledad.")
ours(7, RERUN, "Same Lubin tables", "Kavin's code line by line (kavin.kavin_sw_stats). Lubin's JAN, FEB, MAR and JUN "
     "files are not in hand, so only summer and fall can be reproduced.", "Kavin's code: days 1–30")
ours(7, SAME, "Same Lubin tables", "Every day and every hour; −0.05 placeholders removed; calendar quarters. Seasons "
     "with a missing month are blank.", "every day and hour, placeholders removed")
ours(7, OTHER, "RADFLUX epcradflux1longM1.c2 (Scripps Pier)",
     f"downwelling_shortwave / clearsky_downwelling_shortwave (Long & Ackerman 2000 empirical clear-sky fit), both "
     f"averaged over the same samples; QC = 0 on SW; {win(AVG_WINDOW_SW_MIN)}; SZA ≤ {SZA_MAX_DEG:g}°; sheet seasons. "
     f"On {len(matched):,} matched hours RADFLUX / Lubin transmittance = {sw_ratio.median():.2f} (median): Lubin's "
     "clear-sky denominator is brighter than the measured clear sky.", primary=True)

# --- Row 8: LW down -----------------------------------------------------------------------------------
kavin(8, "Dan Lubin's monthly LW↓ hourly-median tables, DJF/MAM/JJA/SON_LW_Stepplot.csv (PST hours)",
      "Season = mean over the 24 hours of the mean of its three months' median diurnal cycles; std = spread of that "
      "24-hour cycle (not day-to-day variability); campaign = mean and std of all 12 × 24 medians; calendar quarters. "
      "Lubin's LW is the Pier SKYRAD record (monthly means within 0.4 W m⁻² of RADFLUX), not Soledad.")
ours(8, RERUN, "Same Lubin tables (per-month files stand in for the 3-month files)",
     "Kavin's code line by line (kavin.kavin_lw_stats). JAN, FEB, MAR, MAY, JUN, JUL medians are not in hand, so only "
     "fall can be reproduced.", "Lubin's monthly hourly medians, Kavin's code")
ours(8, SAME, "RADFLUX at the Pier, the record Lubin's tables are made from",
     "Kavin's statistic on RADFLUX downwelling_longwave (QC = 0): hourly means in PST, each month's median of each "
     "hour, all twelve months; calendar quarters.", "Kavin's statistic")
ours(8, OTHER, "RADFLUX epcradflux1longM1.c2 (Scripps Pier)",
     f"downwelling_longwave, QC = 0, {win(AVG_WINDOW_LW_MIN)}, day and night; std = spread of the values; sheet "
     "seasons.", primary=True)

# --- Row 10: LWP ------------------------------------------------------------------------------------------
kavin(10, "epcmwrret_turnM1_hourly_averages.nc: hourly means of MWRRET v2 phys_lwp",
      "File: each hour = mean of the QC = 0 samples with LWP > 0; no cloud or rain screen (rebuilt exactly from the ARM "
      "files). Code: hours into a nan(500,13) array by Pacific-time month; MATLAB's growth adds 1,142 zeros that enter the "
      "means. The sheet's seasons are Mar–May, Jun–Aug, Sep–Nov, Dec–Feb, not its column headers; its winter std (92) is "
      "not reproduced.")
ours(10, RERUN, "Same hourly file", "Kavin's array with the zero padding; columns 2:4, 5:7, 8:10, 11:13.",
     "columns 2:4, 5:7")
ours(10, RERUN + " (script as shared)", "Same hourly file",
     "Columns 4:6, 7:9, 10:12, 1:3 as in the shared script = May–Jul, Aug–Oct, Nov–Jan, Feb–Apr (column j holds month "
     "j+1). Does not reproduce the sheet.", "the script as shared")
ours(10, SAME, "Same hourly file", "Padding zeros removed; sheet seasons (UTC).",
     "Kavin's hourly file, no padding zeros; sheet seasons")
ours(10, SAME + ", rain-free", "Same hourly file + Pier Parsivel (LD M1, QC = 0)",
     f"As above, without hours within ±{LWP_RAIN_WINDOW_MIN:g} min of rain at the Pier.", "rain-free hours")
ours(10, OTHER, "MWRRET v1 epcmwrret1liljclouM1.c2 (Pier)",
     f"phys_lwp, QC = 0; cloudy samples only (qc_cloud_base_height = 0); rain-free (no LD M1 rain within "
     f"±{LWP_RAIN_WINDOW_MIN:g} min); averaging window {AVG_PERIOD_LWP:g} min (0 = every sample); sheet seasons.",
     primary=True)
ours(10, OTHER + " (MWRRET v2)", "MWRRET v2 epcmwrret2turnM1.c1 (Pier)",
     f"phys_lwp, QC = 0 and phys_qc_flag = 0; cloudy (cbh_detected > 0, detection_status 1–3); rain-free "
     f"(±{LWP_RAIN_WINDOW_MIN:g} min); averaging window {AVG_PERIOD_LWP:g} min.",
     "MWRRET v2 (3-channel MWR, mwrret2turn.c1): phys_lwp, cloudy scenes, rain-free")

# --- Rows 15-24: no code from Kavin; this check on ARM data ------------------------------------------------
for row in (15, 17, 18, 19, 20, 21, 22, 23, 24):
    kavin(row, sheet_says(row), "Kavin's data, filters and thresholds are not documented (no code shared).")
ours(15, OTHER, "KAZR-ARSCL epcarsclkazr1kolliasM1.c1 (Pier)",
     f"Top of the lowest cloud layer ≤ {LOW_CLOUD_MAX_M:g} m AGL; cloud_source_flag codes "
     f"{CT_CLOUD_SOURCES if CT_CLOUD_SOURCES else 'any'}; averaging window {AVG_PERIOD_CT:g} min (0 = every profile); "
     "sheet seasons.", primary=True)
ours(17, OTHER, "PBLHTTHERMO epcpblhtthermoM1.c1 (Pier) + KAZR-ARSCL",
     f"pbl_height (10-min values, no QC field) only when KAZR-ARSCL sees a cloud top ≤ {LOW_CLOUD_MAX_M:g} m within "
     "±5 min; best match among six readings.", primary=True)
ours(17, OTHER + " (all times)", "PBLHTTHERMO epcpblhtthermoM1.c1 (Pier)", "pbl_height at all times.",
     "PBLHTTHERMO pbl_height, all times")
ours(20, OTHER, "SONDEPARAM epcsondeparamM1.c1 (radiosondes, Pier)",
     "Surface-parcel LCL (parcel type 1), data_quality = 0; one value per launch.", primary=True)
ours(20, OTHER + " (Romps 2017)", "PBLHTSONDE profiles (radiosondes, Pier)",
     "Exact LCL of Romps (2017) from p, T, RH of the lowest ~2 hPa of each sonde; m above the launch level.",
     "Romps (2017)")
ours(21, OTHER, "Ceilometer epcceilM1.b1 (Pier)",
     f"first_cbh with qc_first_cbh = 0, detection_status 1–3, base ≤ {LOW_CLOUD_MAX_M:g} m; averaging window "
     f"{AVG_PERIOD_CBH:g} min (0 = every 16-s sample); sheet seasons.", primary=True)
for row, data in ((22, "Parsivel LD epcldM1.b1 (Pier)"), (23, "Parsivel LD epcldS2.b1 (Mt. Soledad; the sheet's 'LDS1')"),
                  (24, "VDISQUANTS epcvdisquantsM1.c1 (Pier)")):
    ours(row, OTHER, data,
         f"Best of seven readings: monthly mean of the daily accumulation on rain days (mm day⁻¹), std across months; "
         f"raining = rate ≥ {RAIN_THRESHOLD_MM_H:g} mm h⁻¹; rain day ≥ {RAIN_MIN_MM:g} mm; QC = 0 where a qc_ field exists.",
         primary=True)

# --- Rows 25-29, 54-55: FM-120 droplet number, diameter, LWC (Abbey's 15-min file) --------------------------------
FM15 = ("Abbey Williams' 15-min file EPCAPE_15mindata.nc (via Kavin's AMSforAndrew.mlx): ClrCldHaz_class = 1 (cloud) / 2 "
        "(haze), set only while the CVI is on; mean ± std over the class's 15-min intervals; calendar quarters (UTC)")
for row, kavin_var in ((25, "FM120_Nd"), (26, "D_eff = Σ N D³ / Σ N D² of FM120_DSD with each bin's upper edge as its diameter"),
                       (28, "FM120_Nd"), (29, "D_eff of FM120_DSD with bin upper edges"), (54, "FM120_LWC"), (55, "FM120_LWC")):
    kavin(row, "EPCAPE_15mindata.nc (Abbey Williams' 15-min merged file)", f"{FM15}. Variable: {kavin_var}. Reproduced.")
    ours(row, RERUN, "Same 15-min file", f"{kavin_var}; Kavin's averaging.", "Kavin's method: Abbey's 15-min file")
    if row in (26, 29):
        ours(row, SAME + " (bin mid-points)", "Same 15-min file",
             "D_eff of FM120_DSD with the bin mid-points as diameters.", "bin mid-points")
    ours(row, SAME, "FM-120 daily 5-min files + Abbey's 15-min classes",
         "FM-120 5-min rows, each taking ClrCldHaz_class of the 15-min interval holding it; nd_cm3 / ed_um (NaN when N = 0) / "
         "lwc_gm3; mean ± std over rows; sheet seasons.", primary=True)
for row in (25, 26, 28, 29):
    ours(row, SAME + " (Abbey's native-time classes)", "FM-120 daily 5-min files + ET_Dcrit_nativetime.csv classes",
         "FM-120 rows in which > 50 % of Abbey's native-time classified samples are cloud (haze).",
         "ClrCldHaz_class (most native samples)")
    ours(row, SAME + " (Kavin's visibility classes)", "FM-120 daily 5-min files + msd_visibility_*.csv",
         "Rows classed by Kavin's hours code: cloud = row visibility < 1 km and LWC > 0.01 g m⁻³; haze = 1–5 km.",
         "Kavin's visibility/LWC definitions")
for row in (54, 55):
    ours(row, SAME + " (Kavin's visibility classes)", "FM-120 daily 5-min files + msd_visibility_*.csv",
         "lwc_gm3 over rows classed by Kavin's hours code (cloud: < 1 km and LWC > 0.01 g m⁻³; haze: 1–5 km).",
         "Kavin's visibility/LWC definitions")
    ours(row, SAME + " (BAMS-paper definitions)", "FM-120 daily 5-min files + msd_visibility_*.csv (screened)",
         "lwc_gm3 over rows classed by the BAMS-paper definitions of Section 9.", "BAMS-paper definitions")


# --- Rows 27, 30: activated fraction -----------------------------------------------------------------------
for row, kind in ((27, "cloud"), (30, "haze")):
    kavin(row, "ET_Dcrit_nativetime.csv (Abbey Williams' code, via Kavin)",
          f"Share of {kind}-class samples (ClrCldHaz_class {1 if kind == 'cloud' else 2}) with Dta > Dta_critical, both "
          "present; counted per sample. Kavin's code gives the campaign value only. Dta_critical exists only in Jun–Sep and "
          "Dec 2023, so spring = June, fall = December, winter has none.")
    ours(row, RERUN, "Same file", "Kavin's count, applied within each season (sheet seasons, UTC).", primary=True)
    ours(row, SAME, "Same file", "As above, with the file's fully repeated rows counted once.",
         "fully repeated rows counted once")

# --- Rows 31-32: FM-120 cloud and haze hours ---------------------------------------------------------------
for row, kind, rule in ((31, "cloud", "avgvis < 1 km and LWC > 0.01 g m⁻³"),
                        (32, "haze", "1 km < avgvis < 5 km, no LWC condition")):
    kavin(row, "FM-120 5-min files + msd_visibility_*.csv (Kavin's 'Hours Calculations'; sheet author 'Fig. 5')",
          f"avgvis = mean of the raw ~1-s visibility from each FM-120 row to the next row (both ends included; spans "
          f"gaps; no spike screening); {kind} = {rule}; calendar quarters (UTC); hours = rows / 12, truncated to whole hours; "
          "campaign = sum of the truncated seasons.")
    ours(row, RERUN, "Same files", "Reproduces every season exactly once truncated to whole hours.", "Kavin's code:")
    ours(row, SAME, "Same files",
         f"BAMS-paper definitions: cloud = {FM120_VIS_STAT} visibility < {FM120_CLOUD_VIS_MAX_M / 1e3:g} km and LWC > "
         f"{FM120_CLOUD_LWC_MIN_GM3:g} g m⁻³; haze = visibility < {FM120_HAZE_VIS_MAX_M / 1e3:g} km and LWC < "
         f"{FM120_HAZE_LWC_MAX_GM3:g} g m⁻³; visibility screened (0 < vis ≤ 48,270 m), ≥ {FM120_VIS_MIN_SAMPLES} "
         "samples per 5-min interval; sheet seasons.", primary=True)
for row in (31, 32):
    ours(row, SAME + " (filter set 2)", "Same files",
         "Andrew's filter set 2: cloud unchanged; haze = (1 km < visibility < "
         f"{FM120_HAZE2_VIS_MAX_M / 1e3:g} km and LWC < {FM120_HAZE2_LWC_MAX_GM3:g} g m⁻³) or (visibility < 1 km and LWC < "
         f"{FM120_HAZE2_LWC_MAX_GM3:g} g m⁻³); {FM120_VIS_STAT} screened visibility; sheet seasons.", "filter set 2")
ours(31, SAME + " (Kavin's thresholds)", "Same files",
     "Kavin's cloud thresholds with the screened 5-min mean visibility (the other statistic of Section 9).",
     "mean visibility < 1 km and LWC > 0.01")
ours(32, SAME + " (Kavin's thresholds)", "Same files",
     "1 km < screened 5-min mean visibility < 5 km, no LWC condition.", "best match to the sheet")

# --- Rows 33-42: GCVI-AMS hours and residuals (Kavin's AMSforAndrew.mlx, Abbey's 15-min file) ---------------------
for row in range(33, 43):
    kind = "cloud" if row in (33, 35, 36, 37, 38) else "haze"
    code_ = 1 if kind == "cloud" else 2
    if row in (33, 34):
        kavin(row, "EPCAPE_15mindata.nc (Abbey Williams' 15-min file; Kavin's AMSforAndrew.mlx)",
              f"Hours = number of 15-min intervals with ClrCldHaz_class = {code_} (set only while the CVI is on) / 4, whether "
              "or not the AMS reported; calendar quarters (UTC); values rounded. Reproduced."
              + (" The Fall cell '5.0*10^2' is a typo for 5.0*10^1: his code gives 49.5 h." if row == 34 else ""))
        ours(row, RERUN, "Same 15-min file", "Kavin's count, not rounded.", "Kavin's AMS code: Abbey's 15-min")
        ours(row, SAME, "Abbey's 15-min classes + the AMS",
             f"15-min intervals with ClrCldHaz_class = {code_} in which the AMS reported, × 15 min; sheet seasons.",
             primary=True)
        ours(row, SAME + " (Abbey's native-time classes)", "GCVI EF + AMS + ET_Dcrit_nativetime.csv classes on FM-120 rows",
             f"FM-120 rows of the class inside GCVI segments (EF > 1, ≥ {GCVI_MIN_SEGMENT_MIN:g} min) holding AMS data, × 5 min.",
             f"GCVI-AMS {kind} hours: Abbey's classes, FM-120 rows")
    else:
        kavin(row, "EPCAPE_15mindata.nc AMS_mass = AMS_mass_15min_EF (rows: nitrate, sulfate, organics, ammonium, chloride)",
              f"Mean ± std of the 15-min AMS_mass over the intervals with ClrCldHaz_class = {code_}; calendar quarters (UTC). "
              f"AMS_mass = CE-corrected AMS averaged over 15 min, ÷ EF while the CVI is on, except {NO_EF_TEXT}. The "
              "script as shared averages AMS_massfrac for spring; the sheet's spring values are concentrations. Values "
              "are concentrations (µg m⁻³), not mass fractions. Reproduced.")
        ours(row, RERUN, "Same 15-min file", "Kavin's averaging of AMS_mass.", "Kavin's AMS code: mean of Abbey's 15-min")
        ours(row, RERUN + " (script as shared)", "Same 15-min file", "Spring from AMS_massfrac (mass fraction).",
             "Kavin's AMS code as shared")
        ours(row, SAME, "AMS_CE_corrected_msptof.nc + CVI_EF.csv + Abbey's 15-min classes",
             f"Own AMS samples: CE-corrected sample whose 2-min V-mode period lies in a GCVI segment (EF > 1, ≥ "
             f"{GCVI_MIN_SEGMENT_MIN:g} min), ÷ the mean EF over the period; class = 15-min interval holding the sample; "
             "sheet seasons.", primary=True)
        ours(row, SAME + " (Abbey's native-time classes)", "Same + ET_Dcrit_nativetime.csv classes on FM-120 rows",
             "Class = FM-120 row holding the V-mode mid-point, Abbey's native-time classes.",
             f"/ EF, {kind} samples: Abbey's classes, FM-120 rows")
        ours(row, OTHER + " (AMS_ambient.nc)", "AMS_ambient.nc (ambient inlet only, not CE-corrected)",
             f"Ambient samples in Kavin's visibility {kind} rows: tests whether the rows come from this file (they do not).",
             "AMS_ambient.nc as stored")

# --- Build the table ------------------------------------------------------------------------------------------
out_rows, numbers = [], []
for row, calculation, data, source, notes in LINES:
    quantity = (FIG5 if row in (31, 32) else TABLE).loc[row, "quantity"].strip()
    line = {"Sheet row": row, "Quantity": quantity, "Units": UNITS.get(row, ""), "Calculation": calculation, "Data": data}
    if source is None:  # the sheet's value
        ref = sheet_hours(row) if row in (31, 32) else S.table_values(TABLE, row)
        raw = {s: ref.loc[s, "raw"] for s in S.SEASON_ORDER}  # the cell as typed (text, or a number)
        line.update({s: ("" if pd.isna(ref.loc[s, "mean"]) else f"{v:g}" if isinstance(v, (int, float)) else str(v).strip())
                     for s, v in raw.items()})
        line.update({"Δ campaign (%)": np.nan, "median |Δ| over seasons (%)": np.nan, "Primary": ""})
        num = ref[["mean", "std"]].assign(n=np.nan)
    else:
        cmp = pick(row, *source)
        line.update({s: cmp.loc[s, "ours"] for s in S.SEASON_ORDER})
        line.update({"Δ campaign (%)": round(cmp.loc["EPCAPE", "diff_pct"], 1),
                     "median |Δ| over seasons (%)": round(cmp["diff_pct"].abs().median(), 1),
                     "Primary": "yes" if bool(cmp["primary"].iloc[0]) else ""})
        num = pd.DataFrame({"mean": cmp["ours_mean"], "std": cmp["ours_std"], "n": cmp["n"]})
    line["Notes"] = notes
    out_rows.append(line)
    numbers.append(num.reset_index(names="season").assign(**{"Sheet row": row, "Calculation": calculation}))
# one block per sheet row, in sheet order; within a row the lines keep the order they were listed in
SUMMARY_TABLE = pd.DataFrame(out_rows).sort_values("Sheet row", kind="stable").reset_index(drop=True)
NUMBERS_TABLE = (pd.concat(numbers, ignore_index=True)[["Sheet row", "Calculation", "season", "mean", "std", "n"]]
                 .sort_values("Sheet row", kind="stable").reset_index(drop=True))
print(f"{len(SUMMARY_TABLE)} lines for {SUMMARY_TABLE['Sheet row'].nunique()} sheet rows")
display(SUMMARY_TABLE.drop(columns=["Notes", "Data"]))

# --- Save as .xlsx: Summary, Numbers, About ---------------------------------------------------------------------
from openpyxl.styles import Alignment, Font  # openpyxl also reads the sheet (seasonal.py)

xlsx_path = OUT_DIR / "seasonal_averages_summary_table.xlsx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
about = pd.DataFrame({"item": ["made by", "git revision", "sheet read", *SETTINGS.keys()],
                      "value": ["comparisons/seasonal_averages/check_seasonal_averages.ipynb, Section 13", GIT_REVISION,
                                str(TABLE_XLSX), *[str(v) for v in SETTINGS.values()]]})
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    SUMMARY_TABLE.to_excel(xw, sheet_name="Summary", index=False)
    NUMBERS_TABLE.to_excel(xw, sheet_name="Numbers", index=False)
    about.to_excel(xw, sheet_name="About", index=False)
    ws = xw.sheets["Summary"]
    widths = {"Sheet row": 8, "Quantity": 30, "Units": 18, "Calculation": 34, "Data": 40, "Notes": 90,
              **{s: 16 for s in S.SEASON_ORDER}}
    for j, name in enumerate(SUMMARY_TABLE.columns, start=1):
        letter = ws.cell(row=1, column=j).column_letter
        ws.column_dimensions[letter].width = widths.get(name, 12)
        for i in range(1, len(SUMMARY_TABLE) + 2):  # wrap every cell, top-aligned
            ws.cell(row=i, column=j).alignment = Alignment(wrap_text=True, vertical="top")
    for i, calc in enumerate(SUMMARY_TABLE["Calculation"], start=2):  # the sheet's values in bold
        if calc == KAVIN:
            for j in range(1, len(SUMMARY_TABLE.columns) + 1):
                ws.cell(row=i, column=j).font = Font(bold=True)
    ws.freeze_panes = "F2"
print(f"Wrote {xlsx_path}")

137 lines for 32 sheet rows

,Sheet row,Quantity,Units,Calculation,EPCAPE,Spring,Summer,Fall,Winter,Δ campaign (%),median |Δ| over seasons (%),Primary
0,7,Shortwave (SW) Transmittance,fraction (sheet: %),Kavin (sheet value),0.58 ± 0.29,0.50 ± 0.31,0.56 ± 0.28,0.66 ± 0.24,0.61 ± 0.28,NaN,NaN,
1,7,Shortwave (SW) Transmittance,fraction (sheet: %),"Kavin's method, re-run here on his files",–,–,0.56 ± 0.28,0.66 ± 0.24,–,NaN,0.5,
2,7,Shortwave (SW) Transmittance,fraction (sheet: %),This check: same data as Kavin,–,–,0.56 ± 0.27,0.68 ± 0.21,–,NaN,1.8,
3,7,Shortwave (SW) Transmittance,fraction (sheet: %),This check: other data,0.71 ± 0.32,0.61 ± 0.34,0.67 ± 0.31,0.84 ± 0.25,0.76 ± 0.32,21.9,22.3,yes
4,8,Longwave (LW) Flux,W m⁻²,Kavin (sheet value),350 ± 30,370 ± 10,380 ± 10,323 ± 5,308 ± 3,NaN,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...
132,55,Haze Liquid Water Content,g m⁻³,Kavin (sheet value),,0.0077 ± 0.0081,0.0089 ± 0.0091,0.0062 ± 0.0145,0.021 ± 0.033,NaN,NaN,
133,55,Haze Liquid Water Content,g m⁻³,"Kavin's method, re-run here on his files",0.009 ± 0.013,0.0077 ± 0.0081,0.0089 ± 0.0091,0.006 ± 0.015,0.021 ± 0.033,NaN,0.7,
134,55,Haze Liquid Water Content,g m⁻³,This check: same data as Kavin,0.009 ± 0.015,0.0077 ± 0.0094,0.009 ± 0.011,0.006 ± 0.018,0.021 ± 0.036,NaN,0.7,yes
135,55,Haze Liquid Water Content,g m⁻³,This check: same data as Kavin (Kavin's visibility classes),0.011 ± 0.030,0.0049 ± 0.0061,0.008 ± 0.015,0.006 ± 0.026,0.021 ± 0.050,NaN,9.0,


Wrote /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/processed/derived/seasonal_averages_check/seasonal_averages_summary_table.xlsx


## 14. The sheet itself, with this check's rows added (pink)

A copy of `Table-EPCAPE_SeasonalAverages.xlsx` in the layout Andrew uses in his own version:
* Directly below each of Kavin's rows (and the two "Fig. 5" rows his code produced), one or more new rows hold this check's values. They are filled light pink across columns A–N, and their author is "Andrew Buggee".
* **Kavin's rows are not changed**, except for column K (*Notes*). There, his method, data, filters and thresholds are added after any existing note.
* Every pink row's *Notes* give its data product, filters and thresholds.
* Every other row, and the other two sheets, are kept as they are. The `=SUM(...)` formulas of the "Fig. 5" rows are updated to their new row numbers.

Saved as **`Table-EPCAPE_SeasonalAverages - with this check.xlsx`** in `OUT_DIR`. The cell checks that every original row is unchanged apart from the notes column.
The two threaded comments (A1, E35) are kept as ordinary notes: openpyxl, which writes the file, keeps the comment text but not the threaded format.

In [32]:
# =============================================================================
# Section 14: copy of the sheet with this check's rows inserted below Kavin's (pink)
# =============================================================================
# Inputs
#   TABLE_XLSX           the sheet (read only; the copy is written to OUT_DIR)
#   pick, win (Sec. 13)  pick(row, phrase, primary) -> one definition's comparison table from `summary`
#   settings             quoted in the notes, so the notes match this run
# Layout copied from Andrew's own version ("Table-EPCAPE_SeasonalAverages - AJB.xlsx"): new rows right
# below the row they check, author "Andrew Buggee", columns A-N filled with theme colour 5 (accent 2,
# #EA4335 in this workbook's theme) lightened 60 % (tint 0.6), notes in column K under a "Notes" header.
from copy import copy

from openpyxl import load_workbook
from openpyxl.formula.translate import Translator
from openpyxl.styles import PatternFill
from openpyxl.styles.colors import Color

PINK = PatternFill(fill_type="solid", fgColor=Color(theme=5, tint=0.5999938962981048))
FILL_COLUMNS = 14  # A-N, as in Andrew's version
AUTHOR = "Andrew Buggee"
NOTES_COL = 11  # column K
MY_ROWS = {}  # sheet row -> list of (quantity, units, location, product, source, notes); source = (phrase, primary) or None
KAVIN_NOTES = {}  # sheet row -> note on Kavin's method, data, filters and thresholds
SET2_NOTES = {}  # sheet row (31, 32) -> notes of the filter-set-2 rows (also used in Section 15)


def add(row, quantity, units, location, product, source, notes):
    MY_ROWS.setdefault(row, []).append((quantity, units, location, product, source, notes))


def q(row):
    """The quantity text of the sheet row (kept as typed)."""
    return (FIG5 if row in (31, 32) else TABLE).loc[row, "quantity"]


NOT_SHARED = "Kavin's method: not documented (no code shared for this row)."
# --- Row 7: SW transmittance ------------------------------------------------------------------
KAVIN_NOTES[7] = ("Kavin's method (his live script ForAndrewVariablesmlx.mlx, 'Dan LW SW Code'): Dan Lubin's hourly SW "
                  "transmittance tables <MON>_SW_Boxplot.csv (one value per day and hour, hours in PST; SKYRAD is at the Pier, "
                  "not Soledad). Mean ± std over every (day, hour) cell of the season's 3 months; days 1-30 only (day 31 dropped), "
                  "hours 1-23 only (hour 0 dropped), the day-1 -0.05 placeholders included; seasons = calendar quarters (winter = "
                  "Jan-Mar). Re-running his code reproduces summer and fall exactly (the other months' tables were not provided).")
add(7, q(7), "fraction", "Pier", "Lubin's hourly SW transmittance tables (Kavin's files)",
    ("every day and hour, placeholders removed", False),
    "Same data as Kavin (Lubin's <MON>_SW_Boxplot.csv). filters: every day and hour of each month; -0.05 placeholders "
    "removed; mean ± std over all (day, hour) cells; calendar quarters (winter = Jan-Mar). JAN, FEB, MAR and JUN tables "
    "not provided, so EPCAPE, Spring and Winter are n/a.")
add(7, q(7), "fraction", "Pier", "epcradflux1longM1.c2 (RADFLUX)", (None, True),
    f"filters: QC = 0 on downwelling_shortwave; transmittance = downwelling_shortwave / clearsky_downwelling_shortwave "
    f"(RADFLUX's empirical clear-sky fit, Long & Ackerman 2000, not Atwater & Ball), both averaged over the same samples; "
    f"{win(AVG_WINDOW_SW_MIN)}; solar zenith angle <= {SZA_MAX_DEG:g} deg; sheet seasons (UTC). On {len(matched):,} hours "
    f"matched with Lubin's tables, RADFLUX / Lubin = {sw_ratio.median():.2f} (median): Lubin's clear-sky model is "
    "brighter than the measured clear sky.")
# --- Row 8: LW down ----------------------------------------------------------------------------
KAVIN_NOTES[8] = ("Kavin's method ('Dan LW SW Code'): Dan Lubin's monthly LW tables DJF/MAM/JJA/SON_LW_Stepplot.csv = the "
                  "median across days of each hour's LW (hours in PST). Season = mean over the 24 hours of the mean of its 3 "
                  "months' median diurnal cycles; ± = std of that 24-hour cycle (the diurnal spread, not day-to-day "
                  "variability); EPCAPE = mean ± std of all 12 x 24 medians; calendar quarters. Lubin's LW is the Pier SKYRAD "
                  "record (monthly means within 0.4 W/m2 of RADFLUX at the Pier), not Soledad. Re-running his code reproduces "
                  "fall exactly (the DJF, MAM and JJA tables were not provided).")
add(8, q(8), "W/m2", "Pier", "epcradflux1longM1.c2 (RADFLUX; the record Lubin's tables are made from)",
    ("Kavin's statistic", False),
    "Kavin's method on the full record. filters: QC = 0 on downwelling_longwave; hourly means (PST); each month's median "
    "of each hour; season = mean of its 3 months' median diurnal cycles over 24 hours, ± = spread of that cycle; "
    "calendar quarters.")
add(8, q(8), "W/m2", "Pier", "epcradflux1longM1.c2 (RADFLUX)", (None, True),
    f"filters: QC = 0 on downwelling_longwave; {win(AVG_WINDOW_LW_MIN)}; day and night; ± = std of all values; sheet "
    "seasons (UTC).")
# --- Row 10: LWP --------------------------------------------------------------------------------
KAVIN_NOTES[10] = ("Kavin's method ('LWP Code'): epcmwrret_turnM1_hourly_averages.nc = hourly means of MWRRET v2 "
                   "(epcmwrret2turnM1.c1) phys_lwp over the QC = 0 samples with LWP > 0; all sky; no cloud or rain filter. "
                   "Hours sorted by Pacific-time month into a MATLAB array nan(500,13); months with > 500 hours grow it, and "
                   "MATLAB fills the new cells of the other months with 0, so 1,142 zeros enter the means. The seasons that "
                   "reproduce these values are Mar-May, Jun-Aug, Sep-Nov, Dec-Feb (the script as shared averages May-Jul, "
                   "Aug-Oct, Nov-Jan, Feb-Apr instead). Reproduced: EPCAPE, Spring, Summer, Fall; Winter mean only (std 121, "
                   "not 92).")
add(10, q(10), "g m-2", "Pier", "epcmwrret_turnM1_hourly_averages.nc (Kavin's file; MWRRET v2)",
    ("Kavin's hourly file, no padding zeros; sheet seasons", False),
    "Same data as Kavin. filters: his hourly all-sky means (QC = 0 samples with LWP > 0); MATLAB padding zeros "
    "removed; no cloud or rain filter; sheet seasons (UTC).")
add(10, q(10), "g m-2", "Pier", "epcmwrret1liljclouM1.c2 (MWRRET v1)", (None, True),
    f"filters: QC = 0 on phys_lwp; cloudy samples only (qc_cloud_base_height = 0); rain-free: no rain (any rate > 0, QC = 0) "
    f"at the Pier Parsivel (LD M1) within ±{LWP_RAIN_WINDOW_MIN:g} min; averaging window {AVG_PERIOD_LWP:g} min (0 = every "
    "sample); sheet seasons (UTC).")
add(10, q(10), "g m-2", "Pier", "epcmwrret2turnM1.c1 (MWRRET v2)",
    ("MWRRET v2 (3-channel MWR, mwrret2turn.c1): phys_lwp, cloudy scenes, rain-free", False),
    f"filters: QC = 0 on phys_lwp and phys_qc_flag = 0; cloudy samples only (cbh_detected > 0 and detection_status 1-3); "
    f"rain-free (no LD M1 rain within ±{LWP_RAIN_WINDOW_MIN:g} min); averaging window {AVG_PERIOD_LWP:g} min; sheet seasons.")
# --- Rows 15-24 ------------------------------------------------------------------------------------
for row in (15, 17, 20, 21):
    KAVIN_NOTES[row] = NOT_SHARED
add(15, q(15) + " (low clouds only)", "m", "Pier", "epcarsclkazr1kolliasM1.c1 (KAZR-ARSCL)", (None, True),
    f"filters: top of the lowest cloud layer <= {LOW_CLOUD_MAX_M:g} m AGL; cloud_source_flag codes "
    f"{CT_CLOUD_SOURCES if CT_CLOUD_SOURCES else 'any'}; averaging window {AVG_PERIOD_CT:g} min (0 = every 4-s profile); "
    "sheet seasons (UTC).")
add(17, q(17), "m", "Pier", "epcpblhtthermoM1.c1 (PBLHTTHERMO) + KAZR-ARSCL", (None, True),
    f"filters: pbl_height (10-min values; no QC field) only when KAZR-ARSCL has a cloud top <= {LOW_CLOUD_MAX_M:g} m within "
    "±5 min; the best match among six readings; sheet seasons (UTC).")
add(17, q(17), "m", "Pier", "epcpblhtthermoM1.c1 (PBLHTTHERMO)", ("PBLHTTHERMO pbl_height, all times", False),
    "filters: none (pbl_height has no QC field); every 10-min value; sheet seasons (UTC).")
KAVIN_NOTES[18] = "Kavin's method: not documented; no MPL-based PBL-height product for EPCAPE in ARM Data Discovery (2026-10-06)."
add(18, q(18), "m", "Pier", "PBLH-MPL", None, "Not checked: no MPL-based PBL-height product for EPCAPE was found.")
KAVIN_NOTES[19] = "Kavin's method: not documented; 'BL-Depth (Miller)' is not an ARM product for EPCAPE."
add(19, q(19), "m", "Pier", "BL-Depth (Miller)", None, "Not checked: source not identified.")
add(20, q(20), "m", "Pier", "epcsondeparamM1.c1 (SONDEPARAM)", (None, True),
    "filters: surface-parcel LCL (parcel type 1), data_quality = 0 (every input check passed); one value per radiosonde "
    "launch; km -> m; sheet seasons (UTC). An independent Romps (2017) LCL from the lowest ~2 hPa of each sonde agrees "
    "(Section 6).")
add(21, q(21), "m", "Pier", "epcceilM1.b1 (Vaisala ceilometer)", (None, True),
    f"filters: qc_first_cbh = 0; detection_status 1-3 (a cloud base detected); first_cbh <= {LOW_CLOUD_MAX_M:g} m above "
    f"the instrument; averaging window {AVG_PERIOD_CBH:g} min (0 = every 16-s sample); sheet seasons (UTC).")
for row, location, product, qc_text in (
        (22, "Pier", "epcldM1.b1 (Parsivel LD)", "QC = 0 on precip_rate"),
        (23, "Soledad", "epcldS2.b1 (Parsivel LD at Mt. Soledad; the sheet's 'LDS1')", "QC = 0 on precip_rate"),
        (24, "Pier", "epcvdisquantsM1.c1 (VDISQUANTS)", "rain_rate has no QC field; NaN (no drops) counted as 0")):
    KAVIN_NOTES[row] = (NOT_SHARED + " The values match the monthly mean of the daily accumulation on rain days, in "
                        "mm/day with ± across months; the 'mm/hr' label does not fit them."
                        + (" No reading reproduces this row." if row == 23 else ""))
    add(row, q(row), "mm/day", location, product, (None, True),
        f"filters: {qc_text}; daily accumulation on rain days (>= {RAIN_MIN_MM:g} mm), averaged over each calendar month; "
        "value = mean ± std across months; sheet seasons (UTC); the best of seven readings.")
# --- Rows 25-29, 54-55: FM-120 droplets and LWC (Abbey's 15-min file, read by Kavin's AMSforAndrew.mlx) -----------
FM15 = ("Abbey Williams' 15-min file EPCAPE_15mindata.nc (read by Kavin's AMSforAndrew.mlx): ClrCldHaz_class = 1 (cloud) or "
        "2 (haze), set only while the CVI is on; its FM-120 values are the daily 5-min files averaged to 15 min")
for row, units, kavin_var, my_col in (
        (25, "cm-3", "FM120_Nd (2-50 um)", "nd_cm3 (Number Conc, 2-50 um)"),
        (26, "um", "D_eff = sum(N D^3) / sum(N D^2) from FM120_DSD with each bin's UPPER edge as its diameter",
         "ed_um (the FM-120 ED column, the effective diameter), NaN when N = 0"),
        (28, "cm-3", "FM120_Nd (2-50 um)", "nd_cm3 (Number Conc, 2-50 um)"),
        (29, "um", "D_eff = sum(N D^3) / sum(N D^2) from FM120_DSD with each bin's UPPER edge as its diameter",
         "ed_um (the FM-120 ED column, the effective diameter), NaN when N = 0"),
        (54, "g m-3", "FM120_LWC", "lwc_gm3"), (55, "g m-3", "FM120_LWC", "lwc_gm3")):
    kind, code_ = ("cloud", 1) if row in (25, 26, 54) else ("haze", 2)
    mid_note = ""
    if row in (26, 29):
        mid = pick(row, "bin mid-points")["ours"]
        mid_note = (" Upper edges make D_eff larger than the bin mid-points would (mid-points: "
                    + " / ".join(str(v) for v in mid) + ", EPCAPE / Spring / Summer / Fall / Winter).")
    KAVIN_NOTES[row] = (f"Kavin's method ({FM15}): {kavin_var}, mean ± std over the 15-min intervals with ClrCldHaz_class = "
                        f"{code_}; calendar quarters (winter = Jan-Mar), UTC. Re-running it on the file reproduces this row."
                        + mid_note)
    add(row, q(row), units, "Soledad", "EPCAPE_15mindata.nc (Abbey's 15-min file)", ("Kavin's method: Abbey's 15-min file", False),
        "Kavin's method re-run on his file (unrounded).")
    add(row, q(row), units, "Soledad", "FM120 daily 5-min files + Abbey's 15-min classes", (None, True),
        f"filters: FM-120 5-min rows, each taking the ClrCldHaz_class of the 15-min interval holding it (= {code_}, {kind}); "
        f"{my_col}; mean ± std over rows; sheet seasons (UTC).")


# --- Rows 27, 30: activated fraction ---------------------------------------------------------------------
for row, code_ in ((27, 1), (30, 2)):
    KAVIN_NOTES[row] = (f"Kavin's method ('Critical Diameters & % Activated Code from Abbey'): ET_Dcrit_nativetime.csv; rows "
                        f"with ClrCldHaz_class = {code_} and both Dta and Dta_critical; fraction with Dta > Dta_critical, "
                        "counted per sample; his code gives the campaign value only (rounded in a pie chart). Dta_critical "
                        "exists only in Jun-Sep and Dec 2023, so Spring = June only, Fall = December only, no Winter. "
                        "Reproduced exactly.")
    add(row, q(row), "fraction", "Soledad", "ET_Dcrit_nativetime.csv (Abbey Williams)", (None, True),
        f"Kavin's count re-run on his file, per season. filters: ClrCldHaz_class = {code_}; rows with both Dta and "
        "Dta_critical; fraction with Dta > Dta_critical (per sample, not time-weighted); sheet seasons (UTC). Not "
        "independent: Dta and Dta_critical come from Abbey's code. Dta_critical only Jun-Sep and Dec 2023.")
# --- Rows 31-32: FM-120 cloud and haze hours ('Fig. 5'; computed by Kavin's code) --------------------------
for row, kind, kavin_rule, paper_rule in (
        (31, "cloud", "cloud = avgvis < 1 km and LWC > 0.01 g/m3",
         f"{FM120_VIS_STAT} visibility < {FM120_CLOUD_VIS_MAX_M / 1e3:g} km and LWC > {FM120_CLOUD_LWC_MIN_GM3:g} g/m3"),
        (32, "haze", "haze = 1 km < avgvis < 5 km, no LWC condition",
         f"{FM120_VIS_STAT} visibility < {FM120_HAZE_VIS_MAX_M / 1e3:g} km and LWC < {FM120_HAZE_LWC_MAX_GM3:g} g/m3")):
    KAVIN_NOTES[row] = ("Kavin's method ('Hours Calculations (Big File)'): FM-120 5-min rows + msd_visibility_*.csv; avgvis = "
                        "mean of the raw ~1-s visibility from each row's time to the next row's (both ends included; spans "
                        f"instrument gaps; no screening of zeros or spikes); {kavin_rule}; hours = rows / 12; calendar quarters "
                        "(UTC); each season truncated to whole hours; EPCAPE = sum of the seasons. Reproduced exactly.")
    add(row, q(row), "hr", "Soledad", "FM120 & msd_visibility", ("Kavin's code:", False),
        "Kavin's code re-run on the same files, not truncated (truncating each season to whole hours gives the sheet).")
    add(row, q(row), "hr", "Soledad", "FM120 & msd_visibility", (None, True),
        f"filters: BAMS-paper definition: {paper_rule}; interval visibility = {FM120_VIS_STAT} of the screened 1-s samples "
        f"in each 5-min FM-120 interval (0 < vis <= 48,270 m; >= {FM120_VIS_MIN_SAMPLES} samples); hours = rows x 5 min; "
        "sheet seasons (UTC).")
    SET2_NOTES[row] = (
        f"filters: filter set 2 (Andrew, 2026-10-08): cloud = {FM120_VIS_STAT} visibility < {FM120_CLOUD_VIS_MAX_M / 1e3:g} km "
        f"and LWC > {FM120_CLOUD_LWC_MIN_GM3:g} g/m3 (unchanged); haze = (1 km < {FM120_VIS_STAT} visibility < "
        f"{FM120_HAZE2_VIS_MAX_M / 1e3:g} km and LWC < {FM120_HAZE2_LWC_MAX_GM3:g} g/m3) or ({FM120_VIS_STAT} visibility < 1 km and "
        f"LWC < {FM120_HAZE2_LWC_MAX_GM3:g} g/m3), i.e. visibility < {FM120_HAZE2_VIS_MAX_M / 1e3:g} km and LWC < "
        f"{FM120_HAZE2_LWC_MAX_GM3:g} g/m3 except exactly 1 km; visibility screened (0 < vis <= 48,270 m; >= "
        f"{FM120_VIS_MIN_SAMPLES} samples per 5-min interval); hours = rows x 5 min; sheet seasons (UTC)."
        + (" The cloud hours equal the BAMS-paper row above." if row == 31 else
           (" With these thresholds the haze rows differ from the BAMS-paper definition only in intervals with a "
            "visibility of exactly 1 km." if (FM120_HAZE2_VIS_MAX_M, FM120_HAZE2_LWC_MAX_GM3) ==
            (FM120_HAZE_VIS_MAX_M, FM120_HAZE_LWC_MAX_GM3) else "")))
    add(row, q(row), "hr", "Soledad", "FM120 & msd_visibility", ("filter set 2", False), SET2_NOTES[row])
# --- Rows 33-42: GCVI-AMS hours and residuals (Kavin's AMSforAndrew.mlx, Abbey's 15-min file) ----------------------
for row, kind, code_ in ((33, "cloud", 1), (34, "haze", 2)):
    KAVIN_NOTES[row] = (f"Kavin's method (AMSforAndrew.mlx): Abbey Williams' 15-min file EPCAPE_15mindata.nc; hours = number of "
                        f"15-min intervals with ClrCldHaz_class = {code_} (set only while the CVI is on) / 4, whether or not the "
                        "AMS reported; calendar quarters (UTC); values rounded. Reproduced."
                        + (" The Fall cell '5.0*10^2' is a typo for 5.0*10^1: his code gives 49.5 h." if row == 34 else ""))
    add(row, q(row), "hr", "Soledad", "EPCAPE_15mindata.nc (Abbey's 15-min file)", ("Kavin's AMS code: Abbey's 15-min", False),
        "Kavin's code re-run on his file, not rounded.")
    add(row, q(row), "hr", "Soledad", "EPCAPE_15mindata.nc classes + AMS", (None, True),
        f"filters: Abbey's 15-min intervals with ClrCldHaz_class = {code_} in which the AMS reported (AMS_mass present), "
        "x 15 min; sheet seasons (UTC). Kavin's count also includes the class intervals without AMS data.")
for row in range(35, 43):
    kind, code_ = ("cloud", 1) if row <= 38 else ("haze", 2)
    KAVIN_NOTES[row] = (f"Kavin's method (AMSforAndrew.mlx): mean ± std of AMS_mass_15min_EF over Abbey's 15-min intervals with "
                        f"ClrCldHaz_class = {code_}; rows of that 5 x N matrix = nitrate, sulfate, organics, ammonium, chloride "
                        "(Abbey); calendar quarters (UTC). The AMS_mass of EPCAPE_15mindata.nc reproduces every value: it is the "
                        "CE-corrected AMS (AMS_CE_corrected_msptof.nc) averaged over 15 min and divided by EF while the CVI is "
                        f"on, except in {NO_EF_TEXT}; they raise the spring cloud (April) and winter haze (March) values. The script as shared averages "
                        "AMS_massfrac (mass fraction) for spring, but the sheet's spring values are concentrations. All values "
                        "are concentrations (ug m-3), not mass fractions.")
    add(row, q(row), "ug m-3", "Soledad", "EPCAPE_15mindata.nc (Abbey's 15-min file)",
        ("Kavin's AMS code: mean of Abbey's 15-min", False), "Kavin's code re-run on his file.")
    add(row, q(row), "ug m-3", "Soledad", "AMS_CE_corrected_msptof.nc + CVI_EF.csv + Abbey's 15-min classes", (None, True),
        f"filters: own AMS samples: CE-corrected AMS whose 2-min V-mode period lies inside a GCVI segment (EF > 1, >= "
        f"{GCVI_MIN_SEGMENT_MIN:g} min), divided by the mean EF over that period (Shingler et al. 2012); class = the "
        f"ClrCldHaz_class of the 15-min interval holding the sample (= {code_}, {kind}); sheet seasons (UTC). Samples "
        "without a GCVI EF record are excluded, unlike Kavin's spring cloud and winter haze.")

def values_of(row, source):
    """This check's 'mean ± std' per season (E-I), or blanks for a row that could not be checked."""
    if source is None:
        return [""] * 5
    return [("n/a" if v in ("–", "") else v) for v in pick(row, *source)["ours"]]


# --- 1. Read the sheet; confirm every target row is the row we think it is -------------------------------
wb = load_workbook(TABLE_XLSX)
ws = wb["Cloud&MetQuantities"]
for row in MY_ROWS:
    assert str(ws.cell(row, 1).value).strip() == q(row).strip(), (row, ws.cell(row, 1).value)
before = {r: [c.value for c in ws[r]] for r in range(1, ws.max_row + 1)}  # every original row, for the check in step 4
formulas = {(c.row, c.column): c.value for line in ws.iter_rows() for c in line
            if isinstance(c.value, str) and c.value.startswith("=")}
heights = {r: d.height for r, d in ws.row_dimensions.items() if d.height is not None}

# --- 2. Insert the pink rows, bottom row first so the rows still to do keep their numbers ---------------------
for row in sorted(MY_ROWS, reverse=True):
    lines = MY_ROWS[row]
    ws.insert_rows(row + 1, amount=len(lines))
    for k, (quantity, units, location, product, source, notes) in enumerate(lines):
        r = row + 1 + k
        for j, value in enumerate([quantity, units, location, product, *values_of(row, source), AUTHOR, notes], start=1):
            ws.cell(r, j, value)
        for j in range(1, FILL_COLUMNS + 1):
            ws.cell(r, j).fill = PINK
            ws.cell(r, j).font = copy(ws.cell(row, j).font)  # same font as Kavin's row (Arial 10)
    # Kavin's row: only the notes cell changes (an existing note is kept and the method is appended)
    old = ws.cell(row, NOTES_COL).value
    note = KAVIN_NOTES.get(row, "")
    ws.cell(row, NOTES_COL, f"{str(old).strip()} | {note}" if old else note)
ws.cell(2, NOTES_COL, "Notes").font = copy(ws.cell(2, 1).font)  # header, as in Andrew's version
ws.column_dimensions["K"].width = 140


# --- 3. Row numbers after the insertions: fix formulas and row heights ----------------------------------------
def new_row(old):
    """Row number of an original row after all insertions."""
    return old + sum(len(lines) for row, lines in MY_ROWS.items() if row < old)


for (r, col), formula in formulas.items():
    cell = ws.cell(new_row(r), col)
    cell.value = Translator(formula, origin=f"{cell.column_letter}{r}").translate_formula(cell.coordinate)
default_height = heights.get(min(MY_ROWS), None)
for r in list(ws.row_dimensions):
    ws.row_dimensions[r].height = None
for r, h in heights.items():
    ws.row_dimensions[new_row(r)].height = h
for row, lines in MY_ROWS.items():
    for k in range(len(lines)):
        ws.row_dimensions[new_row(row) + 1 + k].height = default_height

# --- 4. Check: every original row unchanged except Kavin's notes cells (and moved formulas); save -----------
for r, values in before.items():
    now = [c.value for c in ws[new_row(r)]]
    for j, (a, b) in enumerate(zip(values, now), start=1):
        if j == NOTES_COL and r in MY_ROWS or (r, j) in formulas or (r == 2 and j == NOTES_COL):
            continue
        assert a == b, (r, j, a, b)
pink_path = OUT_DIR / "Table-EPCAPE_SeasonalAverages - with this check.xlsx"
wb.save(pink_path)
n_added = sum(len(v) for v in MY_ROWS.values())
print(f"Added {n_added} pink rows below {len(MY_ROWS)} of the sheet's rows; original rows unchanged except column K "
      f"of Kavin's rows. Wrote {pink_path}")

# --- 5. Preview of the checked part of the sheet ---------------------------------------------------------------
preview = pd.DataFrame([[ws.cell(r, j).value for j in (1, 4, 5, 6, 7, 8, 9, 10)]
                        for r in range(new_row(min(MY_ROWS)), new_row(max(MY_ROWS)) + len(MY_ROWS[max(MY_ROWS)]) + 1)],
                       columns=["Quantity", "Method / product", "EPCAPE", "Spring", "Summer", "Fall", "Winter", "Author"])
display(preview)

Added 57 pink rows below 32 of the sheet's rows; original rows unchanged except column K of Kavin's rows. Wrote /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/processed/derived/seasonal_averages_check/Table-EPCAPE_SeasonalAverages - with this check.xlsx


,Quantity,Method / product,EPCAPE,Spring,Summer,Fall,Winter,Author
0,Shortwave (SW) Transmittance,"SKYRAD, RADFLUX1LONG",0.58 ± 0.29,0.50 ± 0.31,0.56 ± 0.28,0.66 ± 0.24,0.61 ± 0.28,Kavin
1,Shortwave (SW) Transmittance,Lubin's hourly SW transmittance tables (Kavin's files),n/a,n/a,0.56 ± 0.27,0.68 ± 0.21,n/a,Andrew Buggee
2,Shortwave (SW) Transmittance,epcradflux1longM1.c2 (RADFLUX),0.71 ± 0.32,0.61 ± 0.34,0.67 ± 0.31,0.84 ± 0.25,0.76 ± 0.32,Andrew Buggee
3,Longwave (LW) Flux,From Lubin,350 ± 30,370 ± 10,380 ± 10,323 ± 5,308 ± 3,Kavin
4,Longwave (LW) Flux,epcradflux1longM1.c2 (RADFLUX; the record Lubin's tables are made from),346 ± 34,367 ± 13,384 ± 14,323.0 ± 4.8,308.7 ± 3.5,Andrew Buggee
...,...,...,...,...,...,...,...,...
101,Cloud Liquid Water Content,EPCAPE_15mindata.nc (Abbey's 15-min file),0.14 ± 0.17,0.16 ± 0.15,0.12 ± 0.17,0.17 ± 0.23,0.12 ± 0.16,Andrew Buggee
102,Cloud Liquid Water Content,FM120 daily 5-min files + Abbey's 15-min classes,0.14 ± 0.18,0.16 ± 0.16,0.12 ± 0.17,0.17 ± 0.24,0.12 ± 0.17,Andrew Buggee
103,Haze Liquid Water Content,FM120,None,0.0077 ± 0.0081,0.0089 ± 0.0091,0.0062 ± 0.0145,0.021 ± 0.033,Kavin Pugazhenthi
104,Haze Liquid Water Content,EPCAPE_15mindata.nc (Abbey's 15-min file),0.009 ± 0.013,0.0077 ± 0.0081,0.0089 ± 0.0091,0.006 ± 0.015,0.021 ± 0.033,Andrew Buggee


## 15. Andrew's verification workbook: rows for filter set 2

Andrew keeps his own edited copy of the Section 14 workbook, `Table-EPCAPE_SeasonalAverages - AJB verification - ver 2.xlsx`, in `OUT_DIR`. This cell adds one pink row **directly below the BAMS-paper row** of each FM-120 hours row (31 cloud, 32 haze) with the filter-set-2 values and notes:
* The new row copies the style of the BAMS-paper row.
* Every other cell of the workbook is left as it is. The cell checks this and re-points any formula that moves.
* On a re-run the rows are found again (their notes contain "filter set 2") and updated in place, not added twice.
* Before the first change, a copy of the original is saved alongside as `… (backup before filter set 2).xlsx`.
* If the workbook is open in Excel (an `~$` lock file exists), the cell does nothing and says so.

In [33]:
# =============================================================================
# Section 15: add the filter-set-2 rows to Andrew's verification workbook
# =============================================================================
# Inputs
#   OUT_DIR/Table-EPCAPE_SeasonalAverages - AJB verification - ver 2.xlsx  Andrew's edited copy of the Section 14 output
#   values_of, q, SET2_NOTES (Section 14)  the filter-set-2 values ('mean ± std' per season) and notes
# The anchor for each new row: Kavin's ("Fig. 5") row of the quantity, then the first row below it whose notes start
# with "filters: BAMS-paper definition"; the new row goes directly below that one.
from shutil import copy2

AJB_XLSX = OUT_DIR / "Table-EPCAPE_SeasonalAverages - AJB verification - ver 2.xlsx"
MARKER = "filter set 2"  # text in the notes of the rows written here; finds them again on a re-run
LOCK = AJB_XLSX.with_name("~$" + AJB_XLSX.name)  # Excel's lock file while the workbook is open


def insert_row_at(ws, r):
    """Insert an empty row at r; formulas and row heights below move with their rows (openpyxl moves neither)."""
    formulas = {(c.row, c.column): c.value for line in ws.iter_rows(min_row=r) for c in line
                if isinstance(c.value, str) and c.value.startswith("=")}
    heights = {k: d.height for k, d in ws.row_dimensions.items() if k >= r and d.height is not None}
    ws.insert_rows(r)
    for (row_, col), formula in formulas.items():
        cell = ws.cell(row_ + 1, col)
        cell.value = Translator(formula, origin=f"{cell.column_letter}{row_}").translate_formula(cell.coordinate)
    for k in sorted(heights, reverse=True):
        ws.row_dimensions[k + 1].height = heights[k]


if not AJB_XLSX.is_file():
    print(f"Skipped: {AJB_XLSX.name} not found in {OUT_DIR}")
elif LOCK.exists():
    print(f"Skipped: {AJB_XLSX.name} is open in Excel (lock file {LOCK.name}); close it and re-run this cell")
else:
    backup = AJB_XLSX.with_name(AJB_XLSX.stem + " (backup before filter set 2).xlsx")
    if not backup.exists():
        copy2(AJB_XLSX, backup)  # keep the untouched original once
    wb15 = load_workbook(AJB_XLSX)
    ws15 = wb15["Cloud&MetQuantities"]
    before = [[c.value for c in line] for line in ws15.iter_rows()]  # for the check below
    inserted = []  # row numbers (final) of newly inserted rows
    for row in (32, 31):  # bottom first, so the cloud row's position is not shifted by the haze insert
        anchor = next(r for r in range(1, ws15.max_row + 1)
                      if str(ws15.cell(r, 1).value or "").strip() == q(row).strip() and ws15.cell(r, 10).value == "Fig. 5")
        bams = next(r for r in range(anchor + 1, anchor + 12)
                    if str(ws15.cell(r, NOTES_COL).value or "").startswith("filters: BAMS-paper definition"))
        target = bams + 1
        if MARKER not in str(ws15.cell(target, NOTES_COL).value or ""):  # not there yet: make room
            insert_row_at(ws15, target)
            inserted = [t + 1 if t >= target else t for t in inserted] + [target]
        values = [ws15.cell(bams, 1).value, ws15.cell(bams, 2).value, ws15.cell(bams, 3).value, "FM120 & msd_visibility",
                  *values_of(row, ("filter set 2", False)), ws15.cell(bams, 10).value, SET2_NOTES[row]]
        for j in range(1, FILL_COLUMNS + 1):  # the BAMS-paper row's look (pink fill, Arial 10, ...)
            cell, model = ws15.cell(target, j), ws15.cell(bams, j)
            cell.font, cell.fill, cell.alignment = copy(model.font), copy(model.fill), copy(model.alignment)
            cell.border, cell.number_format = copy(model.border), model.number_format
            cell.value = values[j - 1] if j <= len(values) else None
        ws15.row_dimensions[target].height = ws15.row_dimensions[bams].height
    # check: apart from the inserted rows, every cell is unchanged (formulas only re-pointed)
    #        (rows holding the marker are this cell's own rows and may be refreshed)
    after = [[c.value for c in line] for i, line in enumerate(ws15.iter_rows(), start=1) if i not in inserted]
    assert len(after) == len(before)
    for old, new in zip(before, after):
        if any(MARKER in str(v or "") for v in old + new):
            continue
        for a, b in zip(old, new):
            if a != b and not (isinstance(a, str) and a.startswith("=")):
                raise AssertionError(f"unexpected change: {a!r} -> {b!r}")
    wb15.save(AJB_XLSX)
    print(f"Updated {AJB_XLSX.name}: " + (f"inserted rows {sorted(inserted)}" if inserted else "rows already present, values refreshed")
          + f"; backup of the original: {backup.name}")
    display(pd.DataFrame([[ws15.cell(r, j).value for j in (1, 4, 5, 6, 7, 8, 9, 10)]
                          for r in range(anchor, anchor + 10)],
                         columns=["Quantity", "Method / product", "EPCAPE", "Spring", "Summer", "Fall", "Winter", "Author"],
                         index=range(anchor, anchor + 10)))

Updated Table-EPCAPE_SeasonalAverages - AJB verification - ver 2.xlsx: rows already present, values refreshed; backup of the original: Table-EPCAPE_SeasonalAverages - AJB verification - ver 2 (backup before filter set 2).xlsx


,Quantity,Method / product,EPCAPE,Spring,Summer,Fall,Winter,Author
64,On-Site Cloud Sampling by Fog Drop Monitor,FM120,=SUM(F64:I64),134,212,66,58,Fig. 5
65,On-Site Cloud Sampling by Fog Drop Monitor,FM120 & msd_visibility,473,135,213,66.2,58.7,Andrew Buggee
66,On-Site Cloud Sampling by Fog Drop Monitor,FM120 & msd_visibility,480,137,217,66.8,58.8,Andrew Buggee
67,On-Site Cloud Sampling by Fog Drop Monitor,FM120 & msd_visibility,480,137,217,66.8,58.8,Andrew Buggee
68,None,None,None,None,None,None,None,None
69,On-Site Haze Sampling by Fog Drop Monitor,FM120,=SUM(F69:I69),85,172,49,112,Fig. 5
70,On-Site Haze Sampling by Fog Drop Monitor,FM120 & msd_visibility,420,85.2,173,49.8,112,Andrew Buggee
71,On-Site Haze Sampling by Fog Drop Monitor,FM120 & msd_visibility,395,77.6,161,56,100,Andrew Buggee
72,On-Site Haze Sampling by Fog Drop Monitor,FM120 & msd_visibility,395,77.6,161,56,100,Andrew Buggee
73,None,None,None,None,None,None,None,None
